In [ ]:
# ====================================================================================================
# V3 CELL 1 — SETUP / IMPORTS
# ====================================================================================================



# ------------------------------------------------------------
# 2. Install lightweight sentence-transformers stack
# ------------------------------------------------------------

!pip -q install sentence-transformers


# ------------------------------------------------------------
# 3. Standard imports
# ------------------------------------------------------------

import os
import gc
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path


# ------------------------------------------------------------
# 4. ML / similarity imports
# ------------------------------------------------------------

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 5. Sentence embedding model
# ------------------------------------------------------------

from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 6. V3 project directories
# ------------------------------------------------------------

PROJECT_DIR = (
    "/content/drive/MyDrive/Resume_Model_Project"
)

V2_DIR = os.path.join(
    PROJECT_DIR,
    "V2_100k"
)

V2_INPUT_DIR = os.path.join(
    V2_DIR,
    "Inputs"
)

V2_OUTPUT_DIR = os.path.join(
    V2_DIR,
    "Outputs"
)

V3_DIR = os.path.join(
    PROJECT_DIR,
    "V3"
)

V3_INPUT_DIR = os.path.join(
    V3_DIR,
    "Inputs"
)

V3_OUTPUT_DIR = os.path.join(
    V3_DIR,
    "Outputs"
)


# ------------------------------------------------------------
# 7. Create V3 directories
# ------------------------------------------------------------

os.makedirs(
    V3_INPUT_DIR,
    exist_ok=True
)

os.makedirs(
    V3_OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# 8. Load V2 reference data
# ------------------------------------------------------------

candidate_df = pd.read_pickle(
    os.path.join(
        V2_INPUT_DIR,
        "candidate_df.pkl"
    )
)

entry_job_df = pd.read_pickle(
    os.path.join(
        V2_INPUT_DIR,
        "entry_job_df.pkl"
    )
)


# ------------------------------------------------------------
# 9. Load V2 supervised / evaluation artifacts
# ------------------------------------------------------------

rich_dataset_path = os.path.join(
    V2_OUTPUT_DIR,
    "v2_supervised_dataset_100k_rich.pkl"
)

split_path = os.path.join(
    V2_OUTPUT_DIR,
    "v2_train_validation_split.csv"
)


rich_df = pd.read_pickle(
    rich_dataset_path
)

split_df = pd.read_csv(
    split_path
)


# ------------------------------------------------------------
# 10. Basic verification
# ------------------------------------------------------------

print("=" * 100)
print("V3 CELL 1 — SETUP COMPLETE")
print("=" * 100)

print("\nCandidates:", len(candidate_df))
print("Jobs:", len(entry_job_df))
print("V2 Rich dataset:", len(rich_df))
print("V2 split rows:", len(split_df))

print("\nCandidate columns:")
print(candidate_df.columns.tolist())

print("\nJob columns:")
print(entry_job_df.columns.tolist())

print("\nRich dataset columns:")
print(rich_df.columns.tolist())


# ------------------------------------------------------------
# 11. Load lightweight embedding model
# ------------------------------------------------------------

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print(
    f"\nLoading embedding model: "
    f"{EMBEDDING_MODEL_NAME}"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("✅ Embedding model loaded.")


# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print("\n✅ V3 environment ready.")
print("✅ V2 remains untouched.")
print("✅ No training performed yet.")
print("✅ No existing V2 artifacts modified.")

In [ ]:
# ====================================================================================================
# V3 CELL 2 — BUILD AND CACHE SEMANTIC EMBEDDINGS
# ====================================================================================================

print("=" * 100)
print("V3 CELL 2 — SEMANTIC EMBEDDING GENERATION")
print("=" * 100)


# =================================================================================
# 1. Clean text helper
# =================================================================================

def clean_embedding_text(text):
    """
    Light text cleanup for semantic encoding.
    We intentionally avoid aggressive preprocessing because
    the embedding model benefits from natural language context.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize whitespace
    text = " ".join(
        text.split()
    )

    return text.strip()


# =================================================================================
# 2. Prepare resume texts
# =================================================================================

resume_texts = (
    candidate_df["resume_text"]
    .fillna("")
    .astype(str)
    .map(clean_embedding_text)
    .tolist()
)


# =================================================================================
# 3. Prepare job texts
# =================================================================================

job_texts = (
    entry_job_df["job_text"]
    .fillna("")
    .astype(str)
    .map(clean_embedding_text)
    .tolist()
)


# =================================================================================
# 4. Verify counts
# =================================================================================

assert len(resume_texts) == 1540
assert len(job_texts) == 434

assert all(
    isinstance(x, str)
    for x in resume_texts
)

assert all(
    isinstance(x, str)
    for x in job_texts
)


print("Resume texts:", len(resume_texts))
print("Job texts   :", len(job_texts))


# =================================================================================
# 5. Show a small sanity check
# =================================================================================

print("\nSample resume text:")
print(
    resume_texts[0][:500]
)

print("\nSample job text:")
print(
    job_texts[0][:500]
)


# =================================================================================
# 6. Select embedding device
# =================================================================================

import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nEmbedding device:", device)


# =================================================================================
# 7. Generate resume embeddings
# =================================================================================

print("\nGenerating resume embeddings...")

resume_embeddings = embedding_model.encode(
    resume_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device=device
)


# =================================================================================
# 8. Generate job embeddings
# =================================================================================

print("\nGenerating job embeddings...")

job_embeddings = embedding_model.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device=device
)


# =================================================================================
# 9. Verify embedding dimensions
# =================================================================================

print("\nResume embedding shape:")
print(resume_embeddings.shape)

print("\nJob embedding shape:")
print(job_embeddings.shape)


assert resume_embeddings.shape[0] == 1540
assert job_embeddings.shape[0] == 434

assert resume_embeddings.shape[1] == job_embeddings.shape[1]


# =================================================================================
# 10. Check for invalid values
# =================================================================================

assert np.isfinite(
    resume_embeddings
).all()

assert np.isfinite(
    job_embeddings
).all()


# =================================================================================
# 11. Save embeddings
# =================================================================================

RESUME_EMBEDDINGS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_resume_embeddings.npy"
)

JOB_EMBEDDINGS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_job_embeddings.npy"
)

RESUME_TEXTS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_resume_texts.pkl"
)

JOB_TEXTS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_job_texts.pkl"
)


np.save(
    RESUME_EMBEDDINGS_PATH,
    resume_embeddings
)

np.save(
    JOB_EMBEDDINGS_PATH,
    job_embeddings
)

joblib.dump(
    resume_texts,
    RESUME_TEXTS_PATH
)

joblib.dump(
    job_texts,
    JOB_TEXTS_PATH
)


# =================================================================================
# 12. Save metadata
# =================================================================================

embedding_metadata = {
    "model_name": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(
        resume_embeddings.shape[1]
    ),
    "num_resumes": len(resume_texts),
    "num_jobs": len(job_texts),
    "normalized_embeddings": True,
    "device_used": device
}


metadata_path = os.path.join(
    V3_OUTPUT_DIR,
    "v3_embedding_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        embedding_metadata,
        f,
        indent=4
    )


# =================================================================================
# 13. Final verification
# =================================================================================

print("\n" + "=" * 100)
print("V3 EMBEDDING GENERATION COMPLETE")
print("=" * 100)

print(
    "Resume embeddings:",
    resume_embeddings.shape
)

print(
    "Job embeddings   :",
    job_embeddings.shape
)

print(
    "\nSaved resume embeddings:",
    RESUME_EMBEDDINGS_PATH
)

print(
    "Saved job embeddings   :",
    JOB_EMBEDDINGS_PATH
)

print(
    "Saved metadata:",
    metadata_path
)

print("\n✅ 1,540 resume embeddings cached.")
print("✅ 434 job embeddings cached.")
print("✅ Embeddings are L2-normalized.")
print("✅ V2 artifacts were not modified.")

In [ ]:
# ====================================================================================================
# V3 CELL 2R — LOAD CACHED EMBEDDINGS AFTER RUNTIME DISCONNECT
# ====================================================================================================

print("=" * 100)
print("V3 CELL 2R — LOAD CACHED EMBEDDINGS")
print("=" * 100)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

RESUME_EMBEDDINGS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_resume_embeddings.npy"
)

JOB_EMBEDDINGS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_job_embeddings.npy"
)

RESUME_TEXTS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_resume_texts.pkl"
)

JOB_TEXTS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_job_texts.pkl"
)


# ------------------------------------------------------------
# Load cached artifacts
# ------------------------------------------------------------

resume_embeddings = np.load(
    RESUME_EMBEDDINGS_PATH
)

job_embeddings = np.load(
    JOB_EMBEDDINGS_PATH
)

resume_texts = joblib.load(
    RESUME_TEXTS_PATH
)

job_texts = joblib.load(
    JOB_TEXTS_PATH
)


# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

assert resume_embeddings.shape == (
    1540,
    384
)

assert job_embeddings.shape == (
    434,
    384
)

assert len(resume_texts) == 1540
assert len(job_texts) == 434


print("Resume embeddings:", resume_embeddings.shape)
print("Job embeddings   :", job_embeddings.shape)

print("\n✅ Cached embeddings loaded.")
print("✅ No embedding generation required.")

In [ ]:
# ====================================================================================================
# V3 CELL 3 — FULL 668,360-PAIR SEMANTIC SIMILARITY MATRIX
# ====================================================================================================

print("=" * 100)
print("V3 CELL 3 — FULL SEMANTIC SIMILARITY")
print("=" * 100)


# =================================================================================
# 1. Verify embedding matrices
# =================================================================================

assert resume_embeddings.shape == (
    1540,
    384
)

assert job_embeddings.shape == (
    434,
    384
)


# =================================================================================
# 2. Compute all semantic similarities
#
# Since embeddings are L2-normalized:
#
# cosine_similarity = embedding dot product
#
# =================================================================================

print("\nComputing all resume-job semantic similarities...")

semantic_similarity_matrix = (
    resume_embeddings
    @
    job_embeddings.T
)


# =================================================================================
# 3. Verify matrix
# =================================================================================

print(
    "\nSimilarity matrix shape:",
    semantic_similarity_matrix.shape
)

assert semantic_similarity_matrix.shape == (
    1540,
    434
)

assert np.isfinite(
    semantic_similarity_matrix
).all()


# =================================================================================
# 4. Sanity-check similarity range
# =================================================================================

print(
    "\nMinimum similarity:",
    semantic_similarity_matrix.min()
)

print(
    "Maximum similarity:",
    semantic_similarity_matrix.max()
)

print(
    "Mean similarity:",
    semantic_similarity_matrix.mean()
)


# =================================================================================
# 5. Convert matrix into pair-level dataframe
# =================================================================================

print("\nBuilding pair-level semantic dataset...")


resume_indices = np.repeat(
    np.arange(1540),
    434
)

job_indices = np.tile(
    np.arange(434),
    1540
)


semantic_scores = (
    semantic_similarity_matrix
    .reshape(-1)
)


v3_semantic_df = pd.DataFrame({
    "Resume_Index": resume_indices,
    "Job_Index": job_indices,
    "Semantic_Similarity": semantic_scores
})


# =================================================================================
# 6. Verify exact pair count
# =================================================================================

print(
    "\nPair count:",
    len(v3_semantic_df)
)

assert len(
    v3_semantic_df
) == 668360


assert (
    v3_semantic_df[
        ["Resume_Index", "Job_Index"]
    ]
    .drop_duplicates()
    .shape[0]
    == 668360
)


# =================================================================================
# 7. Save similarity matrix
# =================================================================================

SEMANTIC_MATRIX_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_similarity_matrix.npy"
)

np.save(
    SEMANTIC_MATRIX_PATH,
    semantic_similarity_matrix
)


# =================================================================================
# 8. Save pair-level semantic scores
# =================================================================================

SEMANTIC_PAIRS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_pair_scores.csv"
)

v3_semantic_df.to_csv(
    SEMANTIC_PAIRS_PATH,
    index=False
)


# =================================================================================
# 9. Final output
# =================================================================================

print("\n" + "=" * 100)
print("V3 CELL 3 COMPLETE")
print("=" * 100)

print(
    "Semantic matrix:",
    semantic_similarity_matrix.shape
)

print(
    "Total pairs:",
    len(v3_semantic_df)
)

print(
    "\nSaved matrix:",
    SEMANTIC_MATRIX_PATH
)

print(
    "Saved pair scores:",
    SEMANTIC_PAIRS_PATH
)

print("\n✅ 668,360 resume-job semantic scores generated.")
print("✅ V2 artifacts untouched.")

In [ ]:
# ====================================================================================================
# V3 CELL 3R — RECOVER SEMANTIC PAIR SCORES AFTER RUNTIME DISCONNECT
# ====================================================================================================

SEMANTIC_MATRIX_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_similarity_matrix.npy"
)

SEMANTIC_PAIRS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_pair_scores.csv"
)

print("=" * 100)
print("V3 CELL 3R — SEMANTIC SIMILARITY RECOVERY")
print("=" * 100)


matrix_exists = os.path.exists(
    SEMANTIC_MATRIX_PATH
)

pairs_exists = os.path.exists(
    SEMANTIC_PAIRS_PATH
)

print("Semantic matrix exists:", matrix_exists)
print("Pair-score CSV exists  :", pairs_exists)


# =================================================================================
# CASE 1 — Everything was saved
# =================================================================================

if matrix_exists and pairs_exists:

    print("\n✅ Cell 3 completed before the runtime disconnected.")

    semantic_similarity_matrix = np.load(
        SEMANTIC_MATRIX_PATH
    )

    v3_semantic_df = pd.read_csv(
        SEMANTIC_PAIRS_PATH
    )

    print(
        "Matrix shape:",
        semantic_similarity_matrix.shape
    )

    print(
        "Pair rows:",
        len(v3_semantic_df)
    )


# =================================================================================
# CASE 2 — Matrix saved, CSV not saved
# =================================================================================

elif matrix_exists and not pairs_exists:

    print(
        "\n⚠️ Matrix was saved, "
        "but pair CSV was not."
    )

    print(
        "Rebuilding the pair dataframe "
        "from the saved matrix..."
    )

    semantic_similarity_matrix = np.load(
        SEMANTIC_MATRIX_PATH
    )

    resume_indices = np.repeat(
        np.arange(1540),
        434
    )

    job_indices = np.tile(
        np.arange(434),
        1540
    )

    v3_semantic_df = pd.DataFrame({
        "Resume_Index": resume_indices,
        "Job_Index": job_indices,
        "Semantic_Similarity":
            semantic_similarity_matrix.reshape(-1)
    })

    v3_semantic_df.to_csv(
        SEMANTIC_PAIRS_PATH,
        index=False
    )

    print("✅ Pair dataframe rebuilt and saved.")


# =================================================================================
# CASE 3 — Nothing was saved
# =================================================================================

else:

    print(
        "\n❌ Cell 3 was interrupted "
        "before anything was saved."
    )

    print(
        "Run the original V3 Cell 3 again."
    )

    raise RuntimeError(
        "No cached semantic similarity artifact found."
    )


# =================================================================================
# FINAL VALIDATION
# =================================================================================

assert semantic_similarity_matrix.shape == (
    1540,
    434
)

assert len(v3_semantic_df) == 668360

assert np.isfinite(
    semantic_similarity_matrix
).all()

print("\n" + "=" * 100)
print("✅ V3 SEMANTIC DATA RECOVERY COMPLETE")
print("=" * 100)

print(
    "Semantic matrix:",
    semantic_similarity_matrix.shape
)

print(
    "Semantic pairs:",
    len(v3_semantic_df)
)

print("\nReady to continue to V3 Cell 4.")

In [ ]:
# ============================================================
# V3 CELL 4 — LOCATE FULL 668,360-PAIR LABEL DATASET
# ============================================================

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Resume_Model_Project")

print("=" * 100)
print("V3 CELL 4 — LOCATING FULL 668,360-PAIR LABEL DATASET")
print("=" * 100)

# Keywords likely to appear in the completed labeling files
keywords = [
    "label", "llm", "rating", "score", "annot",
    "round", "batch", "pair", "668", "resume", "job"
]

# Extensions worth checking
valid_ext = {
    ".csv", ".pkl", ".pickle", ".parquet",
    ".json", ".jsonl", ".xlsx"
}

matches = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for fname in files:
        path = Path(root) / fname

        if path.suffix.lower() not in valid_ext:
            continue

        name_lower = fname.lower()

        # Keep files whose names contain relevant labeling terms
        if any(k in name_lower for k in keywords):
            size_mb = path.stat().st_size / (1024 ** 2)

            matches.append({
                "path": str(path),
                "name": fname,
                "size_mb": size_mb,
                "extension": path.suffix.lower()
            })

# Sort largest first — the completed label dataset is likely among the larger files
matches = sorted(matches, key=lambda x: x["size_mb"], reverse=True)

print(f"\nFound {len(matches)} potentially relevant files.\n")

for i, item in enumerate(matches[:100], 1):
    print(f"[{i:02d}] {item['size_mb']:10.2f} MB | "
          f"{item['extension']:8s} | {item['path']}")

print("\n" + "=" * 100)
print("NEXT STEP")
print("=" * 100)
print("Look for the file(s) corresponding to the completed 668,360-pair labeling.")
print("Paste the output here before we build the V3 training dataset.")

In [ ]:
# ============================================================
# V3 CELL 4R — FULL LABELING STATUS AUDIT
# ============================================================

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Resume_Model_Project")

print("=" * 100)
print("V3 CELL 4R — FULL 668,360 LABELING STATUS AUDIT")
print("=" * 100)

# Directories most likely to contain the completed labels
target_dirs = [
    PROJECT_DIR / "V3_Final",
    PROJECT_DIR / "V3",
    PROJECT_DIR / "V2_100k" / "LLM_Labeling",
    PROJECT_DIR / "V2_100k" / "Training",
    PROJECT_DIR / "V2_100k" / "Outputs",
]

matches = []

for base in target_dirs:
    if not base.exists():
        continue

    for root, dirs, files in os.walk(base):
        for fname in files:
            path = Path(root) / fname
            name = fname.lower()

            # Only likely labeling / master / round artifacts
            if any(k in name for k in [
                "label", "labeled", "master", "round",
                "progress", "unlabeled", "final"
            ]):
                matches.append(path)

# Remove duplicates and sort
matches = sorted(set(matches))

print(f"\nFound {len(matches)} relevant files.\n")

for i, path in enumerate(matches, 1):
    size_mb = path.stat().st_size / (1024**2)
    print(f"[{i:03d}] {size_mb:8.2f} MB | {path}")

print("\n" + "=" * 100)
print("CHECKING OBVIOUS MASTER / FINAL ARTIFACTS")
print("=" * 100)

# Explicitly check the suspicious V3 files
v3_final = PROJECT_DIR / "V3_Final" / "Outputs"

if v3_final.exists():
    for path in sorted(v3_final.glob("*")):
        print(path.name)

print("\n" + "=" * 100)
print("NEXT")
print("=" * 100)
print("Paste this output here.")
print("We will identify the REAL completed label source before training V3.")

In [ ]:
# ============================================================
# V3 CELL 5 — MERGE 325 COMPLETED LABELING ROUNDS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import json
import glob

print("=" * 100)
print("V3 CELL 5 — MERGING 325 COMPLETED LABELING ROUNDS")
print("=" * 100)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Resume_Model_Project")

V3_FINAL_DIR = PROJECT_DIR / "V3_Final"
V3_LABEL_DIR = V3_FINAL_DIR / "LLM_Labeling" / "Labeled_Rounds"

V3_OUTPUT_DIR = PROJECT_DIR / "V3" / "Outputs"
V3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Find the 325 completed label files
# ------------------------------------------------------------

label_files = sorted(
    V3_LABEL_DIR.glob("round_*_labels.csv"),
    key=lambda p: int(p.stem.split("_")[1])
)

print(f"\nLabel files found: {len(label_files)}")

if len(label_files) != 325:
    print("⚠️ WARNING: Expected exactly 325 round files.")

# Check round numbering
round_numbers = [
    int(p.stem.split("_")[1])
    for p in label_files
]

expected_rounds = set(range(1, 326))
actual_rounds = set(round_numbers)

missing_rounds = sorted(expected_rounds - actual_rounds)
extra_rounds = sorted(actual_rounds - expected_rounds)

print(f"Missing rounds: {missing_rounds}")
print(f"Unexpected rounds: {extra_rounds}")

if missing_rounds:
    raise RuntimeError(
        f"Missing V3 labeling rounds: {missing_rounds}"
    )

# ------------------------------------------------------------
# Inspect first file
# ------------------------------------------------------------

print("\nFirst round file:")
print(label_files[0])

sample_df = pd.read_csv(label_files[0])

print("\nSample columns:")
print(sample_df.columns.tolist())

print("\nSample shape:")
print(sample_df.shape)

print("\nSample rows:")
display(sample_df.head())

# ------------------------------------------------------------
# Load all rounds
# ------------------------------------------------------------

frames = []

for i, path in enumerate(label_files, 1):

    df = pd.read_csv(path)

    df["_source_round"] = i

    frames.append(df)

    if i % 25 == 0 or i == len(label_files):
        print(f"Loaded {i}/{len(label_files)} rounds")

# ------------------------------------------------------------
# Concatenate
# ------------------------------------------------------------

v3_labels = pd.concat(
    frames,
    ignore_index=True
)

print("\n" + "=" * 100)
print("MERGED LABEL DATASET")
print("=" * 100)

print("Rows:", len(v3_labels))
print("Columns:", len(v3_labels.columns))

print("\nColumns:")
print(v3_labels.columns.tolist())

# ------------------------------------------------------------
# Detect pair identifier
# ------------------------------------------------------------

possible_pair_cols = [
    "Pair_ID",
    "pair_id",
    "PairID",
    "pair",
]

pair_col = next(
    (c for c in possible_pair_cols if c in v3_labels.columns),
    None
)

if pair_col:
    print(f"\nPair identifier detected: {pair_col}")

    duplicate_pairs = v3_labels[pair_col].duplicated().sum()

    print("Duplicate Pair_ID rows:", duplicate_pairs)

else:
    print("\n⚠️ No Pair_ID column detected.")

    possible_index_cols = [
        ("Resume_Index", "Job_Index"),
        ("resume_index", "job_index"),
        ("ResumeID", "JobID")
    ]

    index_pair = next(
        (
            cols for cols in possible_index_cols
            if all(c in v3_labels.columns for c in cols)
        ),
        None
    )

    if index_pair:
        print(
            "Composite pair identifier detected:",
            index_pair
        )

        duplicate_pairs = v3_labels.duplicated(
            subset=list(index_pair)
        ).sum()

        print("Duplicate pair rows:", duplicate_pairs)

# ------------------------------------------------------------
# Detect label column
# ------------------------------------------------------------

possible_label_cols = [
    "Label",
    "label",
    "Score",
    "score",
    "Rating",
    "rating",
    "Match_Label",
    "match_label"
]

label_col = next(
    (c for c in possible_label_cols if c in v3_labels.columns),
    None
)

if label_col is None:
    raise RuntimeError(
        "Could not automatically identify the label column. "
        "Inspect the printed columns above."
    )

print(f"\nLabel column detected: {label_col}")

print("\nLabel distribution:")
print(v3_labels[label_col].value_counts(dropna=False).sort_index())

print("\nMissing labels:")
print(v3_labels[label_col].isna().sum())

# ------------------------------------------------------------
# Save merged dataset
# ------------------------------------------------------------

csv_path = V3_OUTPUT_DIR / "v3_labels_568210_merged.csv"
pkl_path = V3_OUTPUT_DIR / "v3_labels_568210_merged.pkl"

v3_labels.to_csv(csv_path, index=False)
v3_labels.to_pickle(pkl_path)

metadata = {
    "expected_pairs": 568210,
    "rounds": 325,
    "actual_rows": int(len(v3_labels)),
    "label_column": label_col,
    "pair_column": pair_col,
    "duplicate_rows": int(
        v3_labels[pair_col].duplicated().sum()
    ) if pair_col else None
}

metadata_path = V3_OUTPUT_DIR / "v3_label_merge_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("\n" + "=" * 100)
print("✅ V3 LABEL MERGE COMPLETE")
print("=" * 100)

print(f"CSV : {csv_path}")
print(f"PKL : {pkl_path}")
print(f"META: {metadata_path}")

print("\nExpected rows:", 568210)
print("Actual rows  :", len(v3_labels))

if len(v3_labels) == 568210:
    print("✅ EXACT 568,210 V3 pairs recovered.")
else:
    print("⚠️ Row count differs from 568,210 — we will investigate before training.")

In [ ]:
# ============================================================
# V3 CELL 6 — IDENTIFY THE 1,250 UNACCOUNTED PAIRS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Resume_Model_Project")

# ------------------------------------------------------------
# Load completed V3 labels
# ------------------------------------------------------------

labels_path = (
    PROJECT_DIR
    / "V3"
    / "Outputs"
    / "v3_labels_568210_merged.csv"
)

v3_labels = pd.read_csv(labels_path)

print("Completed V3 labels:", len(v3_labels))

# ------------------------------------------------------------
# Load original V3 queue
# ------------------------------------------------------------

queue_path = (
    PROJECT_DIR
    / "V3_Final"
    / "Outputs"
    / "v3_unlabeled_pairs_568210.csv"
)

queue = pd.read_csv(queue_path)

print("Original V3 queue:", len(queue))

print("\nQueue columns:")
print(queue.columns.tolist())

# ------------------------------------------------------------
# Compare Pair_ID sets
# ------------------------------------------------------------

label_ids = set(v3_labels["Pair_ID"].astype(int))
queue_ids = set(queue["Pair_ID"].astype(int))

missing_ids = sorted(queue_ids - label_ids)
extra_ids = sorted(label_ids - queue_ids)

print("\n" + "=" * 100)
print("PAIR COVERAGE CHECK")
print("=" * 100)

print("Queue Pair_IDs :", len(queue_ids))
print("Label Pair_IDs :", len(label_ids))
print("Missing IDs    :", len(missing_ids))
print("Extra IDs      :", len(extra_ids))

print("\nFirst 50 missing Pair_IDs:")
print(missing_ids[:50])

print("\nLast 50 missing Pair_IDs:")
print(missing_ids[-50:])

# ------------------------------------------------------------
# Recover missing rows from queue
# ------------------------------------------------------------

missing_pairs = queue[
    queue["Pair_ID"].astype(int).isin(missing_ids)
].copy()

print("\n" + "=" * 100)
print("MISSING PAIRS")
print("=" * 100)

print("Rows:", len(missing_pairs))

display(missing_pairs.head(20))

# ------------------------------------------------------------
# Save missing-pair audit
# ------------------------------------------------------------

output_dir = PROJECT_DIR / "V3" / "Outputs"

missing_csv = output_dir / "v3_missing_1250_pairs.csv"
missing_pkl = output_dir / "v3_missing_1250_pairs.pkl"

missing_pairs.to_csv(missing_csv, index=False)
missing_pairs.to_pickle(missing_pkl)

print("\nSaved:")
print(missing_csv)
print(missing_pkl)

# ------------------------------------------------------------
# Check whether all completed labels belong to V3 queue
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL COVERAGE")
print("=" * 100)

if not extra_ids:
    print("✅ Every completed V3 label belongs to the original V3 queue.")

if len(missing_ids) == 1250:
    print("⚠️ Exactly 1,250 V3 queue pairs have no recovered label.")
else:
    print(f"⚠️ Missing count is {len(missing_ids)}, not 1,250.")

print("\nDO NOT relabel yet.")
print("We first inspect these 1,250 Pair_IDs and determine why they were omitted.")

In [ ]:
# ============================================================
# V3 CELL 7 — DIAGNOSE THE 1,250 MISSING LABELS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Resume_Model_Project")

LABEL_DIR = (
    PROJECT_DIR
    / "V3_Final"
    / "LLM_Labeling"
    / "Labeled_Rounds"
)

QUEUE_PATH = (
    PROJECT_DIR
    / "V3_Final"
    / "Outputs"
    / "v3_unlabeled_pairs_568210.csv"
)

MERGED_LABEL_PATH = (
    PROJECT_DIR
    / "V3"
    / "Outputs"
    / "v3_labels_568210_merged.csv"
)

# ------------------------------------------------------------
# Load queue + completed labels
# ------------------------------------------------------------

queue = pd.read_csv(QUEUE_PATH)
labels = pd.read_csv(MERGED_LABEL_PATH)

queue["Pair_ID"] = queue["Pair_ID"].astype(int)
labels["Pair_ID"] = labels["Pair_ID"].astype(int)

label_ids = set(labels["Pair_ID"])
missing_ids = sorted(set(queue["Pair_ID"]) - label_ids)

missing_df = queue[
    queue["Pair_ID"].isin(missing_ids)
].copy()

print("=" * 100)
print("V3 CELL 7 — MISSING LABEL DIAGNOSIS")
print("=" * 100)

print("Missing pairs:", len(missing_df))

# ------------------------------------------------------------
# 1. Distribution by Job_Index
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MISSING PAIRS BY JOB_INDEX")
print("=" * 100)

job_counts = (
    missing_df
    .groupby("Job_Index")
    .size()
    .sort_values(ascending=False)
)

print(job_counts)

# ------------------------------------------------------------
# 2. Distribution by Resume_Index
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MISSING PAIRS — RESUME INDEX RANGE")
print("=" * 100)

print(
    missing_df[
        ["Job_Index", "Resume_Index", "Pair_ID"]
    ].head(20)
)

print("\nResume index min/max:")
print(
    missing_df.groupby("Job_Index")["Resume_Index"]
    .agg(["min", "max", "count"])
)

# ------------------------------------------------------------
# 3. Inspect ALL 325 label files
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ROUND SIZE AUDIT")
print("=" * 100)

round_files = sorted(
    LABEL_DIR.glob("round_*_labels.csv"),
    key=lambda p: int(p.stem.split("_")[1])
)

round_info = []

for path in round_files:

    round_num = int(path.stem.split("_")[1])

    df = pd.read_csv(path)

    ids = df["Pair_ID"].astype(int)

    round_info.append({
        "Round": round_num,
        "Rows": len(df),
        "Min_Pair_ID": ids.min() if len(ids) else np.nan,
        "Max_Pair_ID": ids.max() if len(ids) else np.nan,
        "Unique_IDs": ids.nunique()
    })

round_audit = pd.DataFrame(round_info)

print("\nRows per round:")
print(round_audit["Rows"].value_counts().sort_index())

print("\nRounds with unusual row counts:")

mode_rows = round_audit["Rows"].mode()[0]

unusual = round_audit[
    round_audit["Rows"] != mode_rows
]

display(unusual)

# ------------------------------------------------------------
# 4. Which rounds contain missing Pair_IDs?
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("MISSING PAIRS BY ROUND")
print("=" * 100)

missing_set = set(missing_ids)

round_missing = []

for path in round_files:

    round_num = int(path.stem.split("_")[1])

    df = pd.read_csv(path)

    ids = set(df["Pair_ID"].astype(int))

    # This will normally be zero because missing IDs aren't present.
    # We use queue-position mapping below to identify their expected rounds.
    round_missing.append({
        "Round": round_num,
        "Rows": len(df)
    })

# ------------------------------------------------------------
# 5. Map missing pairs to their position in the ORIGINAL QUEUE
# ------------------------------------------------------------

queue = queue.reset_index(drop=True)

queue["Queue_Position"] = np.arange(len(queue))

missing_with_position = queue[
    queue["Pair_ID"].isin(missing_set)
].copy()

print("\n" + "=" * 100)
print("MISSING PAIR QUEUE POSITIONS")
print("=" * 100)

print(
    missing_with_position[
        ["Queue_Position", "Job_Index", "Resume_Index", "Pair_ID"]
    ].head(30)
)

print("\nQueue position range:")
print(
    missing_with_position["Queue_Position"].min(),
    "to",
    missing_with_position["Queue_Position"].max()
)

# ------------------------------------------------------------
# 6. Estimate round from 1,750-pair original batching
# ------------------------------------------------------------

BATCH_SIZE = 1750

missing_with_position["Expected_Round_1750"] = (
    missing_with_position["Queue_Position"] // BATCH_SIZE
) + 1

print("\n" + "=" * 100)
print("EXPECTED ROUND DISTRIBUTION")
print("=" * 100)

print(
    missing_with_position["Expected_Round_1750"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 7. Save complete diagnostic
# ------------------------------------------------------------

output_path = (
    PROJECT_DIR
    / "V3"
    / "Outputs"
    / "v3_missing_1250_diagnostic.csv"
)

missing_with_position.to_csv(
    output_path,
    index=False
)

round_audit_path = (
    PROJECT_DIR
    / "V3"
    / "Outputs"
    / "v3_round_size_audit.csv"
)

round_audit.to_csv(
    round_audit_path,
    index=False
)

print("\n" + "=" * 100)
print("✅ DIAGNOSTIC COMPLETE")
print("=" * 100)

print("Missing diagnostic:")
print(output_path)

print("Round audit:")
print(round_audit_path)

In [ ]:
# ============================================================
# V3 CELL 8R — PREPARE MISSING 1,250-PAIR RECOVERY BATCH
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 100)
print("V3 CELL 8R — PREPARING 1,250 MISSING PAIRS")
print("=" * 100)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Resume_Model_Project")

V3_OUTPUT_DIR = PROJECT_DIR / "V3" / "Outputs"
V2_INPUT_DIR = PROJECT_DIR / "V2_100k" / "Inputs"

missing_path = V3_OUTPUT_DIR / "v3_missing_1250_pairs.csv"

candidate_path = V2_INPUT_DIR / "candidate_df.pkl"
job_path = V2_INPUT_DIR / "entry_job_df.pkl"

# ------------------------------------------------------------
# LOAD MISSING PAIRS
# ------------------------------------------------------------

if not missing_path.exists():
    raise FileNotFoundError(
        f"Missing-pair file not found:\n{missing_path}"
    )

missing_pairs = pd.read_csv(missing_path)

print("Missing pairs loaded:", len(missing_pairs))

assert len(missing_pairs) == 1250
assert missing_pairs["Pair_ID"].is_unique

# ------------------------------------------------------------
# LOAD SOURCE DATA DIRECTLY
# ------------------------------------------------------------

if "candidate_df" not in globals():
    print("\nLoading candidate_df from Drive...")
    candidate_df = pd.read_pickle(candidate_path)

if "entry_job_df" not in globals():
    print("Loading entry_job_df from Drive...")
    entry_job_df = pd.read_pickle(job_path)

print("\nCandidates:", len(candidate_df))
print("Jobs:", len(entry_job_df))

# ------------------------------------------------------------
# VERIFY SOURCE COUNTS
# ------------------------------------------------------------

assert len(candidate_df) == 1540
assert len(entry_job_df) == 434

print("✅ Candidate/job source artifacts verified.")

print("\nCandidate columns:")
print(candidate_df.columns.tolist())

print("\nJob columns:")
print(entry_job_df.columns.tolist())

# ------------------------------------------------------------
# VERIFY REQUIRED COLUMNS
# ------------------------------------------------------------

assert "resume_text" in candidate_df.columns
assert "job_text" in entry_job_df.columns

# ------------------------------------------------------------
# BUILD RECOVERY DATASET
# ------------------------------------------------------------

recovery = missing_pairs.copy()

recovery["Resume_Text"] = recovery["Resume_Index"].map(
    lambda i: candidate_df.iloc[int(i)]["resume_text"]
)

recovery["Job_Text"] = recovery["Job_Index"].map(
    lambda i: entry_job_df.iloc[int(i)]["job_text"]
)

# Add job title when available
if "Title" in entry_job_df.columns:
    recovery["Job_Title"] = recovery["Job_Index"].map(
        lambda i: entry_job_df.iloc[int(i)]["Title"]
    )

# Add experience level when available
if "ExperienceLevel" in entry_job_df.columns:
    recovery["ExperienceLevel"] = recovery["Job_Index"].map(
        lambda i: entry_job_df.iloc[int(i)]["ExperienceLevel"]
    )

# ------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RECOVERY BATCH VALIDATION")
print("=" * 100)

print("Rows:", len(recovery))
print("Unique Pair_IDs:", recovery["Pair_ID"].nunique())

print(
    "Missing Resume_Text:",
    recovery["Resume_Text"].isna().sum()
)

print(
    "Missing Job_Text:",
    recovery["Job_Text"].isna().sum()
)

assert len(recovery) == 1250
assert recovery["Pair_ID"].is_unique
assert recovery["Resume_Text"].notna().all()
assert recovery["Job_Text"].notna().all()

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

csv_path = (
    V3_OUTPUT_DIR
    / "v3_recovery_round_297_1250_input.csv"
)

pkl_path = (
    V3_OUTPUT_DIR
    / "v3_recovery_round_297_1250_input.pkl"
)

recovery.to_csv(csv_path, index=False)
recovery.to_pickle(pkl_path)

print("\nSaved:")
print(csv_path)
print(pkl_path)

# ------------------------------------------------------------
# PREVIEW
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RECOVERY BATCH PREVIEW")
print("=" * 100)

preview_cols = [
    "Pair_ID",
    "Job_Index",
    "Resume_Index"
]

if "Job_Title" in recovery.columns:
    preview_cols.append("Job_Title")

display(recovery[preview_cols].head(20))

print("\n" + "=" * 100)
print("✅ 1,250-PAIR RECOVERY BATCH READY")
print("=" * 100)

In [ ]:
# ============================================================
# V3 CELL 9 — CREATE 1,250-PAIR RECOVERY LABELING PROMPT
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Resume_Model_Project")

V3_OUTPUT_DIR = PROJECT_DIR / "V3" / "Outputs"

input_path = V3_OUTPUT_DIR / "v3_recovery_round_297_1250_input.csv"

recovery = pd.read_csv(input_path)

assert len(recovery) == 1250
assert recovery["Pair_ID"].is_unique

print("=" * 100)
print("V3 CELL 9 — CREATING RECOVERY LABELING PROMPT")
print("=" * 100)

# ------------------------------------------------------------
# Labeling rubric
# ------------------------------------------------------------

instructions = """
V3 RESUME-JOB MATCH LABELING

For each resume-job pair, assign exactly one relevance label:

0 = Weak
1 = Moderate
2 = Strong

Consider the overall suitability of the resume for the job, including:
- Required skills
- Relevant technical/domain experience
- Responsibilities and project experience
- Education/background where relevant
- Overall role alignment

Do not invent experience that is not present in the resume.

Return ONLY the labels in the requested order.
"""

# ------------------------------------------------------------
# Build prompt
# ------------------------------------------------------------

parts = [instructions.strip(), "\n\n" + "=" * 100]

for i, row in recovery.iterrows():

    parts.append(
        f"""
PAIR {i + 1}/1250
Pair_ID: {int(row["Pair_ID"])}
Job_Index: {int(row["Job_Index"])}
Resume_Index: {int(row["Resume_Index"])}

JOB TITLE:
{row.get("Job_Title", "N/A")}

EXPERIENCE LEVEL:
{row.get("ExperienceLevel", "N/A")}

JOB:
{row["Job_Text"]}

RESUME:
{row["Resume_Text"]}

LABEL:
0 = Weak
1 = Moderate
2 = Strong

"""
    )

# ------------------------------------------------------------
# Final output section
# ------------------------------------------------------------

parts.append(
    """
============================================================
FINAL RESPONSE FORMAT
============================================================

Return exactly 1250 integer labels, in the SAME ORDER as the
pairs above.

Example:

1 0 2 1 0 1 ...

Do not include explanations.
Do not include Pair_IDs.
Do not reorder anything.
"""
)

prompt = "\n".join(parts)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

prompt_path = (
    V3_OUTPUT_DIR
    / "v3_recovery_round_297_1250_labeling_prompt.md"
)

with open(prompt_path, "w", encoding="utf-8") as f:
    f.write(prompt)

print("✅ Prompt created.")
print(f"Pairs: {len(recovery)}")
print(f"Prompt size: {len(prompt):,} characters")
print(f"Saved:\n{prompt_path}")

# ------------------------------------------------------------
# Show beginning + ending
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PROMPT PREVIEW")
print("=" * 100)

print(prompt[:5000])

print("\n...\n")

print(prompt[-1500:])

print("\n" + "=" * 100)
print("✅ READY FOR THE 1,250-PAIR LABELING PASS")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 10 — VALIDATING RECOVERY LABELS")
print("=" * 100)

# PASTE THE ENTIRE LLM RESPONSE BETWEEN THE TRIPLE QUOTES
raw_labels = """
0 1 1 1 1 1 2 1 0 1 0 2 0 0 1 0 0 0 0 1 0 1 0 1 1 1 1 2 1 1 1 1 0 0 2 1 1 2 0 2 1 0 1 0 0 1 0 1 1 2 1 0 1 1 1 0 1 0 1 1 0 2 0 0 1 0 0 0 1 2 1 1 1 0 1 1 1 0 2 2 0 1 1 1 1 0 1 1 2 1 0 1 1 1 2 0 1 2 1 0 1 0 1 2 1 0 1 1 1 0 0 0 2 2 1 1 2 1 1 1 0 2 0 1 1 0 0 1 1 0 1 1 0 2 1 1 1 0 0 0 1 0 1 0 1 1 2 1 1 1 2 1 2 1 2 1 1 1 1 2 1 2 1 2 1 1 1 1 2 1 1 1 1 2 1 2 2 1 1 1 1 2 2 1 1 1 1 2 1 2 2 2 1 1 0 0 0 1 0 1 0 1 0 0 1 0 0 2 1 2 1 0 1 2 1 1 1 0 1 1 2 1 2 1 1 1 0 1 1 2 1 1 1 1 1 2 1 1 1 1 1 1 0 2 1 0 1 1 1 2 0 1 2 2 1 1 1 0 2 2 0 0 0 1 2 1 1 0 0 1 1 1 2 0 1 1 0 1 1 1 1 0 2 1 0 1 1 0 1 0 1 1 1 0 1 1 2 1 0 0 1 1 0 1 0 1 1 0 2 0 1 1 0 1 2 1 1 1 1 0 1 0 0 2 2 2 1 0 1 2 1 1 1 1 2 2 2 0 2 2 1 0 2 1 0 0 2 1 1 0 1 0 0 2 1 0 1 2 0 2 2 1 1 0 0 1 0 0 1 2 1 1 0 0 2 0 2 0 1 1 1 1 1 2 1 1 2 0 1 1 0 0 1 1 2 1 0 0 0 0 1 0 0 1 2 2 2 1 1 1 0 1 1 1 1 0 1 1 1 1 1 0 1 2 1 1 2 1 2 2 1 2 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 2 1 1 1 1 2 1 2 1 1 1 2 2 1 1 1 2 1 1 1 2 1 2 1 1 1 1 1 0 1 0 1 1 1 2 0 0 2 2 0 0 1 1 1 1 2 2 2 1 0 2 1 0 0 0 1 1 1 0 0 1 1 0 1 1 2 0 0 2 0 1 0 0 1 2 0 2 1 1 2 2 1 2 2 1 0 2 0 0 2 1 2 1 2 1 0 1 1 1 1 1 1 2 1 1 2 1 1 0 0 1 1 1 0 1 1 2 0 2 0 1 1 0 2 2 1 0 2 0 0 2 1 0 0 0 0 0 1 0 0 0 1 1 2 2 1 0 0 0 0 1 2 2 0 0 2 2 0 1 1 0 0 1 0 1 2 0 1 1 0 1 1 1 0 0 1 1 0 0 0 1 1 1 1 1 1 0 1 2 0 2 2 1 1 1 1 0 0 0 1 0 2 2 0 1 1 0 1 1 1 0 1 2 1 1 1 0 1 1 0 1 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 1 1 1 1 1 0 1 1 1 2 2 0 1 1 1 2 1 0 1 0 1 1 2 1 0 2 1 2 2 0 1 0 0 1 0 1 1 0 1 1 1 1 1 1 1 0 0 2 0 0 1 1 2 1 1 1 0 1 0 1 1 1 1 0 2 0 0 1 0 2 0 0 1 0 0 0 0 1 1 2 1 0 1 0 0 1 0 1 1 1 1 0 1 0 1 0 1 1 1 2 0 1 1 0 1 1 1 1 1 1 0 2 0 1 1 2 0 2 1 1 1 0 1 1 1 0 1 1 1 0 1 2 2 0 2 2 1 0 2 1 1 0 1 0 2 0 1 2 2 1 1 1 1 0 1 0 1 0 0 1 1 0 1 0 1 2 1 2 1 1 0 0 0 0 2 1 1 2 2 2 1 0 1 1 0 0 1 0 0 1 0 1 1 0 0 0 1 1 1 0 1 1 2 1 1 1 1 1 1 1 0 1 1 0 1 1 2 2 2 0 1 1 1 2 1 1 1 1 0 1 2 2 1 1 0 2 1 0 1 0 1 1 0 2 1 1 0 0 0 0 1 1 1 1 0 1 2 0 2 1 1 1 2 2 1 2 1 1 2 1 1 2 2 2 2 1 1 2 1 2 1 1 1 2 2 1 1 2 2 2 2 1 2 2 2 0 1 0 0 2 1 0 1 0 1 1 0 0 1 1 1 0 1 1 1 1 1 1 2 1 1 1 1 0 1 1 1 0 0 2 1 2 1 0 2 0 1 0 0 2 1 1 1 1 1 0 1 2 0 1 2 0 2 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 2 1 1 2 1 1 1 1 2 2 1 2 1 2 1 2 2 1 1 1 2 2 1 2 1 2 1 2 1 2 2 1 1 2 1 1 1 1 1 1 1 2 1 1 1 1 2 2 1 1 1 1 2 1 1 2 1 2 1 2 1 1 1 2 1 1 1 1 1 0 2 1 1 1 1 1 1 2 1 0 1 2 1 0 2 0 0 2 0 1 1 1 2 0 0 1 1 1 0 0 1 0 1 1 1 1 1 1 2 0 0 1 1 1 0 0 2 0 1 1 1 2 1 1 2 1 1 1 1 1 1 2 2 1 1 2 1 1 1 2 1 1 1 2 0 0 1 1 0 2 1 1 1 1 0 0 1 1 1 1 1 0 1 0 2 0 1 1 0 0 0 0 1 0 1 1 1 1 2 1 1 1 1 1 2 1 2 1 1 1 1 1 1 2 1 2 1 1
"""

labels = raw_labels.split()

# Basic validation
print("Labels received:", len(labels))

# Check that every item is an integer 0/1/2
try:
    labels_int = [int(x) for x in labels]
except ValueError:
    raise ValueError("❌ Some output is not an integer label.")

invalid = [x for x in labels_int if x not in [0, 1, 2]]

if invalid:
    raise ValueError(f"❌ Invalid labels found: {invalid[:20]}")

if len(labels_int) != 1250:
    raise ValueError(
        f"❌ Expected 1250 labels, but received {len(labels_int)}"
    )

print("✅ Exactly 1250 labels received.")
print("✅ All labels are valid (0, 1, 2).")

from collections import Counter

print("\nLabel distribution:")
print(Counter(labels_int))

In [ ]:
print("=" * 100)
print("V3 CELL 11 — RESTORING THE 1,250 MISSING LABELS")
print("=" * 100)

import os
import pandas as pd
import pickle
import json

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
RECOVERY_INPUT = os.path.join(
    V3_OUTPUT_DIR,
    "v3_recovery_round_297_1250_input.csv"
)

EXISTING_LABELS = os.path.join(
    V3_OUTPUT_DIR,
    "v3_labels_568210_merged.csv"
)

RECOVERY_LABELS_OUT = os.path.join(
    V3_OUTPUT_DIR,
    "v3_recovery_round_297_1250_labels.csv"
)

COMPLETE_LABELS_OUT = os.path.join(
    V3_OUTPUT_DIR,
    "v3_labels_568210_complete.csv"
)

COMPLETE_LABELS_PKL = os.path.join(
    V3_OUTPUT_DIR,
    "v3_labels_568210_complete.pkl"
)

# ------------------------------------------------------------------
# Load the 1,250 missing pairs
# ------------------------------------------------------------------
recovery_df = pd.read_csv(RECOVERY_INPUT)

print("Recovery pairs loaded:", len(recovery_df))

if len(recovery_df) != 1250:
    raise ValueError(
        f"❌ Expected 1250 recovery pairs, found {len(recovery_df)}"
    )

# ------------------------------------------------------------------
# Your validated labels from Cell 10
# ------------------------------------------------------------------
# labels_int should already exist from Cell 10.
if "labels_int" not in globals():
    raise RuntimeError(
        "❌ labels_int not found. Run Cell 10 first."
    )

if len(labels_int) != 1250:
    raise ValueError(
        f"❌ Expected 1250 labels, found {len(labels_int)}"
    )

# ------------------------------------------------------------------
# Attach labels by EXACT SAME ORDER
# ------------------------------------------------------------------
recovery_labels_df = pd.DataFrame({
    "Pair_ID": recovery_df["Pair_ID"].astype(int),
    "Label": labels_int
})

# Extra safety checks
if recovery_labels_df["Pair_ID"].duplicated().any():
    raise ValueError("❌ Duplicate Pair_IDs found in recovery pairs.")

if recovery_labels_df["Pair_ID"].isna().any():
    raise ValueError("❌ Missing Pair_ID found.")

# ------------------------------------------------------------------
# Save recovery labels separately
# ------------------------------------------------------------------
recovery_labels_df.to_csv(
    RECOVERY_LABELS_OUT,
    index=False
)

with open(RECOVERY_LABELS_OUT.replace(".csv", ".pkl"), "wb") as f:
    pickle.dump(recovery_labels_df, f)

print("✅ Recovery labels saved:")
print(RECOVERY_LABELS_OUT)

# ------------------------------------------------------------------
# Load original 566,960 labels
# ------------------------------------------------------------------
existing_df = pd.read_csv(EXISTING_LABELS)

print("\nExisting labels:", len(existing_df))

# ------------------------------------------------------------------
# Combine existing + recovered
# ------------------------------------------------------------------
complete_df = pd.concat(
    [existing_df[["Pair_ID", "Label"]],
     recovery_labels_df[["Pair_ID", "Label"]]],
    ignore_index=True
)

# ------------------------------------------------------------------
# Validate
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("FINAL VALIDATION")
print("=" * 100)

print("Total rows:", len(complete_df))
print("Unique Pair_IDs:", complete_df["Pair_ID"].nunique())
print("Duplicate Pair_IDs:", complete_df["Pair_ID"].duplicated().sum())
print("Missing labels:", complete_df["Label"].isna().sum())

expected_total = 568210

if len(complete_df) != expected_total:
    raise ValueError(
        f"❌ Expected {expected_total} rows, found {len(complete_df)}"
    )

if complete_df["Pair_ID"].nunique() != expected_total:
    raise ValueError("❌ Pair_ID count is not 568,210.")

if complete_df["Pair_ID"].duplicated().sum() != 0:
    raise ValueError("❌ Duplicate Pair_IDs detected.")

if complete_df["Label"].isna().sum() != 0:
    raise ValueError("❌ Missing labels detected.")

# Confirm valid label values
invalid_final = complete_df[
    ~complete_df["Label"].isin([0, 1, 2])
]

if len(invalid_final) > 0:
    raise ValueError(
        f"❌ Found {len(invalid_final)} invalid labels."
    )

# Sort into Pair_ID order
complete_df = complete_df.sort_values(
    "Pair_ID"
).reset_index(drop=True)

# ------------------------------------------------------------------
# Save complete dataset
# ------------------------------------------------------------------
complete_df.to_csv(
    COMPLETE_LABELS_OUT,
    index=False
)

with open(COMPLETE_LABELS_PKL, "wb") as f:
    pickle.dump(complete_df, f)

# ------------------------------------------------------------------
# Final statistics
# ------------------------------------------------------------------
print("\n✅ COMPLETE V3 LABEL DATASET RESTORED")
print(f"Rows: {len(complete_df):,}")
print(f"Unique Pair_IDs: {complete_df['Pair_ID'].nunique():,}")

print("\nLabel distribution:")
print(complete_df["Label"].value_counts().sort_index())

print("\nSaved:")
print(COMPLETE_LABELS_OUT)
print(COMPLETE_LABELS_PKL)

print("\n" + "=" * 100)
print("🎉 568,210 / 568,210 LABELS RESTORED")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 12R — FINAL DATA INTEGRITY CHECK")
print("=" * 100)

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
QUEUE_PATH = os.path.abspath(
    os.path.join(
        V3_DIR,
        "..",
        "V3_Final",
        "Outputs",
        "v3_unlabeled_pairs_568210.csv"
    )
)

LABEL_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_labels_568210_complete.csv"
)

SEMANTIC_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_pair_scores.csv"
)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
queue_df = pd.read_csv(QUEUE_PATH)
labels_df = pd.read_csv(LABEL_PATH)
semantic_df = pd.read_csv(SEMANTIC_PATH)

print("Queue rows:", len(queue_df))
print("Label rows:", len(labels_df))
print("Semantic rows:", len(semantic_df))

print("\nSemantic columns:")
print(list(semantic_df.columns))

# ------------------------------------------------------------
# Basic required-column checks
# ------------------------------------------------------------
required_queue = {"Pair_ID", "Resume_Index", "Job_Index"}
required_labels = {"Pair_ID", "Label"}
required_semantic = {
    "Resume_Index",
    "Job_Index",
    "Semantic_Similarity"
}

if not required_queue.issubset(queue_df.columns):
    raise ValueError(
        f"❌ Queue missing columns: "
        f"{required_queue - set(queue_df.columns)}"
    )

if not required_labels.issubset(labels_df.columns):
    raise ValueError(
        f"❌ Labels missing columns: "
        f"{required_labels - set(labels_df.columns)}"
    )

if not required_semantic.issubset(semantic_df.columns):
    raise ValueError(
        f"❌ Semantic file missing columns: "
        f"{required_semantic - set(semantic_df.columns)}"
    )

# ------------------------------------------------------------
# Pair_ID checks
# ------------------------------------------------------------
queue_ids = set(queue_df["Pair_ID"].astype(int))
label_ids = set(labels_df["Pair_ID"].astype(int))

missing_from_labels = queue_ids - label_ids
extra_in_labels = label_ids - queue_ids

print("\nPair_ID coverage:")
print("Queue Pair_IDs:", len(queue_ids))
print("Label Pair_IDs:", len(label_ids))
print("Missing labels:", len(missing_from_labels))
print("Extra labels:", len(extra_in_labels))

if missing_from_labels:
    raise ValueError(
        f"❌ {len(missing_from_labels)} queue pairs are missing labels."
    )

if extra_in_labels:
    raise ValueError(
        f"❌ {len(extra_in_labels)} label Pair_IDs are not in the queue."
    )

# ------------------------------------------------------------
# Duplicate checks
# ------------------------------------------------------------
if queue_df["Pair_ID"].duplicated().any():
    raise ValueError("❌ Duplicate Pair_IDs in queue.")

if labels_df["Pair_ID"].duplicated().any():
    raise ValueError("❌ Duplicate Pair_IDs in labels.")

# Semantic uniqueness is checked using Resume_Index + Job_Index
semantic_dupes = semantic_df.duplicated(
    subset=["Resume_Index", "Job_Index"]
).sum()

if semantic_dupes > 0:
    raise ValueError(
        f"❌ Found {semantic_dupes} duplicate semantic pairs."
    )

# ------------------------------------------------------------
# Label validity
# ------------------------------------------------------------
if labels_df["Label"].isna().any():
    raise ValueError("❌ Missing labels detected.")

if not labels_df["Label"].isin([0, 1, 2]).all():
    raise ValueError("❌ Invalid label values detected.")

# ------------------------------------------------------------
# Build composite pair keys
# ------------------------------------------------------------
queue_pairs = set(
    zip(
        queue_df["Resume_Index"].astype(int),
        queue_df["Job_Index"].astype(int)
    )
)

semantic_pairs = set(
    zip(
        semantic_df["Resume_Index"].astype(int),
        semantic_df["Job_Index"].astype(int)
    )
)

missing_semantic = queue_pairs - semantic_pairs
extra_semantic = semantic_pairs - queue_pairs

print("\nSemantic pair coverage:")
print("Queue Resume/Job pairs:", len(queue_pairs))
print("Semantic Resume/Job pairs:", len(semantic_pairs))
print("Queue pairs missing semantic score:", len(missing_semantic))
print("Extra semantic pairs:", len(extra_semantic))

if missing_semantic:
    raise ValueError(
        f"❌ {len(missing_semantic)} queue pairs are missing semantic scores."
    )

# Extra semantic pairs are EXPECTED because semantic scores contain
# all 668,360 possible resume-job combinations, while the V3 queue
# contains only the 568,210 pairs not used in V2.

# ------------------------------------------------------------
# Semantic score validity
# ------------------------------------------------------------
if semantic_df["Semantic_Similarity"].isna().any():
    raise ValueError("❌ Missing semantic similarity values detected.")

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("✅ V3 DATA INTEGRITY CHECK PASSED")
print("=" * 100)

print(f"✅ {len(queue_ids):,} V3 labeled pairs")
print(f"✅ {len(semantic_df):,} total semantic pair scores")
print("✅ All V3 queue pairs have semantic scores")
print("✅ Pair_ID mapping is consistent")
print("✅ No duplicate Pair_IDs")
print("✅ No duplicate semantic Resume/Job pairs")
print("✅ No missing labels")
print("✅ All labels are valid")

print("\n" + "=" * 100)
print("🚀 READY TO BUILD THE V3 TRAINING DATASET")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 13 — LOCATING V2 RICH-6 FEATURE ARTIFACT")
print("=" * 100)

import os
import glob
import pandas as pd

# ------------------------------------------------------------
# Make sure V2 paths exist
# ------------------------------------------------------------
if "PROJECT_DIR" not in globals():
    PROJECT_DIR = "/content/drive/MyDrive/Resume_Model_Project"

V2_DIR = os.path.join(PROJECT_DIR, "V2_100k")
V2_OUTPUT_DIR = os.path.join(V2_DIR, "Outputs")

print("V2 output directory:")
print(V2_OUTPUT_DIR)

if not os.path.exists(V2_OUTPUT_DIR):
    raise FileNotFoundError(
        f"❌ V2 output directory not found:\n{V2_OUTPUT_DIR}"
    )

# ------------------------------------------------------------
# List candidate files
# ------------------------------------------------------------
all_files = []

for root, dirs, files in os.walk(V2_OUTPUT_DIR):
    for file in files:
        path = os.path.join(root, file)

        if any(key in file.lower() for key in [
            "feature",
            "train",
            "pair",
            "rich",
            "dataset"
        ]):
            all_files.append(path)

print("\nCandidate V2 files:")
for path in all_files:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{size_mb:8.2f} MB  {path}")

# ------------------------------------------------------------
# Inspect CSV headers for Rich-6 columns
# ------------------------------------------------------------
rich6_cols = {
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity"
}

matches = []

for path in all_files:

    if not path.lower().endswith(".csv"):
        continue

    try:
        header = pd.read_csv(path, nrows=0)

        cols = set(header.columns)

        if rich6_cols.issubset(cols):
            matches.append((path, header.columns.tolist()))

    except Exception as e:
        print(f"⚠️ Could not inspect {path}: {e}")

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("RICH-6 CANDIDATES")
print("=" * 100)

if not matches:
    print("⚠️ No CSV containing all Rich-6 columns was found.")
    print("\nWe will locate the original V2 feature-generation code next.")
else:
    for path, cols in matches:
        print("\n✅ Possible Rich-6 artifact:")
        print(path)
        print("\nColumns:")
        print(cols)

print("\n" + "=" * 100)
print("CELL 13 COMPLETE")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 14 — LOCATING FULL-POOL V1/V2 FEATURE MATRICES")
print("=" * 100)

import os
import glob
import pandas as pd
import pickle

PROJECT_DIR = "/content/drive/MyDrive/Resume_Model_Project"

# Search these likely project locations
search_roots = [
    os.path.join(PROJECT_DIR, "V2_Development"),
    os.path.join(PROJECT_DIR, "V2_100k"),
    os.path.join(PROJECT_DIR, "V3"),
    os.path.join(PROJECT_DIR, "V3_Final"),
]

# Keywords we care about
keywords = [
    "cosine",
    "similarity",
    "skill_score",
    "role_similarity",
    "matrix",
    "feature"
]

candidates = []

for root in search_roots:
    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):
        for filename in filenames:
            lower = filename.lower()

            if any(k in lower for k in keywords):
                path = os.path.join(dirpath, filename)

                # Avoid huge text/label files unless clearly matrix-like
                size_mb = os.path.getsize(path) / (1024 * 1024)

                candidates.append((path, size_mb))

# Deduplicate
seen = set()
unique_candidates = []

for path, size_mb in candidates:
    if path not in seen:
        seen.add(path)
        unique_candidates.append((path, size_mb))

unique_candidates.sort(key=lambda x: x[1])

print(f"\nFound {len(unique_candidates)} potentially relevant files.\n")

for i, (path, size_mb) in enumerate(unique_candidates, 1):
    print(f"[{i:03d}] {size_mb:8.2f} MB | {path}")

# ------------------------------------------------------------------
# Try to identify matrix-like pickle/numpy artifacts
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("MATRIX-LIKE ARTIFACTS")
print("=" * 100)

for path, size_mb in unique_candidates:

    lower = path.lower()

    if not (
        lower.endswith(".pkl")
        or lower.endswith(".pickle")
        or lower.endswith(".npy")
    ):
        continue

    # Focus on files with likely matrix names
    if not any(k in lower for k in [
        "cosine",
        "skill_score",
        "role_similarity",
        "similarity_matrix",
        "matrix"
    ]):
        continue

    try:

        if lower.endswith(".npy"):
            import numpy as np
            obj = np.load(path, allow_pickle=True)

        else:
            with open(path, "rb") as f:
                obj = pickle.load(f)

        shape = getattr(obj, "shape", None)

        if shape is not None:
            print(f"\n✅ {path}")
            print("   type :", type(obj))
            print("   shape:", shape)

        elif isinstance(obj, pd.DataFrame):
            print(f"\n✅ {path}")
            print("   type   :", type(obj))
            print("   shape  :", obj.shape)
            print("   columns:", obj.columns.tolist()[:20])

    except Exception as e:
        # Many project files may not be directly loadable; that's okay.
        pass

print("\n" + "=" * 100)
print("CELL 14 COMPLETE")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 15 — BUILDING V3 TRAINING DATASET")
print("=" * 100)

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
PROJECT_DIR = "/content/drive/MyDrive/Resume_Model_Project"

V2_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "V2_100k",
    "Outputs"
)

V3_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "V3",
    "Outputs"
)

QUEUE_PATH = os.path.join(
    PROJECT_DIR,
    "V3_Final",
    "Outputs",
    "v3_unlabeled_pairs_568210.csv"
)

LABEL_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_labels_568210_complete.csv"
)

RICH6_PATH = os.path.join(
    V2_OUTPUT_DIR,
    "v2_supervised_dataset_100k_rich.csv"
)

COSINE_PATH = os.path.join(
    V2_OUTPUT_DIR,
    "v1_similarity_matrix.npy"
)

SKILL_PATH = os.path.join(
    V2_OUTPUT_DIR,
    "v1_skill_score_matrix.npy"
)

FINAL_PATH = os.path.join(
    V2_OUTPUT_DIR,
    "v1_final_matrix.npy"
)

ROLE_PATH = os.path.join(
    V2_OUTPUT_DIR,
    "role_similarity_matrix.npy"
)

SEMANTIC_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_similarity_matrix.npy"
)

# ------------------------------------------------------------------
# Load matrices
# ------------------------------------------------------------------
print("\nLoading matrices...")

cosine_matrix = np.load(COSINE_PATH)
skill_matrix = np.load(SKILL_PATH)
final_matrix = np.load(FINAL_PATH)
role_matrix = np.load(ROLE_PATH)
semantic_matrix = np.load(SEMANTIC_PATH)

print("Cosine matrix :", cosine_matrix.shape)
print("Skill matrix  :", skill_matrix.shape)
print("Final matrix  :", final_matrix.shape)
print("Role matrix   :", role_matrix.shape)
print("Semantic matrix:", semantic_matrix.shape)

# ------------------------------------------------------------------
# Verify exact expected shapes
# ------------------------------------------------------------------
assert cosine_matrix.shape == (434, 1540)
assert skill_matrix.shape == (434, 1540)
assert final_matrix.shape == (434, 1540)
assert role_matrix.shape == (434, 1540)
assert semantic_matrix.shape == (1540, 434)

print("\n✅ All matrix shapes verified.")

# ------------------------------------------------------------------
# Load V3 queue + labels
# ------------------------------------------------------------------
queue_df = pd.read_csv(QUEUE_PATH)
labels_df = pd.read_csv(LABEL_PATH)

print("\nQueue rows :", len(queue_df))
print("Label rows:", len(labels_df))

assert len(queue_df) == 568210
assert len(labels_df) == 568210

# ------------------------------------------------------------------
# Convert indices
# ------------------------------------------------------------------
resume_idx = queue_df["Resume_Index"].astype(int).to_numpy()
job_idx = queue_df["Job_Index"].astype(int).to_numpy()

assert resume_idx.min() >= 0
assert resume_idx.max() < 1540

assert job_idx.min() >= 0
assert job_idx.max() < 434

print("✅ Resume / Job indices are in valid ranges.")

# ------------------------------------------------------------------
# Align labels to queue by Pair_ID
# ------------------------------------------------------------------
label_lookup = labels_df.set_index("Pair_ID")["Label"]

queue_pair_ids = queue_df["Pair_ID"].astype(int)

labels = queue_pair_ids.map(label_lookup).to_numpy()

assert not pd.isna(labels).any()
labels = labels.astype(int)

assert np.isin(labels, [0, 1, 2]).all()

print("✅ Labels aligned to V3 queue.")

# ------------------------------------------------------------------
# Extract matrices for each queued pair
#
# Old matrices:
#     [Job_Index, Resume_Index]
#
# Semantic matrix:
#     [Resume_Index, Job_Index]
# ------------------------------------------------------------------
print("\nExtracting pair-level features...")

cosine = cosine_matrix[job_idx, resume_idx]
skill_score = skill_matrix[job_idx, resume_idx]
v1_final_score = final_matrix[job_idx, resume_idx]
role_similarity = role_matrix[job_idx, resume_idx]

semantic_similarity = semantic_matrix[resume_idx, job_idx]

# ------------------------------------------------------------------
# Recompute Normalized_Cosine from raw cosine
#
# Per-job Min-Max normalization across the 1,540 candidate pool.
# ------------------------------------------------------------------
job_min = cosine_matrix.min(axis=1)
job_max = cosine_matrix.max(axis=1)

denominator = job_max - job_min

normalized_cosine = np.zeros_like(cosine, dtype=np.float64)

non_constant = denominator[job_idx] != 0

normalized_cosine[non_constant] = (
    cosine[non_constant] - job_min[job_idx[non_constant]]
) / denominator[job_idx[non_constant]]

# Numerical safety
normalized_cosine = np.clip(
    normalized_cosine,
    0.0,
    1.0
)

# ------------------------------------------------------------------
# Get exact required-skill counts from the V2 Rich-6 dataset
# ------------------------------------------------------------------
print("\nLoading V2 Rich-6 reference data...")

rich6_ref = pd.read_csv(
    RICH6_PATH,
    usecols=[
        "Job_Index",
        "Required_Skill_Count",
        "Matched_Skill_Count",
        "Skill_Score",
        "Normalized_Cosine",
        "Role_Similarity"
    ]
)

print("V2 Rich-6 reference rows:", len(rich6_ref))

# Required skill count should be constant for a given job
required_counts = (
    rich6_ref[
        ["Job_Index", "Required_Skill_Count"]
    ]
    .drop_duplicates()
)

required_count_per_job = (
    rich6_ref
    .groupby("Job_Index")["Required_Skill_Count"]
    .nunique()
)

bad_jobs = required_count_per_job[
    required_count_per_job > 1
]

if len(bad_jobs) > 0:
    raise ValueError(
        f"❌ Required_Skill_Count is inconsistent for jobs: "
        f"{bad_jobs.index.tolist()}"
    )

required_map = (
    required_counts
    .drop_duplicates("Job_Index")
    .set_index("Job_Index")["Required_Skill_Count"]
)

# Check that all 434 jobs are represented
missing_jobs = set(range(434)) - set(required_map.index)

if missing_jobs:
    raise ValueError(
        f"❌ Missing Required_Skill_Count for jobs: {sorted(missing_jobs)}"
    )

required_skill_count = (
    pd.Series(job_idx)
    .map(required_map)
    .to_numpy()
)

if pd.isna(required_skill_count).any():
    raise ValueError(
        "❌ Could not map Required_Skill_Count to every V3 pair."
    )

required_skill_count = required_skill_count.astype(int)

# ------------------------------------------------------------------
# Matched skill count
#
# skill_score = matched / required
# Therefore:
# matched = skill_score * required
#
# Round because the underlying count is integer.
# ------------------------------------------------------------------
matched_skill_count = np.rint(
    skill_score * required_skill_count
).astype(int)

# Sanity bounds
if (matched_skill_count < 0).any():
    raise ValueError("❌ Negative matched-skill count.")

if (
    matched_skill_count > required_skill_count
).any():
    raise ValueError(
        "❌ Matched skill count exceeds required skill count."
    )

# ------------------------------------------------------------------
# Build final V3 table
# ------------------------------------------------------------------
v3_df = pd.DataFrame({
    "Pair_ID": queue_pair_ids.to_numpy(),

    "Job_Index": job_idx,
    "Resume_Index": resume_idx,

    "Cosine": cosine,
    "Skill_Score": skill_score,
    "Normalized_Cosine": normalized_cosine,

    "Required_Skill_Count": required_skill_count,
    "Matched_Skill_Count": matched_skill_count,

    "Role_Similarity": role_similarity,

    "Semantic_Similarity": semantic_similarity,

    "Label": labels
})

# ------------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------------
assert len(v3_df) == 568210
assert v3_df["Pair_ID"].nunique() == 568210
assert v3_df["Pair_ID"].duplicated().sum() == 0

feature_columns = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

assert not v3_df[feature_columns].isna().any().any()

# ------------------------------------------------------------------
# IMPORTANT VALIDATION AGAINST ORIGINAL V2 DATA
#
# Verify our extraction reproduces the stored V2 features.
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("VALIDATING AGAINST STORED V2 RICH-6 FEATURES")
print("=" * 100)

sample = rich6_ref.copy()

sample_resume = None
sample_job = None

# The V2 reference file contains Resume_Index too,
# so reload it specifically for this validation.
rich6_validation = pd.read_csv(
    RICH6_PATH,
    usecols=[
        "Pair_ID",
        "Job_Index",
        "Resume_Index",
        "Cosine",
        "Skill_Score",
        "Normalized_Cosine",
        "Required_Skill_Count",
        "Matched_Skill_Count",
        "Role_Similarity"
    ]
)

vr = rich6_validation["Resume_Index"].astype(int).to_numpy()
vj = rich6_validation["Job_Index"].astype(int).to_numpy()

# Sample up to 10,000 rows to keep validation lightweight
rng = np.random.default_rng(42)

sample_n = min(10000, len(rich6_validation))

sample_positions = rng.choice(
    len(rich6_validation),
    size=sample_n,
    replace=False
)

vr_s = vr[sample_positions]
vj_s = vj[sample_positions]

# Recompute / extract
cosine_check = cosine_matrix[vj_s, vr_s]
skill_check = skill_matrix[vj_s, vr_s]
role_check = role_matrix[vj_s, vr_s]

job_min_check = cosine_matrix.min(axis=1)
job_max_check = cosine_matrix.max(axis=1)

den_check = (
    job_max_check[vj_s] -
    job_min_check[vj_s]
)

norm_check = np.zeros_like(
    cosine_check,
    dtype=np.float64
)

mask = den_check != 0

norm_check[mask] = (
    cosine_check[mask] -
    job_min_check[vj_s[mask]]
) / den_check[mask]

stored = rich6_validation.iloc[sample_positions]

# Maximum absolute differences
cosine_diff = np.max(
    np.abs(
        cosine_check -
        stored["Cosine"].to_numpy()
    )
)

skill_diff = np.max(
    np.abs(
        skill_check -
        stored["Skill_Score"].to_numpy()
    )
)

norm_diff = np.max(
    np.abs(
        norm_check -
        stored["Normalized_Cosine"].to_numpy()
    )
)

role_diff = np.max(
    np.abs(
        role_check -
        stored["Role_Similarity"].to_numpy()
    )
)

required_diff = np.max(
    np.abs(
        required_map.loc[vj_s].to_numpy() -
        stored["Required_Skill_Count"].to_numpy()
    )
)

matched_check = np.rint(
    skill_check *
    stored["Required_Skill_Count"].to_numpy()
).astype(int)

matched_diff = np.max(
    np.abs(
        matched_check -
        stored["Matched_Skill_Count"].to_numpy()
    )
)

print(f"Sample size: {sample_n:,}")

print(f"Cosine max abs diff:             {cosine_diff:.12f}")
print(f"Skill_Score max abs diff:        {skill_diff:.12f}")
print(f"Normalized_Cosine max abs diff:  {norm_diff:.12f}")
print(f"Required_Skill_Count max diff:   {required_diff}")
print(f"Matched_Skill_Count max diff:    {matched_diff}")
print(f"Role_Similarity max abs diff:    {role_diff:.12f}")

# Tight tolerance
if cosine_diff > 1e-6:
    raise ValueError("❌ Cosine mismatch detected.")

if skill_diff > 1e-6:
    raise ValueError("❌ Skill_Score mismatch detected.")

if norm_diff > 1e-6:
    raise ValueError("❌ Normalized_Cosine mismatch detected.")

if required_diff != 0:
    raise ValueError("❌ Required skill count mismatch.")

if matched_diff != 0:
    raise ValueError("❌ Matched skill count mismatch.")

if role_diff > 1e-5:
    raise ValueError(
        "❌ Role_Similarity mismatch detected."
    )

print("\n✅ V2 Rich-6 reproduction check PASSED.")

# ------------------------------------------------------------------
# Semantic sanity check
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("SEMANTIC FEATURE SANITY CHECK")
print("=" * 100)

print(
    "Semantic min:",
    float(np.min(semantic_similarity))
)

print(
    "Semantic max:",
    float(np.max(semantic_similarity))
)

print(
    "Semantic mean:",
    float(np.mean(semantic_similarity))
)

if not np.isfinite(semantic_similarity).all():
    raise ValueError(
        "❌ Non-finite semantic similarity values detected."
    )

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
V3_DATASET_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

V3_DATASET_PKL = os.path.join(
    V3_OUTPUT_DIR,
    "v3_supervised_dataset_568210_rich6_semantic.pkl"
)

v3_df.to_csv(
    V3_DATASET_CSV,
    index=False
)

v3_df.to_pickle(
    V3_DATASET_PKL
)

# ------------------------------------------------------------------
# Final summary
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("✅ V3 TRAINING DATASET CREATED")
print("=" * 100)

print(f"Rows: {len(v3_df):,}")
print(f"Features: {len(feature_columns)}")

print("\nFeatures:")
for feature in feature_columns:
    print("  •", feature)

print("\nLabel distribution:")
print(
    v3_df["Label"]
    .value_counts()
    .sort_index()
)

print("\nSaved:")
print(V3_DATASET_CSV)
print(V3_DATASET_PKL)

print("\n" + "=" * 100)
print("🚀 V3 DATASET READY FOR MODEL TRAINING")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 16 — LOCATING CLEAN RANKING BENCHMARK")
print("=" * 100)

import os
import pandas as pd
import pickle

PROJECT_DIR = "/content/drive/MyDrive/Resume_Model_Project"

search_roots = [
    os.path.join(PROJECT_DIR, "V2_100k"),
    os.path.join(PROJECT_DIR, "V2_Development"),
    os.path.join(PROJECT_DIR, "V3"),
    os.path.join(PROJECT_DIR, "V3_Final"),
]

candidates = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):

        for filename in filenames:

            lower = filename.lower()

            # Benchmark/ranking-related names
            if not any(k in lower for k in [
                "ranking",
                "benchmark",
                "evaluation"
            ]):
                continue

            path = os.path.join(dirpath, filename)

            if not (
                lower.endswith(".csv")
                or lower.endswith(".pkl")
                or lower.endswith(".pickle")
            ):
                continue

            size_mb = os.path.getsize(path) / (1024 * 1024)

            candidates.append((path, size_mb))

# Deduplicate
seen = set()
unique_candidates = []

for path, size_mb in candidates:
    if path not in seen:
        seen.add(path)
        unique_candidates.append((path, size_mb))

unique_candidates.sort()

print(f"\nFound {len(unique_candidates)} ranking/evaluation candidates.\n")

for i, (path, size_mb) in enumerate(unique_candidates, 1):
    print(f"[{i:03d}] {size_mb:8.2f} MB | {path}")

# ------------------------------------------------------------------
# Inspect likely tabular candidates
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("INSPECTING CANDIDATES")
print("=" * 100)

found_2540 = []

for path, size_mb in unique_candidates:

    try:

        lower = path.lower()

        if lower.endswith(".csv"):
            df = pd.read_csv(path)

        else:
            with open(path, "rb") as f:
                obj = pickle.load(f)

            if isinstance(obj, pd.DataFrame):
                df = obj
            else:
                continue

        print("\nFile:")
        print(path)

        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())

        # Look for the expected benchmark structure
        cols = set(df.columns)

        possible_pair_cols = {
            "Pair_ID",
            "Resume_Index",
            "Job_Index"
        }

        if (
            len(df) == 2540
            and possible_pair_cols.issubset(cols)
        ):
            found_2540.append(path)
            print("🎯 THIS LOOKS LIKE THE 2,540-PAIR BENCHMARK")

    except Exception as e:
        print(f"⚠️ Could not inspect: {e}")

print("\n" + "=" * 100)
print("BENCHMARK CANDIDATES")
print("=" * 100)

if found_2540:
    for path in found_2540:
        print("✅", path)
else:
    print("⚠️ No exact 2,540-row benchmark artifact found yet.")

print("\n" + "=" * 100)
print("CELL 16 COMPLETE")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 17 — CHECKING FOR ORIGINAL RANKING BENCHMARK IN MEMORY")
print("=" * 100)

import pandas as pd

# Possible names used during the V2/V3 work
possible_names = [
    "ranking_df_73",
    "ranking_df",
    "clean_ranking_df",
    "benchmark_df",
    "ranking_benchmark_df",
    "eval_ranking_df",
]

found = []

for name in possible_names:
    if name in globals():

        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):

            print(f"\n✅ Found DataFrame: {name}")
            print("   Rows:", len(obj))
            print("   Columns:", obj.columns.tolist())

            if len(obj) == 2540:
                found.append(name)

# Also scan ALL currently loaded DataFrames
print("\n" + "=" * 100)
print("SCANNING ALL DATAFRAMES IN MEMORY")
print("=" * 100)

memory_matches = []

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        if len(obj) == 2540:

            print(f"\n🎯 {name}")
            print("   Shape:", obj.shape)
            print("   Columns:", obj.columns.tolist())

            memory_matches.append(name)

# ------------------------------------------------------------------
# Determine result
# ------------------------------------------------------------------
print("\n" + "=" * 100)

if memory_matches:

    print("✅ 2,540-ROW DATAFRAME(S) FOUND IN MEMORY")

    for name in memory_matches:
        obj = globals()[name]

        required = {
            "Resume_Index",
            "Job_Index"
        }

        if required.issubset(obj.columns):

            print(f"\n🎯 BEST CANDIDATE: {name}")

            print("\nPreview:")
            print(obj.head())

            print("\nUnique resumes:",
                  obj["Resume_Index"].nunique())

            print("Unique jobs:",
                  obj["Job_Index"].nunique())

            # Check whether it has exactly 10 jobs per resume
            counts = obj.groupby("Resume_Index").size()

            print("\nRows per resume:")
            print(counts.value_counts().sort_index())

            if len(counts) == 254 and (counts == 10).all():

                print("\n🔥 THIS MATCHES THE EXPECTED")
                print("   254 resumes × 10 jobs = 2,540 pairs")

else:

    print("❌ No 2,540-row DataFrame found in current memory.")

print("\n" + "=" * 100)
print("CELL 17 COMPLETE")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 18 — CREATING NEW COMMON RANKING BENCHMARK")
print("=" * 100)

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Reuse the V3 dataset created in Cell 15
# ------------------------------------------------------------------
V3_DATASET_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

v3_all = pd.read_csv(V3_DATASET_PATH)

print("V3 dataset rows:", len(v3_all))

assert len(v3_all) == 568210

# ------------------------------------------------------------------
# We want:
#   254 resumes
#   10 labeled jobs per resume
#   = 2,540 test pairs
#
# To make the benchmark reproducible:
#   random seed = 42
# ------------------------------------------------------------------
RNG_SEED = 42
N_RESUMES = 254
N_JOBS_PER_RESUME = 10

rng = np.random.default_rng(RNG_SEED)

# ------------------------------------------------------------------
# Find resumes having at least 10 V3-labeled pairs
# ------------------------------------------------------------------
resume_counts = (
    v3_all
    .groupby("Resume_Index")
    .size()
)

eligible_resumes = resume_counts[
    resume_counts >= N_JOBS_PER_RESUME
].index.to_numpy()

print("Eligible resumes:", len(eligible_resumes))

if len(eligible_resumes) < N_RESUMES:
    raise ValueError(
        f"❌ Only {len(eligible_resumes)} resumes have at least "
        f"{N_JOBS_PER_RESUME} V3-labeled jobs."
    )

# ------------------------------------------------------------------
# Select 254 resumes
# ------------------------------------------------------------------
selected_resumes = np.sort(
    rng.choice(
        eligible_resumes,
        size=N_RESUMES,
        replace=False
    )
)

print("Selected resumes:", len(selected_resumes))

# ------------------------------------------------------------------
# For each selected resume, sample exactly 10 jobs
# ------------------------------------------------------------------
benchmark_parts = []

for resume_idx in selected_resumes:

    resume_rows = v3_all[
        v3_all["Resume_Index"] == resume_idx
    ]

    # Deterministic per-resume sampling using the same RNG
    chosen_positions = rng.choice(
        len(resume_rows),
        size=N_JOBS_PER_RESUME,
        replace=False
    )

    chosen = resume_rows.iloc[
        chosen_positions
    ].copy()

    benchmark_parts.append(chosen)

benchmark_df = pd.concat(
    benchmark_parts,
    ignore_index=True
)

# ------------------------------------------------------------------
# Sort for deterministic ordering
# ------------------------------------------------------------------
benchmark_df = benchmark_df.sort_values(
    ["Resume_Index", "Job_Index"]
).reset_index(drop=True)

# ------------------------------------------------------------------
# Validation
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("BENCHMARK VALIDATION")
print("=" * 100)

print("Rows:", len(benchmark_df))
print(
    "Unique resumes:",
    benchmark_df["Resume_Index"].nunique()
)

rows_per_resume = (
    benchmark_df
    .groupby("Resume_Index")
    .size()
)

print("\nRows per resume:")
print(rows_per_resume.value_counts().sort_index())

# Exact structural checks
assert len(benchmark_df) == 2540
assert benchmark_df["Resume_Index"].nunique() == 254
assert (rows_per_resume == 10).all()

# Pair uniqueness
assert benchmark_df[
    ["Resume_Index", "Job_Index"]
].duplicated().sum() == 0

assert benchmark_df["Pair_ID"].duplicated().sum() == 0

print("\n✅ 254 resumes × 10 jobs = 2,540 unique pairs")

# ------------------------------------------------------------------
# Ensure benchmark is NOT duplicated in the V3 training pool
# ------------------------------------------------------------------
benchmark_pair_ids = set(
    benchmark_df["Pair_ID"].astype(int)
)

all_pair_ids = set(
    v3_all["Pair_ID"].astype(int)
)

assert benchmark_pair_ids.issubset(all_pair_ids)

train_pool_df = v3_all[
    ~v3_all["Pair_ID"].isin(benchmark_pair_ids)
].copy()

print("\nV3 training pool after benchmark holdout:")
print("Rows:", len(train_pool_df))

assert len(train_pool_df) == 568210 - 2540
assert train_pool_df["Pair_ID"].duplicated().sum() == 0

# ------------------------------------------------------------------
# Label distribution
# ------------------------------------------------------------------
print("\nBenchmark label distribution:")
print(
    benchmark_df["Label"]
    .value_counts()
    .sort_index()
)

print("\nTraining-pool label distribution:")
print(
    train_pool_df["Label"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------------
# Save benchmark + clean V3 training pool
# ------------------------------------------------------------------
BENCHMARK_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_common_ranking_benchmark_2540.csv"
)

BENCHMARK_PKL = os.path.join(
    V3_OUTPUT_DIR,
    "v3_common_ranking_benchmark_2540.pkl"
)

TRAIN_POOL_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_training_pool_565670.csv"
)

TRAIN_POOL_PKL = os.path.join(
    V3_OUTPUT_DIR,
    "v3_training_pool_565670.pkl"
)

benchmark_df.to_csv(
    BENCHMARK_CSV,
    index=False
)

benchmark_df.to_pickle(
    BENCHMARK_PKL
)

train_pool_df.to_csv(
    TRAIN_POOL_CSV,
    index=False
)

train_pool_df.to_pickle(
    TRAIN_POOL_PKL
)

# ------------------------------------------------------------------
# Save benchmark metadata
# ------------------------------------------------------------------
metadata = {
    "random_seed": RNG_SEED,
    "benchmark_resumes": N_RESUMES,
    "jobs_per_resume": N_JOBS_PER_RESUME,
    "benchmark_pairs": len(benchmark_df),
    "original_v3_pairs": len(v3_all),
    "training_pool_pairs": len(train_pool_df),
    "selection_note": (
        "New common benchmark created because the exact historical "
        "2,540-pair benchmark could not be recovered."
    )
}

metadata_path = os.path.join(
    V3_OUTPUT_DIR,
    "v3_common_benchmark_metadata.json"
)

import json

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------------
# Final output
# ------------------------------------------------------------------
print("\n" + "=" * 100)
print("✅ COMMON BENCHMARK CREATED")
print("=" * 100)

print(f"Benchmark:      {len(benchmark_df):,} pairs")
print(f"Resumes:        {benchmark_df['Resume_Index'].nunique():,}")
print(f"Jobs/resume:    {N_JOBS_PER_RESUME}")
print(f"Training pool:  {len(train_pool_df):,} pairs")
print(f"Seed:            {RNG_SEED}")

print("\nSaved:")
print(BENCHMARK_CSV)
print(BENCHMARK_PKL)
print(TRAIN_POOL_CSV)
print(TRAIN_POOL_PKL)
print(metadata_path)

print("\n" + "=" * 100)
print("🚀 BENCHMARK FROZEN — READY FOR V3 TRAINING")
print("=" * 100)

In [ ]:
!pip install -q catboost

In [ ]:
print("=" * 100)
print("V3 CELL 19 — TRAINING V3 SEMANTIC + RICH-6 CATBOOST MODEL")
print("=" * 100)

import os
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ============================================================================
# PATHS
# ============================================================================
TRAIN_POOL_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_training_pool_565670.csv"
)

MODEL_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_rich7_catboost.cbm"
)

TRAIN_SPLIT_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_train_validation_split.csv"
)

METADATA_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_model_metadata.json"
)

# ============================================================================
# LOAD TRAINING POOL
# ============================================================================
v3_train_pool = pd.read_csv(TRAIN_POOL_CSV)

print("Loaded training pool:", v3_train_pool.shape)

assert len(v3_train_pool) == 565670

# ============================================================================
# FEATURES
# ============================================================================
FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

TARGET = "Label"

missing_features = [
    c for c in FEATURES
    if c not in v3_train_pool.columns
]

if missing_features:
    raise ValueError(
        f"❌ Missing required features: {missing_features}"
    )

assert TARGET in v3_train_pool.columns

print("\nFeatures:")
for i, feature in enumerate(FEATURES, 1):
    print(f"{i}. {feature}")

# ============================================================================
# BASIC SANITY CHECKS
# ============================================================================
print("\n" + "=" * 100)
print("FEATURE SANITY CHECK")
print("=" * 100)

print(
    v3_train_pool[FEATURES + [TARGET]]
    .isna()
    .sum()
)

if v3_train_pool[FEATURES + [TARGET]].isna().sum().sum() != 0:
    raise ValueError("❌ NaN values detected.")

print("\nLabel distribution:")
print(
    v3_train_pool[TARGET]
    .value_counts()
    .sort_index()
)

# ============================================================================
# RESUME-LEVEL TRAIN / VALIDATION SPLIT
# ============================================================================
RANDOM_SEED = 42

all_train_resumes = np.sort(
    v3_train_pool["Resume_Index"]
    .unique()
)

print("\nTraining-pool resumes:", len(all_train_resumes))

rng = np.random.default_rng(RANDOM_SEED)

shuffled_resumes = rng.permutation(
    all_train_resumes
)

n_train_resumes = int(
    len(shuffled_resumes) * 0.80
)

train_resumes = np.sort(
    shuffled_resumes[:n_train_resumes]
)

val_resumes = np.sort(
    shuffled_resumes[n_train_resumes:]
)

# ============================================================================
# BUILD SPLITS
# ============================================================================
train_mask = v3_train_pool["Resume_Index"].isin(
    train_resumes
)

val_mask = v3_train_pool["Resume_Index"].isin(
    val_resumes
)

train_df = v3_train_pool.loc[
    train_mask
].copy()

val_df = v3_train_pool.loc[
    val_mask
].copy()

print("\n" + "=" * 100)
print("TRAIN / VALIDATION SPLIT")
print("=" * 100)

print("Training resumes:", len(train_resumes))
print("Validation resumes:", len(val_resumes))

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(val_df))

# ============================================================================
# LEAKAGE CHECK
# ============================================================================
resume_overlap = set(train_resumes).intersection(
    set(val_resumes)
)

print("\nResume overlap:", len(resume_overlap))

assert len(resume_overlap) == 0

print("✅ No resume appears in both train and validation.")

# ============================================================================
# SAVE SPLIT ASSIGNMENTS
# ============================================================================
split_df = pd.DataFrame({
    "Resume_Index": np.concatenate([
        train_resumes,
        val_resumes
    ]),
    "Split": (
        ["train"] * len(train_resumes)
        +
        ["validation"] * len(val_resumes)
    )
})

split_df.to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)

# ============================================================================
# PREPARE MATRICES
# ============================================================================
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

print("\nX_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

# ============================================================================
# TRAIN CATBOOST
#
# SAME CORE HYPERPARAMETERS AS V2
# ============================================================================
print("\n" + "=" * 100)
print("STARTING CATBOOST TRAINING")
print("=" * 100)

model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=50,
    thread_count=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=False
)

print("\n✅ V3 TRAINING COMPLETE")

# ============================================================================
# VALIDATION PREDICTIONS
# ============================================================================
val_predictions = model.predict(X_val)

# Keep prediction range visible for inspection
print("\nValidation prediction range:")
print("Min:", float(val_predictions.min()))
print("Max:", float(val_predictions.max()))
print("Mean:", float(val_predictions.mean()))

# ============================================================================
# VALIDATION METRICS
# ============================================================================
rmse = np.sqrt(
    mean_squared_error(
        y_val,
        val_predictions
    )
)

mae = mean_absolute_error(
    y_val,
    val_predictions
)

r2 = r2_score(
    y_val,
    val_predictions
)

print("\n" + "=" * 100)
print("V3 VALIDATION METRICS")
print("=" * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE : {mae:.6f}")
print(f"R²  : {r2:.6f}")

# ============================================================================
# FEATURE IMPORTANCE
# ============================================================================
importance = model.get_feature_importance()

feature_importance_df = (
    pd.DataFrame({
        "Feature": FEATURES,
        "Importance": importance
    })
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("V3 FEATURE IMPORTANCE")
print("=" * 100)

print(
    feature_importance_df.to_string(index=False)
)

# ============================================================================
# SAVE MODEL
# ============================================================================
model.save_model(
    MODEL_PATH
)

print("\n✅ Model saved:")
print(MODEL_PATH)

# ============================================================================
# SAVE METADATA
# ============================================================================
metadata = {
    "model_name": "V3 Semantic + Rich6 CatBoost",
    "features": FEATURES,
    "target": TARGET,
    "training_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "training_resumes": int(len(train_resumes)),
    "validation_resumes": int(len(val_resumes)),
    "random_seed": RANDOM_SEED,
    "catboost_parameters": {
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "loss_function": "RMSE",
        "random_seed": 42
    },
    "validation_metrics": {
        "RMSE": float(rmse),
        "MAE": float(mae),
        "R2": float(r2)
    },
    "feature_importance": {
        row["Feature"]: float(row["Importance"])
        for _, row in feature_importance_df.iterrows()
    }
}

with open(
    METADATA_PATH,
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print("\n✅ Metadata saved:")
print(METADATA_PATH)

print("\n" + "=" * 100)
print("🚀 CELL 19 COMPLETE")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 20 — PREPARING V1 / V2 / V3 COMMON EVALUATION")
print("=" * 100)

import os
import glob
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

# ============================================================================
# PATHS
# ============================================================================

BENCHMARK_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_common_ranking_benchmark_2540.csv"
)

V1_FINAL_MATRIX_PATH = (
    "/content/drive/MyDrive/Resume_Model_Project/"
    "V2_100k/Outputs/v1_final_matrix.npy"
)

V2_ROOT = (
    "/content/drive/MyDrive/Resume_Model_Project/"
    "V2_100k"
)

V3_MODEL_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_semantic_rich7_catboost.cbm"
)

# ============================================================================
# LOAD BENCHMARK
# ============================================================================

benchmark = pd.read_csv(BENCHMARK_PATH)

print("\nBenchmark shape:", benchmark.shape)
print("Columns:")
print(benchmark.columns.tolist())

assert len(benchmark) == 2540

# ============================================================================
# FIND V2 CATBOOST MODEL
# ============================================================================

print("\n" + "=" * 100)
print("SEARCHING FOR V2 CATBOOST MODEL")
print("=" * 100)

possible_v2_models = []

for ext in ["*.cbm", "*.model", "*.bin", "*.pkl", "*.pickle"]:
    possible_v2_models.extend(
        glob.glob(
            os.path.join(V2_ROOT, "**", ext),
            recursive=True
        )
    )

# Show likely model-related files
model_candidates = [
    p for p in possible_v2_models
    if any(
        token in os.path.basename(p).lower()
        for token in [
            "v2",
            "catboost",
            "model",
            "final"
        ]
    )
]

print("Potential V2 model files:")
for p in model_candidates:
    print("  ", p)

if not model_candidates:
    raise FileNotFoundError(
        "❌ Could not find a possible V2 CatBoost model under V2_100k."
    )

# ============================================================================
# LOAD V1 FINAL MATRIX
# ============================================================================

print("\n" + "=" * 100)
print("LOADING V1 FINAL SCORE MATRIX")
print("=" * 100)

v1_final_matrix = np.load(
    V1_FINAL_MATRIX_PATH
)

print("V1 matrix shape:", v1_final_matrix.shape)

assert v1_final_matrix.shape == (434, 1540)

print("✅ V1 matrix loaded.")

# ============================================================================
# LOAD V3 MODEL
# ============================================================================

print("\n" + "=" * 100)
print("LOADING V3 MODEL")
print("=" * 100)

v3_model = CatBoostRegressor()

v3_model.load_model(
    V3_MODEL_PATH
)

print("✅ V3 model loaded.")

# ============================================================================
# BASIC BENCHMARK VALIDATION
# ============================================================================

print("\n" + "=" * 100)
print("BENCHMARK STRUCTURE")
print("=" * 100)

print(
    "Unique resumes:",
    benchmark["Resume_Index"].nunique()
)

print(
    "Unique jobs:",
    benchmark["Job_Index"].nunique()
)

rows_per_resume = (
    benchmark
    .groupby("Resume_Index")
    .size()
)

print("\nRows per resume:")
print(
    rows_per_resume
    .value_counts()
    .sort_index()
)

assert benchmark["Resume_Index"].nunique() == 254
assert (rows_per_resume == 10).all()

assert (
    benchmark[
        ["Resume_Index", "Job_Index"]
    ]
    .duplicated()
    .sum()
    == 0
)

print("\n✅ Benchmark is valid.")

print("\n" + "=" * 100)
print("CELL 20 COMPLETE")
print("=" * 100)

print("""
Next:
    V1 → stored final scores
    V2 → frozen Rich-6 CatBoost
    V3 → new Semantic + Rich-6 CatBoost

Then evaluate all three on the EXACT SAME 2,540 pairs.
""")

In [ ]:
print("=" * 100)
print("V3 CELL 21 — LOADING + VERIFYING FROZEN V2 RICH-6 MODEL")
print("=" * 100)

import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

# ============================================================================
# EXACT FROZEN V2 MODEL
# ============================================================================

V2_MODEL_PATH = (
    "/content/drive/MyDrive/Resume_Model_Project/"
    "V2_100k/Outputs/v2_final_catboost_rich6_all100k.cbm"
)

# ============================================================================
# EXPECTED V2 FEATURES
# ============================================================================

V2_FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity"
]

# ============================================================================
# LOAD MODEL
# ============================================================================

if not os.path.exists(V2_MODEL_PATH):
    raise FileNotFoundError(
        f"❌ V2 model not found:\n{V2_MODEL_PATH}"
    )

v2_model = CatBoostRegressor()

v2_model.load_model(
    V2_MODEL_PATH
)

print("\n✅ V2 model loaded")
print(V2_MODEL_PATH)

# ============================================================================
# MODEL INFORMATION
# ============================================================================

print("\n" + "=" * 100)
print("V2 MODEL INFORMATION")
print("=" * 100)

print("Tree count:", v2_model.tree_count_)

try:
    print("Model feature count:", v2_model.feature_count_)
except Exception:
    print("Model feature count: unavailable")

try:
    print("Model feature names:")
    print(v2_model.feature_names_)
except Exception as e:
    print("Could not read feature names:", e)

# ============================================================================
# VERIFY 6 FEATURES
# ============================================================================

print("\n" + "=" * 100)
print("VERIFYING RICH-6 FEATURE SET")
print("=" * 100)

try:
    loaded_feature_names = list(v2_model.feature_names_)

    if loaded_feature_names:
        print("Loaded feature names:")
        for i, f in enumerate(loaded_feature_names, 1):
            print(f"{i}. {f}")

        if loaded_feature_names == V2_FEATURES:
            print("\n🔥 EXACT RICH-6 FEATURE ORDER CONFIRMED")
        else:
            print("\n⚠️ Feature names/order differ from expected Rich-6.")
            print("Expected:")
            print(V2_FEATURES)

except Exception as e:
    print("⚠️ Could not fully verify feature names:", e)

# ============================================================================
# VERIFY TEST DATA COLUMNS
# ============================================================================

required_v2_columns = (
    V2_FEATURES +
    ["Resume_Index", "Job_Index", "Pair_ID", "Label"]
)

missing_columns = [
    c for c in required_v2_columns
    if c not in benchmark.columns
]

print("\nBenchmark missing required columns:", missing_columns)

if missing_columns:
    raise ValueError(
        f"❌ Benchmark missing columns: {missing_columns}"
    )

print("✅ Benchmark contains all V2 features.")

# ============================================================================
# SAMPLE PREDICTION TEST
# ============================================================================

X_benchmark_v2 = benchmark[
    V2_FEATURES
].copy()

print("\nV2 benchmark matrix:", X_benchmark_v2.shape)

v2_sample_predictions = v2_model.predict(
    X_benchmark_v2.head(10)
)

print("\nSample V2 predictions:")
print(v2_sample_predictions)

print("\nPrediction range on first 10:")
print("Min:", float(v2_sample_predictions.min()))
print("Max:", float(v2_sample_predictions.max()))

# ============================================================================
# FINAL
# ============================================================================

print("\n" + "=" * 100)
print("✅ CELL 21 COMPLETE")
print("=" * 100)

print("""
V1 → stored V1 final-score matrix
V2 → frozen Rich-6 CatBoost
V3 → Semantic + Rich-6 CatBoost

Next cell:
Generate predictions for ALL 2,540 benchmark pairs.
""")

In [ ]:
print("=" * 100)
print("V3 CELL 22 — GENERATING V1 / V2 / V3 BENCHMARK PREDICTIONS")
print("=" * 100)

import os
import numpy as np
import pandas as pd

# ============================================================================
# PATHS
# ============================================================================

PREDICTIONS_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_common_benchmark_predictions_2540.csv"
)

PREDICTIONS_PKL = os.path.join(
    V3_OUTPUT_DIR,
    "v3_common_benchmark_predictions_2540.pkl"
)

# ============================================================================
# COPY BENCHMARK
# ============================================================================

eval_df = benchmark.copy()

# ============================================================================
# BASIC INDEX VALIDATION
# ============================================================================

print("\nBenchmark rows:", len(eval_df))

assert len(eval_df) == 2540

print(
    "Resume index range:",
    int(eval_df["Resume_Index"].min()),
    "to",
    int(eval_df["Resume_Index"].max())
)

print(
    "Job index range:",
    int(eval_df["Job_Index"].min()),
    "to",
    int(eval_df["Job_Index"].max())
)

# ============================================================================
# V1 PREDICTIONS
#
# Stored V1 matrix:
#   rows    = Job_Index
#   columns = Resume_Index
#   shape   = (434, 1540)
# ============================================================================

print("\n" + "=" * 100)
print("GENERATING V1 PREDICTIONS")
print("=" * 100)

eval_df["V1_Score"] = [
    float(
        v1_final_matrix[
            int(job_idx),
            int(resume_idx)
        ]
    )
    for job_idx, resume_idx
    in zip(
        eval_df["Job_Index"],
        eval_df["Resume_Index"]
    )
]

print("✅ V1 predictions generated.")

# ============================================================================
# V2 PREDICTIONS
# ============================================================================

print("\n" + "=" * 100)
print("GENERATING V2 PREDICTIONS")
print("=" * 100)

X_v2_eval = eval_df[
    V2_FEATURES
].copy()

eval_df["V2_Score"] = v2_model.predict(
    X_v2_eval
)

print("✅ V2 predictions generated.")

# ============================================================================
# V3 PREDICTIONS
# ============================================================================

print("\n" + "=" * 100)
print("GENERATING V3 PREDICTIONS")
print("=" * 100)

V3_FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

X_v3_eval = eval_df[
    V3_FEATURES
].copy()

eval_df["V3_Score"] = v3_model.predict(
    X_v3_eval
)

print("✅ V3 predictions generated.")

# ============================================================================
# PREDICTION SANITY CHECK
# ============================================================================

print("\n" + "=" * 100)
print("PREDICTION SANITY CHECK")
print("=" * 100)

prediction_columns = [
    "V1_Score",
    "V2_Score",
    "V3_Score"
]

print("\nNaN counts:")
print(
    eval_df[prediction_columns]
    .isna()
    .sum()
)

assert (
    eval_df[prediction_columns]
    .isna()
    .sum()
    .sum()
    == 0
)

print("\nV1 range:")
print(
    "Min:", float(eval_df["V1_Score"].min()),
    "Max:", float(eval_df["V1_Score"].max())
)

print("\nV2 range:")
print(
    "Min:", float(eval_df["V2_Score"].min()),
    "Max:", float(eval_df["V2_Score"].max())
)

print("\nV3 range:")
print(
    "Min:", float(eval_df["V3_Score"].min()),
    "Max:", float(eval_df["V3_Score"].max())
)

# ============================================================================
# CHECK THAT EACH RESUME HAS EXACTLY 10 BENCHMARK JOBS
# ============================================================================

rows_per_resume = (
    eval_df
    .groupby("Resume_Index")
    .size()
)

assert len(rows_per_resume) == 254
assert (rows_per_resume == 10).all()

print(
    "\n✅ Every benchmark resume has exactly 10 jobs."
)

# ============================================================================
# CHECK UNIQUE PAIRS
# ============================================================================

pair_duplicates = (
    eval_df[
        ["Resume_Index", "Job_Index"]
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate resume-job pairs:",
    pair_duplicates
)

assert pair_duplicates == 0

# ============================================================================
# SHOW SAMPLE
# ============================================================================

print("\n" + "=" * 100)
print("SAMPLE PREDICTIONS")
print("=" * 100)

sample_columns = [
    "Pair_ID",
    "Resume_Index",
    "Job_Index",
    "Label",
    "V1_Score",
    "V2_Score",
    "V3_Score"
]

print(
    eval_df[sample_columns]
    .head(15)
    .to_string(index=False)
)

# ============================================================================
# SAVE
# ============================================================================

eval_df.to_csv(
    PREDICTIONS_CSV,
    index=False
)

eval_df.to_pickle(
    PREDICTIONS_PKL
)

print("\n" + "=" * 100)
print("✅ PREDICTIONS SAVED")
print("=" * 100)

print(PREDICTIONS_CSV)
print(PREDICTIONS_PKL)

print("\nShape:", eval_df.shape)

print("\n" + "=" * 100)
print("🚀 CELL 22 COMPLETE")
print("=" * 100)

print("""
Next:
    Evaluate V1 vs V2 vs V3 on the SAME 2,540 benchmark pairs.

    Metrics:
        Top-1
        Hit@3
        Hit@5
        NDCG
        Pairwise Accuracy
""")

In [ ]:
print("=" * 100)
print("V3 CELL 23 — V1 vs V2 vs V3 RANKING EVALUATION")
print("=" * 100)

import os
import numpy as np
import pandas as pd

# ============================================================================
# LOAD SAVED PREDICTIONS
# ============================================================================

PREDICTIONS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_common_benchmark_predictions_2540.csv"
)

eval_df = pd.read_csv(PREDICTIONS_PATH)

print("Loaded:", eval_df.shape)

assert len(eval_df) == 2540

# ============================================================================
# CONFIG
# ============================================================================

MODELS = {
    "V1": "V1_Score",
    "V2": "V2_Score",
    "V3": "V3_Score"
}

TOP_K_VALUES = [1, 3, 5]

# ============================================================================
# NDCG
# ============================================================================
#
# Graded relevance:
#   Weak     = 0
#   Moderate = 1
#   Strong   = 2
#
# Standard DCG:
#   sum((2^rel - 1) / log2(rank + 1))
#
# We normalize against the ideal ranking for that same resume.
# ============================================================================

def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances, dtype=float)[:k]

    if len(relevances) == 0:
        return 0.0

    positions = np.arange(1, len(relevances) + 1)

    gains = (2 ** relevances) - 1
    discounts = np.log2(positions + 1)

    return float(
        np.sum(gains / discounts)
    )


def ndcg_at_k(y_true, y_score, k=None):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    if k is None:
        k = len(y_true)

    # Stable sorting makes ties deterministic.
    predicted_order = np.argsort(
        -y_score,
        kind="mergesort"
    )

    ideal_order = np.argsort(
        -y_true,
        kind="mergesort"
    )

    predicted_relevance = y_true[
        predicted_order
    ]

    ideal_relevance = y_true[
        ideal_order
    ]

    dcg = dcg_at_k(
        predicted_relevance,
        k
    )

    ideal_dcg = dcg_at_k(
        ideal_relevance,
        k
    )

    if ideal_dcg == 0:
        return 0.0

    return float(
        dcg / ideal_dcg
    )


# ============================================================================
# TOP-1 / HIT@K
# ============================================================================
#
# Top-1:
#   Did the model place one of the best-label jobs first?
#
# Hit@K:
#   Does the top-K contain at least one job whose label equals the
#   best label available for that resume?
# ============================================================================

def top1_hit(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    predicted_order = np.argsort(
        -y_score,
        kind="mergesort"
    )

    top_label = y_true[
        predicted_order[0]
    ]

    best_label = np.max(y_true)

    return int(
        top_label == best_label
    )


def hit_at_k(y_true, y_score, k):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    predicted_order = np.argsort(
        -y_score,
        kind="mergesort"
    )

    top_k_labels = y_true[
        predicted_order[:k]
    ]

    best_label = np.max(y_true)

    return int(
        np.any(top_k_labels == best_label)
    )


# ============================================================================
# PAIRWISE ACCURACY
# ============================================================================
#
# Only pairs with DIFFERENT ground-truth labels are informative.
#
# For every pair (i,j):
#
#   true_i > true_j AND score_i > score_j  -> correct
#   true_i < true_j AND score_i < score_j  -> correct
#   otherwise                              -> incorrect
#
# Predicted-score ties are therefore incorrect for a strictly ordered
# ground-truth pair.
# ============================================================================

def pairwise_accuracy(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n = len(y_true)

    correct = 0
    total = 0

    for i in range(n):
        for j in range(i + 1, n):

            # Ignore ground-truth ties
            if y_true[i] == y_true[j]:
                continue

            total += 1

            true_direction = (
                1
                if y_true[i] > y_true[j]
                else -1
            )

            score_direction = (
                1
                if y_score[i] > y_score[j]
                else
                -1
                if y_score[i] < y_score[j]
                else 0
            )

            if true_direction == score_direction:
                correct += 1

    if total == 0:
        return np.nan

    return float(
        correct / total
    )


# ============================================================================
# EVALUATE MODEL
# ============================================================================

results = []

per_resume_rows = []

for model_name, score_column in MODELS.items():

    print("\n" + "=" * 100)
    print(f"EVALUATING {model_name}")
    print("=" * 100)

    top1_values = []
    hit3_values = []
    hit5_values = []
    ndcg_values = []
    pairwise_values = []

    for resume_idx, group in (
        eval_df
        .groupby("Resume_Index", sort=True)
    ):

        y_true = group["Label"].to_numpy(
            dtype=float
        )

        y_score = group[score_column].to_numpy(
            dtype=float
        )

        top1 = top1_hit(
            y_true,
            y_score
        )

        hit3 = hit_at_k(
            y_true,
            y_score,
            3
        )

        hit5 = hit_at_k(
            y_true,
            y_score,
            5
        )

        ndcg = ndcg_at_k(
            y_true,
            y_score,
            k=len(y_true)
        )

        pairwise = pairwise_accuracy(
            y_true,
            y_score
        )

        top1_values.append(top1)
        hit3_values.append(hit3)
        hit5_values.append(hit5)
        ndcg_values.append(ndcg)
        pairwise_values.append(pairwise)

        per_resume_rows.append({
            "Model": model_name,
            "Resume_Index": int(resume_idx),
            "Top1": top1,
            "Hit@3": hit3,
            "Hit@5": hit5,
            "NDCG": ndcg,
            "Pairwise_Accuracy": pairwise
        })

    summary = {
        "Model": model_name,
        "Top-1": np.mean(top1_values),
        "Hit@3": np.mean(hit3_values),
        "Hit@5": np.mean(hit5_values),
        "NDCG": np.mean(ndcg_values),
        "Pairwise Accuracy": np.nanmean(pairwise_values)
    }

    results.append(summary)

# ============================================================================
# RESULTS TABLE
# ============================================================================

results_df = pd.DataFrame(
    results
)

print("\n" + "=" * 100)
print("🏆 MAIN RANKING RESULTS")
print("=" * 100)

display(
    results_df.style.format({
        "Top-1": "{:.6f}",
        "Hit@3": "{:.6f}",
        "Hit@5": "{:.6f}",
        "NDCG": "{:.6f}",
        "Pairwise Accuracy": "{:.6f}"
    })
)

# ============================================================================
# PERCENTAGE VIEW
# ============================================================================

percentage_df = results_df.copy()

for col in [
    "Top-1",
    "Hit@3",
    "Hit@5",
    "NDCG",
    "Pairwise Accuracy"
]:
    percentage_df[col] *= 100

print("\n" + "=" * 100)
print("PERCENTAGE VIEW")
print("=" * 100)

display(
    percentage_df.style.format({
        "Top-1": "{:.2f}%",
        "Hit@3": "{:.2f}%",
        "Hit@5": "{:.2f}%",
        "NDCG": "{:.2f}%",
        "Pairwise Accuracy": "{:.2f}%"
    })
)

# ============================================================================
# DETERMINE WINNERS
# ============================================================================

print("\n" + "=" * 100)
print("METRIC WINNERS")
print("=" * 100)

metric_columns = [
    "Top-1",
    "Hit@3",
    "Hit@5",
    "NDCG",
    "Pairwise Accuracy"
]

for metric in metric_columns:

    best_idx = results_df[metric].idxmax()

    best_model = results_df.loc[
        best_idx,
        "Model"
    ]

    best_value = results_df.loc[
        best_idx,
        metric
    ]

    print(
        f"{metric:20s} → "
        f"{best_model} "
        f"({best_value:.6f})"
    )

# ============================================================================
# V2 → V3 IMPROVEMENT
# ============================================================================

v2_row = results_df[
    results_df["Model"] == "V2"
].iloc[0]

v3_row = results_df[
    results_df["Model"] == "V3"
].iloc[0]

print("\n" + "=" * 100)
print("V2 → V3 IMPROVEMENT")
print("=" * 100)

improvement_rows = []

for metric in metric_columns:

    v2_value = v2_row[metric]
    v3_value = v3_row[metric]

    absolute_change = (
        v3_value - v2_value
    )

    relative_change = (
        absolute_change / v2_value * 100
        if v2_value != 0
        else np.nan
    )

    improvement_rows.append({
        "Metric": metric,
        "V2": v2_value,
        "V3": v3_value,
        "Absolute Δ": absolute_change,
        "Relative Δ (%)": relative_change
    })

improvement_df = pd.DataFrame(
    improvement_rows
)

display(
    improvement_df.style.format({
        "V2": "{:.6f}",
        "V3": "{:.6f}",
        "Absolute Δ": "{:+.6f}",
        "Relative Δ (%)": "{:+.2f}%"
    })
)

# ============================================================================
# PER-RESUME RESULTS
# ============================================================================

per_resume_df = pd.DataFrame(
    per_resume_rows
)

# ============================================================================
# SAVE EVERYTHING
# ============================================================================

RESULTS_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_v1_v2_v3_ranking_results_2540.csv"
)

PER_RESUME_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_v1_v2_v3_per_resume_ranking_results_2540.csv"
)

IMPROVEMENT_CSV = os.path.join(
    V3_OUTPUT_DIR,
    "v3_v2_to_v3_improvement_2540.csv"
)

results_df.to_csv(
    RESULTS_CSV,
    index=False
)

per_resume_df.to_csv(
    PER_RESUME_CSV,
    index=False
)

improvement_df.to_csv(
    IMPROVEMENT_CSV,
    index=False
)

print("\nSaved:")
print(RESULTS_CSV)
print(PER_RESUME_CSV)
print(IMPROVEMENT_CSV)

# ============================================================================
# EXTRA DIAGNOSTIC — STRONG-ONLY HIT RATES
# ============================================================================
#
# This is NOT part of the main benchmark metrics.
# It tells us how often the model puts an actually Strong (label=2)
# job into its top-K.
# ============================================================================

print("\n" + "=" * 100)
print("EXTRA DIAGNOSTIC — STRONG-ONLY HIT RATES")
print("=" * 100)

strong_rows = []

for model_name, score_column in MODELS.items():

    strong3 = []
    strong5 = []

    for _, group in eval_df.groupby(
        "Resume_Index",
        sort=True
    ):

        y_true = group["Label"].to_numpy(
            dtype=float
        )

        y_score = group[score_column].to_numpy(
            dtype=float
        )

        order = np.argsort(
            -y_score,
            kind="mergesort"
        )

        strong3.append(
            int(
                np.any(
                    y_true[order[:3]] == 2
                )
            )
        )

        strong5.append(
            int(
                np.any(
                    y_true[order[:5]] == 2
                )
            )
        )

    strong_rows.append({
        "Model": model_name,
        "Strong@3": np.mean(strong3),
        "Strong@5": np.mean(strong5)
    })

strong_df = pd.DataFrame(
    strong_rows
)

display(
    strong_df.style.format({
        "Strong@3": "{:.2%}",
        "Strong@5": "{:.2%}"
    })
)

print("\n" + "=" * 100)
print("🚀 CELL 23 COMPLETE")
print("=" * 100)

In [ ]:
print("=" * 100)
print("V3 CELL 24 — PAIRED STATISTICAL COMPARISON")
print("=" * 100)

import os
import numpy as np
import pandas as pd

from scipy.stats import (
    wilcoxon,
    binomtest
)

# ============================================================================
# LOAD PER-RESUME RESULTS
# ============================================================================

PER_RESUME_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_v1_v2_v3_per_resume_ranking_results_2540.csv"
)

per_resume = pd.read_csv(
    PER_RESUME_PATH
)

print("Loaded:", per_resume.shape)

assert per_resume["Resume_Index"].nunique() == 254

# ============================================================================
# PIVOT TO ONE ROW PER RESUME
# ============================================================================

wide = (
    per_resume
    .pivot(
        index="Resume_Index",
        columns="Model",
        values=[
            "Top1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise_Accuracy"
        ]
    )
)

# Flatten MultiIndex columns
wide.columns = [
    f"{metric}_{model}"
    for metric, model in wide.columns
]

wide = wide.reset_index()

print("\nWide shape:", wide.shape)

# ============================================================================
# FUNCTION: COMPARE V2 AND V3
# ============================================================================

def compare_metric(metric):

    v2 = wide[f"{metric}_V2"].to_numpy(
        dtype=float
    )

    v3 = wide[f"{metric}_V3"].to_numpy(
        dtype=float
    )

    diff = v3 - v2

    wins = int(np.sum(diff > 0))
    losses = int(np.sum(diff < 0))
    ties = int(np.sum(diff == 0))

    # Exact sign test among non-ties
    non_ties = wins + losses

    if non_ties > 0:
        sign_test = binomtest(
            wins,
            n=non_ties,
            p=0.5,
            alternative="two-sided"
        )
        sign_p = sign_test.pvalue
    else:
        sign_p = np.nan

    # Wilcoxon on non-zero differences
    nonzero_diff = diff[diff != 0]

    if len(nonzero_diff) > 0:
        try:
            wilcoxon_result = wilcoxon(
                nonzero_diff,
                alternative="two-sided"
            )
            wilcoxon_stat = float(
                wilcoxon_result.statistic
            )
            wilcoxon_p = float(
                wilcoxon_result.pvalue
            )
        except Exception:
            wilcoxon_stat = np.nan
            wilcoxon_p = np.nan
    else:
        wilcoxon_stat = np.nan
        wilcoxon_p = np.nan

    # Bootstrap confidence interval for mean improvement
    rng = np.random.default_rng(42)

    bootstrap_means = []

    for _ in range(10000):
        sample = rng.choice(
            diff,
            size=len(diff),
            replace=True
        )

        bootstrap_means.append(
            np.mean(sample)
        )

    bootstrap_means = np.asarray(
        bootstrap_means
    )

    ci_low, ci_high = np.percentile(
        bootstrap_means,
        [2.5, 97.5]
    )

    return {
        "Metric": metric,
        "V2_Mean": float(np.mean(v2)),
        "V3_Mean": float(np.mean(v3)),
        "Mean_Δ": float(np.mean(diff)),
        "V3_Wins": wins,
        "V2_Wins": losses,
        "Ties": ties,
        "Sign_Test_p": float(sign_p),
        "Wilcoxon_p": float(wilcoxon_p),
        "Bootstrap_95%_CI_Low": float(ci_low),
        "Bootstrap_95%_CI_High": float(ci_high)
    }


# ============================================================================
# RUN COMPARISONS
# ============================================================================

metrics = [
    "Top1",
    "Hit@3",
    "Hit@5",
    "NDCG",
    "Pairwise_Accuracy"
]

stat_rows = []

for metric in metrics:

    print(
        f"\nAnalyzing {metric}..."
    )

    stat_rows.append(
        compare_metric(metric)
    )

stats_df = pd.DataFrame(
    stat_rows
)

# ============================================================================
# DISPLAY
# ============================================================================

print("\n" + "=" * 100)
print("📊 V2 vs V3 PAIRED STATISTICAL RESULTS")
print("=" * 100)

display(
    stats_df.style.format({
        "V2_Mean": "{:.6f}",
        "V3_Mean": "{:.6f}",
        "Mean_Δ": "{:+.6f}",
        "Sign_Test_p": "{:.6g}",
        "Wilcoxon_p": "{:.6g}",
        "Bootstrap_95%_CI_Low": "{:+.6f}",
        "Bootstrap_95%_CI_High": "{:+.6f}"
    })
)

# ============================================================================
# HUMAN-READABLE SUMMARY
# ============================================================================

print("\n" + "=" * 100)
print("V3 WIN / LOSS SUMMARY")
print("=" * 100)

for _, row in stats_df.iterrows():

    metric = row["Metric"]

    print(
        f"\n{metric}"
    )

    print(
        f"  V3 wins:  {int(row['V3_Wins'])}"
    )

    print(
        f"  V2 wins:  {int(row['V2_Wins'])}"
    )

    print(
        f"  Ties:     {int(row['Ties'])}"
    )

    print(
        f"  Mean Δ:   {row['Mean_Δ']:+.6f}"
    )

    print(
        f"  Sign-test p: {row['Sign_Test_p']:.6g}"
    )

    print(
        f"  Wilcoxon p:  {row['Wilcoxon_p']:.6g}"
    )

    print(
        f"  95% bootstrap CI: "
        f"[{row['Bootstrap_95%_CI_Low']:+.6f}, "
        f"{row['Bootstrap_95%_CI_High']:+.6f}]"
    )

# ============================================================================
# SAVE
# ============================================================================

STATS_PATH = os.path.join(
    V3_OUTPUT_DIR,
    "v3_v2_paired_statistical_comparison_2540.csv"
)

stats_df.to_csv(
    STATS_PATH,
    index=False
)

print("\n✅ Saved:")
print(STATS_PATH)

# ============================================================================
# V3 TOP-1 VS V2 — EXACT COUNTS
# ============================================================================

print("\n" + "=" * 100)
print("TOP-1 EXACT RESUME COUNTS")
print("=" * 100)

for model in ["V1", "V2", "V3"]:

    count = int(
        wide[f"Top1_{model}"].sum()
    )

    print(
        f"{model}: {count} / 254 resumes"
    )

print("\n" + "=" * 100)
print("🚀 CELL 24 COMPLETE")
print("=" * 100)

In [ ]:
print("=" * 100)
print("SEARCHING V2 PROJECT FOR NOTEBOOKS / PYTHON SOURCE")
print("=" * 100)

import os

ROOT = "/content/drive/MyDrive/Resume_Model_Project/V2_100k"

for root, dirs, files in os.walk(ROOT):
    for file in files:
        if file.endswith((".ipynb", ".py")):
            print(os.path.join(root, file))

In [ ]:
# ============================================================
# V3 REPEATED UNSEEN-BENCHMARK EVALUATION
# 10 x (254 resumes × 10 jobs)
# V1 vs V2 vs V3
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

V3_DATA = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

V3_MODEL = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_semantic_rich7_catboost.cbm"
)

V2_MODEL = os.path.join(
    ROOT,
    "V2_100k",
    "Outputs",
    "v2_final_catboost_rich6_all100k.cbm"
)

OUT_DIR = os.path.join(ROOT, "V3", "Outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 100)
print("V3 REPEATED UNSEEN-BENCHMARK EVALUATION")
print("=" * 100)

# ------------------------------------------------------------
# 1. Load V3 labelled dataset
# ------------------------------------------------------------

df = pd.read_csv(V3_DATA)

print(f"\n✅ V3 dataset loaded: {df.shape}")
print("Unique resumes:", df["Resume_Index"].nunique())
print("Unique jobs:", df["Job_Index"].nunique())

# ------------------------------------------------------------
# 2. Reconstruct the SAME resume-level 80/20 split
#    used for V3 training
#
#    First preference: reuse existing Cell 19 variables if
#    they are still present.
# ------------------------------------------------------------

all_resumes = np.sort(df["Resume_Index"].unique())

val_resumes = None

# Try common variable names that may still exist from Cell 19
for name in [
    "val_resumes",
    "val_resume_indices",
    "validation_resumes",
    "val_resume_set"
]:
    if name in globals():
        candidate = globals()[name]

        try:
            candidate = np.array(list(candidate))
            if len(candidate) == 308:
                val_resumes = np.sort(candidate)
                print(f"\n✅ Reusing existing Cell 19 variable: {name}")
                break
        except Exception:
            pass

# If Cell 19 variables are gone, reproduce the split
# using the same seed and 20% resume-level validation split.
if val_resumes is None:

    train_resumes_rebuilt, val_resumes_rebuilt = train_test_split(
        all_resumes,
        test_size=0.20,
        random_state=42
    )

    train_resumes_rebuilt = np.sort(train_resumes_rebuilt)
    val_resumes_rebuilt = np.sort(val_resumes_rebuilt)

    val_resumes = val_resumes_rebuilt

    print("\n⚠️ Cell 19 resume split variables were unavailable.")
    print("Reconstructed using train_test_split(test_size=0.20, random_state=42).")

print("\nResume split:")
print("Total resumes :", len(all_resumes))
print("Validation    :", len(val_resumes))

assert len(all_resumes) == 1540
assert len(val_resumes) == 308, (
    f"Expected 308 validation resumes, got {len(val_resumes)}"
)

# ------------------------------------------------------------
# 3. Keep ONLY rows belonging to held-out validation resumes
# ------------------------------------------------------------

val_df = df[df["Resume_Index"].isin(val_resumes)].copy()

print("\nHeld-out validation pool:")
print("Rows:", len(val_df))
print("Resumes:", val_df["Resume_Index"].nunique())

# Every validation resume should have many jobs available
jobs_per_resume = val_df.groupby("Resume_Index").size()

print(
    "Jobs/resume:",
    f"min={jobs_per_resume.min()}, "
    f"max={jobs_per_resume.max()}, "
    f"mean={jobs_per_resume.mean():.2f}"
)

assert val_df["Resume_Index"].nunique() == 308
assert jobs_per_resume.min() >= 10

# ------------------------------------------------------------
# 4. Load exact frozen V2 + V3 models
# ------------------------------------------------------------

v2_model = CatBoostRegressor()
v2_model.load_model(V2_MODEL)

v3_model = CatBoostRegressor()
v3_model.load_model(V3_MODEL)

print("\n✅ V2 model loaded")
print("✅ V3 model loaded")

# Exact frozen feature order
V2_FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
]

V3_FEATURES = V2_FEATURES + [
    "Semantic_Similarity"
]

print("\nV2 features:", V2_FEATURES)
print("V3 features:", V3_FEATURES)

# ------------------------------------------------------------
# 5. Ranking metric functions
# ------------------------------------------------------------

def top1_metric(group):
    best_label = group["Label"].max()
    top_row = group.sort_values(
        "Predicted_Score",
        ascending=False
    ).iloc[0]

    return float(top_row["Label"] == best_label)


def hit_at_k(group, k):
    best_label = group["Label"].max()

    top_k = group.sort_values(
        "Predicted_Score",
        ascending=False
    ).head(k)

    return float((top_k["Label"] == best_label).any())


def ndcg_metric(group):
    """
    Standard graded NDCG:
        gain = 2^relevance - 1
    Uses the same 10-job ranking setup.
    """

    ranked = group.sort_values(
        "Predicted_Score",
        ascending=False
    )

    relevance = ranked["Label"].to_numpy(dtype=float)

    discounts = np.log2(
        np.arange(2, len(relevance) + 2)
    )

    gains = (2 ** relevance) - 1

    dcg = np.sum(gains / discounts)

    ideal_relevance = np.sort(
        relevance
    )[::-1]

    ideal_gains = (2 ** ideal_relevance) - 1

    idcg = np.sum(ideal_gains / discounts)

    if idcg == 0:
        return 0.0

    return float(dcg / idcg)


def pairwise_metric(group):
    """
    Pairwise ranking accuracy.

    For every pair with unequal ground-truth labels,
    predicted ordering must agree strictly.

    Ties in predicted score count as incorrect.
    """

    labels = group["Label"].to_numpy()
    scores = group["Predicted_Score"].to_numpy()

    correct = 0
    total = 0

    n = len(group)

    for i in range(n):
        for j in range(i + 1, n):

            if labels[i] == labels[j]:
                continue

            total += 1

            true_diff = labels[i] - labels[j]
            pred_diff = scores[i] - scores[j]

            if true_diff * pred_diff > 0:
                correct += 1

    if total == 0:
        return np.nan

    return correct / total


def evaluate_predictions(pred_df):
    grouped = pred_df.groupby(
        "Resume_Index",
        sort=False
    )

    return {
        "Top-1": grouped.apply(top1_metric).mean(),
        "Hit@3": grouped.apply(lambda g: hit_at_k(g, 3)).mean(),
        "Hit@5": grouped.apply(lambda g: hit_at_k(g, 5)).mean(),
        "NDCG": grouped.apply(ndcg_metric).mean(),
        "Pairwise": grouped.apply(pairwise_metric).mean(),
    }


# ------------------------------------------------------------
# 6. Run 10 repeated samples
# ------------------------------------------------------------

TRIAL_RESULTS = []
ALL_SAMPLE_ROWS = []

BASE_SEEDS = list(range(1001, 1011))

print("\n" + "=" * 100)
print("STARTING 10 TRIALS")
print("=" * 100)

for trial_no, seed in enumerate(BASE_SEEDS, start=1):

    rng = np.random.RandomState(seed)

    # --------------------------------------------------------
    # Sample 254 VALIDATION resumes
    # --------------------------------------------------------

    sampled_resumes = rng.choice(
        val_resumes,
        size=254,
        replace=False
    )

    sampled_resumes = np.sort(sampled_resumes)

    # --------------------------------------------------------
    # For each resume, sample exactly 10 validation jobs
    # --------------------------------------------------------

    chunks = []

    for resume_idx in sampled_resumes:

        g = val_df[
            val_df["Resume_Index"] == resume_idx
        ]

        sampled_rows = g.sample(
            n=10,
            random_state=seed + int(resume_idx)
        )

        chunks.append(sampled_rows)

    sample_df = pd.concat(
        chunks,
        ignore_index=True
    )

    # Safety checks
    assert len(sample_df) == 2540
    assert sample_df["Resume_Index"].nunique() == 254
    assert (
        sample_df.groupby("Resume_Index").size() == 10
    ).all()

    # --------------------------------------------------------
    # V1 prediction
    # --------------------------------------------------------

    sample_df["V1_Score"] = (
        0.25 * sample_df["Normalized_Cosine"]
        + 0.75 * sample_df["Skill_Score"]
    )

    # --------------------------------------------------------
    # V2 prediction
    # --------------------------------------------------------

    sample_df["V2_Score"] = v2_model.predict(
        sample_df[V2_FEATURES]
    )

    # --------------------------------------------------------
    # V3 prediction
    # --------------------------------------------------------

    sample_df["V3_Score"] = v3_model.predict(
        sample_df[V3_FEATURES]
    )

    # --------------------------------------------------------
    # Evaluate each model
    # --------------------------------------------------------

    trial_metric_rows = []

    for model_name, score_col in [
        ("V1", "V1_Score"),
        ("V2", "V2_Score"),
        ("V3", "V3_Score"),
    ]:

        eval_df = sample_df[
            [
                "Resume_Index",
                "Job_Index",
                "Label",
                score_col
            ]
        ].copy()

        eval_df = eval_df.rename(
            columns={score_col: "Predicted_Score"}
        )

        metrics = evaluate_predictions(eval_df)

        trial_metric_rows.append({
            "Trial": trial_no,
            "Seed": seed,
            "Model": model_name,
            **metrics
        })

    # Save sampled rows for this trial
    trial_saved = sample_df.copy()
    trial_saved["Trial"] = trial_no
    trial_saved["Seed"] = seed

    ALL_SAMPLE_ROWS.append(trial_saved)

    TRIAL_RESULTS.extend(trial_metric_rows)

    print(
        f"\nTrial {trial_no:02d}/10 "
        f"(seed={seed}) completed — 254 resumes × 10 jobs = 2540 pairs"
    )

# ------------------------------------------------------------
# 7. Results table
# ------------------------------------------------------------

results_df = pd.DataFrame(TRIAL_RESULTS)

print("\n" + "=" * 100)
print("PER-TRIAL RESULTS")
print("=" * 100)

print(
    results_df.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 8. Wide table for easier V2 vs V3 comparison
# ------------------------------------------------------------

wide = results_df.pivot(
    index="Trial",
    columns="Model",
    values=["Top-1", "Hit@3", "Hit@5", "NDCG", "Pairwise"]
)

wide.columns = [
    f"{metric}_{model}"
    for metric, model in wide.columns
]

wide = wide.reset_index()

# ------------------------------------------------------------
# 9. Mean / Std summary
# ------------------------------------------------------------

summary_rows = []

for model in ["V1", "V2", "V3"]:

    subset = results_df[
        results_df["Model"] == model
    ]

    summary_rows.append({
        "Model": model,

        "Top-1 Mean": subset["Top-1"].mean(),
        "Top-1 Std": subset["Top-1"].std(ddof=1),

        "Hit@3 Mean": subset["Hit@3"].mean(),
        "Hit@3 Std": subset["Hit@3"].std(ddof=1),

        "Hit@5 Mean": subset["Hit@5"].mean(),
        "Hit@5 Std": subset["Hit@5"].std(ddof=1),

        "NDCG Mean": subset["NDCG"].mean(),
        "NDCG Std": subset["NDCG"].std(ddof=1),

        "Pairwise Mean": subset["Pairwise"].mean(),
        "Pairwise Std": subset["Pairwise"].std(ddof=1),
    })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 100)
print("10-TRIAL SUMMARY")
print("=" * 100)

print(
    summary_df.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 10. V2 → V3 improvement
# ------------------------------------------------------------

v2 = summary_df[
    summary_df["Model"] == "V2"
].iloc[0]

v3 = summary_df[
    summary_df["Model"] == "V3"
].iloc[0]

improvements = pd.DataFrame({
    "Metric": [
        "Top-1",
        "Hit@3",
        "Hit@5",
        "NDCG",
        "Pairwise"
    ],
    "V2 Mean": [
        v2["Top-1 Mean"],
        v2["Hit@3 Mean"],
        v2["Hit@5 Mean"],
        v2["NDCG Mean"],
        v2["Pairwise Mean"]
    ],
    "V3 Mean": [
        v3["Top-1 Mean"],
        v3["Hit@3 Mean"],
        v3["Hit@5 Mean"],
        v3["NDCG Mean"],
        v3["Pairwise Mean"]
    ]
})

improvements["Absolute Delta"] = (
    improvements["V3 Mean"]
    - improvements["V2 Mean"]
)

improvements["Relative Improvement %"] = (
    improvements["Absolute Delta"]
    / improvements["V2 Mean"]
    * 100
)

print("\n" + "=" * 100)
print("V2 → V3 IMPROVEMENT")
print("=" * 100)

print(
    improvements.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 11. Save everything
# ------------------------------------------------------------

trial_path = os.path.join(
    OUT_DIR,
    "v3_10x_unseen_validation_trial_results.csv"
)

summary_path = os.path.join(
    OUT_DIR,
    "v3_10x_unseen_validation_summary.csv"
)

improvement_path = os.path.join(
    OUT_DIR,
    "v3_10x_unseen_validation_v2_to_v3_improvement.csv"
)

samples_path = os.path.join(
    OUT_DIR,
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

results_df.to_csv(
    trial_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)

improvements.to_csv(
    improvement_path,
    index=False
)

all_samples_df = pd.concat(
    ALL_SAMPLE_ROWS,
    ignore_index=True
)

all_samples_df.to_csv(
    samples_path,
    index=False
)

# ------------------------------------------------------------
# 12. Final sanity checks
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL CHECKS")
print("=" * 100)

print("Total trials:", results_df["Trial"].nunique())
print("Rows evaluated per trial: 2540")
print("Expected total sampled rows:", 10 * 2540)
print("Actual sampled rows:", len(all_samples_df))

print(
    "\n✅ Files saved:"
)

print(trial_path)
print(summary_path)
print(improvement_path)
print(samples_path)

print("\n" + "=" * 100)
print("DONE 🔥")
print("=" * 100)

In [ ]:
from warnings import filterwarnings

filterwarnings("ignore")

In [ ]:
# ============================================================
# V3 IMPROVEMENT — BUILD FULL VERIFIED LABEL POOL
# ============================================================

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

V2_PATH = os.path.join(
    ROOT,
    "V2_100k",
    "Outputs",
    "v2_supervised_dataset_100k_rich.csv"
)

V3_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

print("=" * 100)
print("BUILDING FULL VERIFIED LABEL POOL")
print("=" * 100)

v2_df = pd.read_csv(V2_PATH)
v3_df = pd.read_csv(V3_PATH)

print("\nV2 dataset:", v2_df.shape)
print("V3 dataset:", v3_df.shape)

# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------

print("\nV2 columns:")
print(v2_df.columns.tolist())

print("\nV3 columns:")
print(v3_df.columns.tolist())

# ------------------------------------------------------------
# Compare pair identities
# ------------------------------------------------------------

v2_pairs = set(
    zip(
        v2_df["Resume_Index"],
        v2_df["Job_Index"]
    )
)

v3_pairs = set(
    zip(
        v3_df["Resume_Index"],
        v3_df["Job_Index"]
    )
)

overlap = v2_pairs & v3_pairs

print("\nPair overlap:")
print("V2 unique pairs:", len(v2_pairs))
print("V3 unique pairs:", len(v3_pairs))
print("Overlap:", len(overlap))

# ------------------------------------------------------------
# V3 already has semantic similarity.
# V2 does not, so calculate semantic similarity later for
# the V2-only rows using the existing full semantic matrix.
# ------------------------------------------------------------

print("\nV2-only pairs:")
print(len(v2_pairs - v3_pairs))

print("\nV3-only pairs:")
print(len(v3_pairs - v2_pairs))

# ------------------------------------------------------------
# Sanity
# ------------------------------------------------------------

assert len(overlap) == 0, (
    "V2 and V3 labelled pools overlap. "
    "Do not combine until this is investigated."
)

print("\n✅ V2 and V3 labelled pools are disjoint.")

combined_count = len(v2_pairs) + len(v3_pairs)

print("\nCombined verified labelled pairs:", combined_count)

print("\nExpected:")
print("100,000 + 568,210 = 668,210")

assert combined_count == 668210

print("\n" + "=" * 100)
print("POOL VERIFICATION COMPLETE ✅")
print("=" * 100)

In [ ]:
# ============================================================
# CLEAN V3 DATA-SIZE EXPERIMENT
#
# Baseline V3 vs Expanded V3
# Same resume-level holdout
# Same 10 x 2540 evaluation pairs
#
# IMPORTANT:
# We intentionally DO NOT rely on the historical
# 452511 / 113159 row counts because Cell 19's exact
# resume split was not preserved.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

V2_PATH = os.path.join(
    ROOT,
    "V2_100k",
    "Outputs",
    "v2_supervised_dataset_100k_rich.csv"
)

V3_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

OLD_SAMPLES_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

SEMANTIC_MATRIX_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_semantic_similarity_matrix.npy"
)

OUT_DIR = os.path.join(
    ROOT,
    "V3",
    "Outputs"
)

os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 100)
print("CLEAN V3 DATA-SIZE EXPERIMENT")
print("=" * 100)

# ============================================================
# 1. Load datasets
# ============================================================

v2_df = pd.read_csv(V2_PATH)
v3_df = pd.read_csv(V3_PATH)

eval_samples = pd.read_csv(
    OLD_SAMPLES_PATH
)

print("\nV2 labelled pool:", v2_df.shape)
print("V3 labelled pool:", v3_df.shape)
print("Existing evaluation samples:", eval_samples.shape)

assert len(v2_df) == 100000
assert len(v3_df) == 568210
assert len(eval_samples) == 25400

# ============================================================
# 2. Recover the EXACT 308-resume holdout used to generate
#    the existing 10-trial evaluation samples.
#
#    Every trial sampled its 254 resumes from this pool.
# ============================================================

holdout_resumes = np.sort(
    eval_samples["Resume_Index"].unique()
)

print("\nEvaluation resume pool:")
print("Unique resumes:", len(holdout_resumes))

assert len(holdout_resumes) == 308

print(
    "✅ These are the exact 308 resumes used as the source "
    "for the existing 10 evaluation trials."
)

# ============================================================
# 3. Training resumes = everyone else
# ============================================================

all_resumes = np.sort(
    v3_df["Resume_Index"].unique()
)

train_resumes = np.sort(
    np.setdiff1d(
        all_resumes,
        holdout_resumes
    )
)

print("\nResume split:")
print("Total resumes:", len(all_resumes))
print("Training resumes:", len(train_resumes))
print("Holdout resumes:", len(holdout_resumes))

assert len(all_resumes) == 1540
assert len(train_resumes) == 1232
assert len(holdout_resumes) == 308

assert len(
    set(train_resumes) &
    set(holdout_resumes)
) == 0

# ============================================================
# 4. Build BASELINE V3 training data
#
#    Use V3 labelled pairs belonging to only the 1232
#    training resumes.
# ============================================================

baseline_train = v3_df[
    v3_df["Resume_Index"].isin(
        train_resumes
    )
].copy()

baseline_holdout = v3_df[
    v3_df["Resume_Index"].isin(
        holdout_resumes
    )
].copy()

print("\nBaseline V3:")
print("Training rows:", len(baseline_train))
print("Holdout rows:", len(baseline_holdout))

assert not (
    baseline_train["Resume_Index"].isin(
        holdout_resumes
    )
).any()

# ============================================================
# 5. Build EXPANDED V3 training data
#
#    Add ALL V2-labelled pairs whose resumes belong to the
#    same 1232 training resumes.
# ============================================================

v2_extra = v2_df[
    v2_df["Resume_Index"].isin(
        train_resumes
    )
].copy()

print("\nV2 additional training pairs:",
      len(v2_extra))

assert not (
    v2_extra["Resume_Index"].isin(
        holdout_resumes
    )
).any()

# ============================================================
# 6. Add Semantic_Similarity to V2 rows
# ============================================================

semantic_matrix = np.load(
    SEMANTIC_MATRIX_PATH,
    mmap_mode="r"
)

print(
    "\nSemantic matrix:",
    semantic_matrix.shape
)

assert semantic_matrix.shape == (
    1540,
    434
)

v2_extra["Semantic_Similarity"] = [
    float(
        semantic_matrix[
            int(resume_idx),
            int(job_idx)
        ]
    )
    for resume_idx, job_idx in zip(
        v2_extra["Resume_Index"],
        v2_extra["Job_Index"]
    )
]

# ============================================================
# 7. Exact Rich7 feature set
# ============================================================

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity",
]

LABEL = "Label"

# ============================================================
# 8. Keep only required columns
# ============================================================

baseline_train = baseline_train[
    FEATURES + [
        LABEL,
        "Resume_Index",
        "Job_Index",
        "Pair_ID"
    ]
].copy()

v2_extra = v2_extra[
    FEATURES + [
        LABEL,
        "Resume_Index",
        "Job_Index",
        "Pair_ID"
    ]
].copy()

# ============================================================
# 9. Verify V2 extra pairs don't overlap V3 training pairs
# ============================================================

baseline_keys = set(
    zip(
        baseline_train["Resume_Index"],
        baseline_train["Job_Index"]
    )
)

extra_keys = set(
    zip(
        v2_extra["Resume_Index"],
        v2_extra["Job_Index"]
    )
)

overlap = baseline_keys & extra_keys

print(
    "\nV2/V3 training-pair overlap:",
    len(overlap)
)

assert len(overlap) == 0

# ============================================================
# 10. Expanded training pool
# ============================================================

expanded_train = pd.concat(
    [
        baseline_train,
        v2_extra
    ],
    ignore_index=True
)

print("\n" + "=" * 100)
print("TRAINING POOL SIZES")
print("=" * 100)

print(
    f"Baseline V3 training rows : {len(baseline_train):,}"
)

print(
    f"Additional V2 rows        : {len(v2_extra):,}"
)

print(
    f"Expanded V3 training rows : {len(expanded_train):,}"
)

print(
    "\nLabel distribution — Baseline:"
)

print(
    baseline_train["Label"]
    .value_counts()
    .sort_index()
)

print(
    "\nLabel distribution — Expanded:"
)

print(
    expanded_train["Label"]
    .value_counts()
    .sort_index()
)

# ============================================================
# 11. Absolute leakage checks
# ============================================================

baseline_leak = baseline_train[
    baseline_train["Resume_Index"].isin(
        holdout_resumes
    )
]

expanded_leak = expanded_train[
    expanded_train["Resume_Index"].isin(
        holdout_resumes
    )
]

print(
    "\nBaseline holdout leakage:",
    len(baseline_leak)
)

print(
    "Expanded holdout leakage:",
    len(expanded_leak)
)

assert len(baseline_leak) == 0
assert len(expanded_leak) == 0

print(
    "\n✅ ZERO RESUME-LEVEL LEAKAGE"
)

# ============================================================
# 12. Save training pools
# ============================================================

baseline_path = os.path.join(
    OUT_DIR,
    "v3_clean_baseline_training_1232resumes.csv"
)

expanded_path = os.path.join(
    OUT_DIR,
    "v3_clean_expanded_training_668k_pool_1232resumes.csv"
)

baseline_train.to_csv(
    baseline_path,
    index=False
)

expanded_train.to_csv(
    expanded_path,
    index=False
)

print("\nSaved:")
print(baseline_path)
print(expanded_path)

# ============================================================
# 13. Train BASELINE V3 on this exact split
#
#    Same architecture/hyperparameters.
#    This gives us a fair baseline under the exact same
#    holdout that the expanded model will use.
# ============================================================

print("\n" + "=" * 100)
print("TRAINING CLEAN BASELINE V3")
print("=" * 100)

baseline_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

baseline_model.fit(
    baseline_train[FEATURES],
    baseline_train[LABEL]
)

print("✅ Baseline V3 trained")

baseline_model_path = os.path.join(
    OUT_DIR,
    "v3_clean_baseline_catboost_rich7.cbm"
)

baseline_model.save_model(
    baseline_model_path
)

# ============================================================
# 14. Train EXPANDED V3
# ============================================================

print("\n" + "=" * 100)
print("TRAINING EXPANDED V3")
print("=" * 100)

expanded_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

expanded_model.fit(
    expanded_train[FEATURES],
    expanded_train[LABEL]
)

print("✅ Expanded V3 trained")

expanded_model_path = os.path.join(
    OUT_DIR,
    "v3_expanded_668k_clean_catboost_rich7.cbm"
)

expanded_model.save_model(
    expanded_model_path
)

# ============================================================
# 15. Predict on the EXACT SAME 25,400 pairs
# ============================================================

eval_samples["Clean_Baseline_V3_Score"] = (
    baseline_model.predict(
        eval_samples[FEATURES]
    )
)

eval_samples["Expanded_V3_Score"] = (
    expanded_model.predict(
        eval_samples[FEATURES]
    )
)

assert (
    eval_samples["Clean_Baseline_V3_Score"]
    .isna()
    .sum()
    == 0
)

assert (
    eval_samples["Expanded_V3_Score"]
    .isna()
    .sum()
    == 0
)

print(
    "\n✅ Predictions generated for all 25,400 evaluation pairs."
)

# ============================================================
# 16. Ranking metrics
# ============================================================

def top1_metric(group, score_col):

    best_label = group["Label"].max()

    ranked = group.sort_values(
        score_col,
        ascending=False
    )

    return float(
        ranked.iloc[0]["Label"] == best_label
    )


def hit_at_k(group, score_col, k):

    best_label = group["Label"].max()

    ranked = group.sort_values(
        score_col,
        ascending=False
    ).head(k)

    return float(
        (ranked["Label"] == best_label).any()
    )


def ndcg_metric(group, score_col):

    ranked = group.sort_values(
        score_col,
        ascending=False
    )

    relevance = ranked["Label"].to_numpy(
        dtype=float
    )

    discounts = np.log2(
        np.arange(2, len(relevance) + 2)
    )

    gains = (2 ** relevance) - 1

    dcg = np.sum(
        gains / discounts
    )

    ideal = np.sort(
        relevance
    )[::-1]

    ideal_gains = (2 ** ideal) - 1

    idcg = np.sum(
        ideal_gains / discounts
    )

    if idcg == 0:
        return 0.0

    return float(
        dcg / idcg
    )


def pairwise_metric(group, score_col):

    labels = group["Label"].to_numpy()
    scores = group[score_col].to_numpy()

    correct = 0
    total = 0

    n = len(group)

    for i in range(n):

        for j in range(i + 1, n):

            if labels[i] == labels[j]:
                continue

            total += 1

            if (
                (labels[i] - labels[j])
                *
                (scores[i] - scores[j])
                > 0
            ):
                correct += 1

    if total == 0:
        return np.nan

    return correct / total


def evaluate_model(
    data,
    score_col
):

    results = []

    for trial in sorted(
        data["Trial"].unique()
    ):

        trial_df = data[
            data["Trial"] == trial
        ]

        grouped = trial_df.groupby(
            "Resume_Index",
            sort=False
        )

        results.append({
            "Trial": trial,

            "Top-1": grouped.apply(
                lambda g:
                top1_metric(
                    g,
                    score_col
                )
            ).mean(),

            "Hit@3": grouped.apply(
                lambda g:
                hit_at_k(
                    g,
                    score_col,
                    3
                )
            ).mean(),

            "Hit@5": grouped.apply(
                lambda g:
                hit_at_k(
                    g,
                    score_col,
                    5
                )
            ).mean(),

            "NDCG": grouped.apply(
                lambda g:
                ndcg_metric(
                    g,
                    score_col
                )
            ).mean(),

            "Pairwise": grouped.apply(
                lambda g:
                pairwise_metric(
                    g,
                    score_col
                )
            ).mean()
        })

    return pd.DataFrame(
        results
    )


# ============================================================
# 17. Evaluate both models
# ============================================================

baseline_results = evaluate_model(
    eval_samples,
    "Clean_Baseline_V3_Score"
)

baseline_results["Model"] = (
    "Clean_Baseline_V3"
)

expanded_results = evaluate_model(
    eval_samples,
    "Expanded_V3_Score"
)

expanded_results["Model"] = (
    "Expanded_V3_668k"
)

all_results = pd.concat(
    [
        baseline_results,
        expanded_results
    ],
    ignore_index=True
)

# ============================================================
# 18. Per-trial comparison
# ============================================================

print("\n" + "=" * 100)
print("PER-TRIAL RESULTS")
print("=" * 100)

print(
    all_results[
        [
            "Trial",
            "Model",
            "Top-1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise"
        ]
    ]
    .round(6)
    .to_string(index=False)
)

# ============================================================
# 19. Mean comparison
# ============================================================

summary_rows = []

for model_name in [
    "Clean_Baseline_V3",
    "Expanded_V3_668k"
]:

    m = all_results[
        all_results["Model"] == model_name
    ]

    summary_rows.append({
        "Model": model_name,

        "Top-1 Mean": m["Top-1"].mean(),
        "Top-1 Std": m["Top-1"].std(
            ddof=1
        ),

        "Hit@3 Mean": m["Hit@3"].mean(),
        "Hit@3 Std": m["Hit@3"].std(
            ddof=1
        ),

        "Hit@5 Mean": m["Hit@5"].mean(),
        "Hit@5 Std": m["Hit@5"].std(
            ddof=1
        ),

        "NDCG Mean": m["NDCG"].mean(),
        "NDCG Std": m["NDCG"].std(
            ddof=1
        ),

        "Pairwise Mean": m["Pairwise"].mean(),
        "Pairwise Std": m["Pairwise"].std(
            ddof=1
        )
    })

summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 100)
print("CLEAN BASELINE V3 vs EXPANDED V3")
print("=" * 100)

print(
    summary_df.round(6).to_string(
        index=False
    )
)

# ============================================================
# 20. Calculate improvement
# ============================================================

base = summary_df[
    summary_df["Model"] ==
    "Clean_Baseline_V3"
].iloc[0]

expanded = summary_df[
    summary_df["Model"] ==
    "Expanded_V3_668k"
].iloc[0]

comparison_rows = []

for metric in [
    "Top-1",
    "Hit@3",
    "Hit@5",
    "NDCG",
    "Pairwise"
]:

    old = base[
        f"{metric} Mean"
    ]

    new = expanded[
        f"{metric} Mean"
    ]

    delta = new - old

    comparison_rows.append({
        "Metric": metric,
        "Baseline_V3_Mean": old,
        "Expanded_V3_Mean": new,
        "Absolute_Delta": delta,
        "Relative_Improvement_%": (
            delta / old * 100
        )
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

print("\n" + "=" * 100)
print("DATA EXPANSION IMPROVEMENT")
print("=" * 100)

print(
    comparison_df.round(6).to_string(
        index=False
    )
)

# ============================================================
# 21. Save outputs
# ============================================================

results_path = os.path.join(
    OUT_DIR,
    "v3_clean_datasize_experiment_trial_results.csv"
)

summary_path = os.path.join(
    OUT_DIR,
    "v3_clean_datasize_experiment_summary.csv"
)

comparison_path = os.path.join(
    OUT_DIR,
    "v3_clean_datasize_experiment_improvement.csv"
)

predictions_path = os.path.join(
    OUT_DIR,
    "v3_clean_datasize_experiment_25400_predictions.csv"
)

all_results.to_csv(
    results_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

eval_samples.to_csv(
    predictions_path,
    index=False
)

print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)

print(results_path)
print(summary_path)
print(comparison_path)
print(predictions_path)

print("\n" + "=" * 100)
print("DONE 🔥")
print("=" * 100)

In [ ]:
# ============================================================
# MODEL BATTLE — BATCH 1
# XGBoost + HistGradientBoosting + CatBoost
# RAM-SAFE / SEQUENTIAL
# ============================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    from xgboost import XGBRegressor
except ImportError:
    !pip -q install xgboost
    from xgboost import XGBRegressor

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

DATA_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

EVAL_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

CATBOOST_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_clean_baseline_catboost_rich7.cbm"
)

OUT_DIR = os.path.join(
    ROOT, "V3", "Outputs"
)

os.makedirs(OUT_DIR, exist_ok=True)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

print("=" * 100)
print("BATCH 1 — XGBOOST + HIST + CATBOOST")
print("=" * 100)

df = pd.read_csv(DATA_PATH)
eval_df = pd.read_csv(EVAL_PATH)

holdout_resumes = set(
    eval_df["Resume_Index"].unique()
)

train_df = df[
    ~df["Resume_Index"].isin(holdout_resumes)
].copy()

print("Training rows:", len(train_df))
print("Training resumes:", train_df["Resume_Index"].nunique())
print("Evaluation rows:", len(eval_df))

assert len(train_df) == 454451
assert train_df["Resume_Index"].nunique() == 1232
assert len(eval_df) == 25400

X_train = train_df[FEATURES].astype(np.float32)
y_train = train_df["Label"].astype(np.float32)

X_eval = eval_df[FEATURES].astype(np.float32)

del train_df, df
gc.collect()

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

def top1(group):
    best = group["Label"].max()
    return float(
        group.sort_values(
            "Score", ascending=False
        ).iloc[0]["Label"] == best
    )


def hitk(group, k):
    best = group["Label"].max()
    top = group.sort_values(
        "Score", ascending=False
    ).head(k)

    return float(
        (top["Label"] == best).any()
    )


def ndcg(group):
    ranked = group.sort_values(
        "Score", ascending=False
    )

    rel = ranked["Label"].to_numpy(float)

    discounts = np.log2(
        np.arange(2, len(rel) + 2)
    )

    gains = (2 ** rel) - 1

    dcg = np.sum(
        gains / discounts
    )

    ideal = np.sort(rel)[::-1]
    ideal_gains = (2 ** ideal) - 1

    idcg = np.sum(
        ideal_gains / discounts
    )

    return 0.0 if idcg == 0 else dcg / idcg


def pairwise(group):
    labels = group["Label"].to_numpy()
    scores = group["Score"].to_numpy()

    correct = 0
    total = 0

    n = len(labels)

    for i in range(n):
        for j in range(i + 1, n):

            if labels[i] == labels[j]:
                continue

            total += 1

            if (
                (labels[i] - labels[j])
                *
                (scores[i] - scores[j])
                > 0
            ):
                correct += 1

    return np.nan if total == 0 else correct / total


def evaluate(scores):

    temp = eval_df[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label"
        ]
    ].copy()

    temp["Score"] = scores

    output = []

    for trial, trial_df in temp.groupby(
        "Trial", sort=True
    ):

        t1 = []
        h3 = []
        h5 = []
        nd = []
        pw = []

        for _, g in trial_df.groupby(
            "Resume_Index", sort=False
        ):

            t1.append(top1(g))
            h3.append(hitk(g, 3))
            h5.append(hitk(g, 5))
            nd.append(ndcg(g))
            pw.append(pairwise(g))

        output.append({
            "Trial": trial,

            "Top-1": np.mean(t1),
            "Hit@3": np.mean(h3),
            "Hit@5": np.mean(h5),
            "NDCG": np.mean(nd),
            "Pairwise": np.nanmean(pw)
        })

    return pd.DataFrame(output)


# ------------------------------------------------------------
# Run helper
# ------------------------------------------------------------

def run_model(name, model=None, use_existing=False):

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    start = time.time()

    if use_existing:

        model = CatBoostRegressor()
        model.load_model(CATBOOST_PATH)

        print("✅ Loaded existing clean V3 CatBoost.")

    else:

        model.fit(
            X_train,
            y_train
        )

        print(
            f"✅ Training finished in "
            f"{time.time() - start:.2f}s"
        )

    pred_start = time.time()

    scores = model.predict(
        X_eval
    )

    predict_time = time.time() - pred_start

    scores = np.asarray(
        scores,
        dtype=np.float32
    )

    print(
        f"Prediction time: {predict_time:.2f}s"
    )

    assert len(scores) == 25400
    assert np.isfinite(scores).all()

    metrics = evaluate(scores)

    metrics["Model"] = name
    metrics["Fit_Seconds"] = (
        0 if use_existing
        else time.time() - start
    )
    metrics["Predict_Seconds"] = predict_time

    path = os.path.join(
        OUT_DIR,
        f"v3_bakeoff_{name.lower()}_results.csv"
    )

    metrics.to_csv(
        path,
        index=False
    )

    print("\nRESULT:")
    print(
        metrics.round(6).to_string(
            index=False
        )
    )

    print(
        "\n✅ Saved:",
        path
    )

    del model
    del scores
    gc.collect()

    return metrics


results = []

# ============================================================
# XGBoost
# ============================================================

xgb = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=2,
    tree_method="hist",
    max_bin=64
)

results.append(
    run_model(
        "XGBoost",
        xgb
    )
)

del xgb
gc.collect()

# ============================================================
# HistGradientBoosting
# ============================================================

hist = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

results.append(
    run_model(
        "HistGradientBoosting",
        hist
    )
)

del hist
gc.collect()

# ============================================================
# CatBoost
# ============================================================

results.append(
    run_model(
        "CatBoost",
        use_existing=True
    )
)

# ============================================================
# Combined batch result
# ============================================================

batch_results = pd.concat(
    results,
    ignore_index=True
)

batch_path = os.path.join(
    OUT_DIR,
    "v3_bakeoff_batch1_results.csv"
)

batch_results.to_csv(
    batch_path,
    index=False
)

print("\n" + "=" * 100)
print("BATCH 1 COMPLETE 🔥")
print("=" * 100)

print(
    batch_results.round(6).to_string(
        index=False
    )
)

print("\nSaved:", batch_path)

In [ ]:
# ============================================================
# MODEL BATTLE — BATCH 2
# ExtraTrees + RandomForest + DecisionTree
# ============================================================

import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

DATA_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

EVAL_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

OUT_DIR = os.path.join(
    ROOT, "V3", "Outputs"
)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

print("=" * 100)
print("BATCH 2 — EXTRATREES + RANDOM FOREST + DECISION TREE")
print("=" * 100)

df = pd.read_csv(DATA_PATH)
eval_df = pd.read_csv(EVAL_PATH)

holdout_resumes = set(
    eval_df["Resume_Index"].unique()
)

train_df = df[
    ~df["Resume_Index"].isin(
        holdout_resumes
    )
]

X_train = train_df[
    FEATURES
].astype(np.float32)

y_train = train_df[
    "Label"
].astype(np.float32)

X_eval = eval_df[
    FEATURES
].astype(np.float32)

print(
    "Training:",
    X_train.shape
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

def evaluate(scores):

    d = eval_df[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label"
        ]
    ].copy()

    d["Score"] = scores

    rows = []

    for trial, tdf in d.groupby(
        "Trial",
        sort=True
    ):

        a = []
        b = []
        c = []
        n = []
        p = []

        for _, g in tdf.groupby(
            "Resume_Index",
            sort=False
        ):

            best = g["Label"].max()

            ranked = g.sort_values(
                "Score",
                ascending=False
            )

            a.append(
                float(
                    ranked.iloc[0]["Label"]
                    == best
                )
            )

            b.append(
                float(
                    (
                        ranked.head(3)["Label"]
                        == best
                    ).any()
                )
            )

            c.append(
                float(
                    (
                        ranked.head(5)["Label"]
                        == best
                    ).any()
                )
            )

            rel = ranked["Label"].to_numpy(float)

            discounts = np.log2(
                np.arange(2, len(rel) + 2)
            )

            gains = (2 ** rel) - 1

            dcg = np.sum(
                gains / discounts
            )

            ideal = np.sort(rel)[::-1]
            ig = (2 ** ideal) - 1

            idcg = np.sum(
                ig / discounts
            )

            n.append(
                0.0 if idcg == 0
                else dcg / idcg
            )

            labels = g["Label"].to_numpy()
            scores_g = g["Score"].to_numpy()

            correct = 0
            total = 0

            for i in range(len(g)):
                for j in range(i + 1, len(g)):

                    if labels[i] == labels[j]:
                        continue

                    total += 1

                    if (
                        (labels[i] - labels[j])
                        *
                        (scores_g[i] - scores_g[j])
                        > 0
                    ):
                        correct += 1

            p.append(
                np.nan if total == 0
                else correct / total
            )

        rows.append({
            "Trial": trial,
            "Top-1": np.mean(a),
            "Hit@3": np.mean(b),
            "Hit@5": np.mean(c),
            "NDCG": np.mean(n),
            "Pairwise": np.nanmean(p)
        })

    return pd.DataFrame(rows)


def run(name, model):

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    start = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_time = time.time() - start

    print(
        f"Training: {fit_time:.2f}s"
    )

    start = time.time()

    scores = model.predict(
        X_eval
    )

    predict_time = time.time() - start

    print(
        f"Prediction: {predict_time:.2f}s"
    )

    scores = np.asarray(
        scores,
        dtype=np.float32
    )

    result = evaluate(
        scores
    )

    result["Model"] = name
    result["Fit_Seconds"] = fit_time
    result["Predict_Seconds"] = predict_time

    path = os.path.join(
        OUT_DIR,
        f"v3_bakeoff_{name.lower()}_results.csv"
    )

    result.to_csv(
        path,
        index=False
    )

    print(
        result.round(6).to_string(
            index=False
        )
    )

    print(
        "✅ Saved:",
        path
    )

    del model
    del scores
    gc.collect()

    return result


results = []

# ------------------------------------------------------------
# ExtraTrees
# ------------------------------------------------------------

results.append(
    run(
        "ExtraTrees",
        ExtraTreesRegressor(
            n_estimators=150,
            random_state=42,
            n_jobs=2,
            max_features=1.0
        )
    )
)

# ------------------------------------------------------------
# RandomForest
# ------------------------------------------------------------

results.append(
    run(
        "RandomForest",
        RandomForestRegressor(
            n_estimators=150,
            random_state=42,
            n_jobs=2,
            max_features=1.0
        )
    )
)

# ------------------------------------------------------------
# Decision Tree
# ------------------------------------------------------------

results.append(
    run(
        "DecisionTree",
        DecisionTreeRegressor(
            random_state=42
        )
    )
)

batch = pd.concat(
    results,
    ignore_index=True
)

path = os.path.join(
    OUT_DIR,
    "v3_bakeoff_batch2_results.csv"
)

batch.to_csv(
    path,
    index=False
)

print("\n" + "=" * 100)
print("BATCH 2 COMPLETE 🔥")
print("=" * 100)

print(
    batch.round(6).to_string(
        index=False
    )
)

print("\nSaved:", path)

del X_train, y_train, X_eval
gc.collect()

In [ ]:
# ============================================================
# MODEL BATTLE — BATCH 3
# KNN + Linear Regression + Ridge
# ============================================================

import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

DATA_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

EVAL_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

OUT_DIR = os.path.join(
    ROOT, "V3", "Outputs"
)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

print("=" * 100)
print("BATCH 3 — KNN + LINEAR REGRESSION + RIDGE")
print("=" * 100)

df = pd.read_csv(DATA_PATH)
eval_df = pd.read_csv(EVAL_PATH)

holdout_resumes = set(
    eval_df["Resume_Index"].unique()
)

train_df = df[
    ~df["Resume_Index"].isin(
        holdout_resumes
    )
]

X_train = train_df[
    FEATURES
].astype(np.float32)

y_train = train_df[
    "Label"
].astype(np.float32)

X_eval = eval_df[
    FEATURES
].astype(np.float32
)

# ------------------------------------------------------------
# Ranking metrics
# ------------------------------------------------------------

def evaluate(scores):

    d = eval_df[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label"
        ]
    ].copy()

    d["Score"] = scores

    rows = []

    for trial, tdf in d.groupby(
        "Trial",
        sort=True
    ):

        top1 = []
        hit3 = []
        hit5 = []
        ndcg = []
        pairwise = []

        for _, g in tdf.groupby(
            "Resume_Index",
            sort=False
        ):

            ranked = g.sort_values(
                "Score",
                ascending=False
            )

            best = g["Label"].max()

            top1.append(
                float(
                    ranked.iloc[0]["Label"]
                    == best
                )
            )

            hit3.append(
                float(
                    (
                        ranked.head(3)["Label"]
                        == best
                    ).any()
                )
            )

            hit5.append(
                float(
                    (
                        ranked.head(5)["Label"]
                        == best
                    ).any()
                )
            )

            rel = ranked["Label"].to_numpy(float)

            discounts = np.log2(
                np.arange(2, len(rel) + 2)
            )

            gains = (2 ** rel) - 1

            dcg = np.sum(
                gains / discounts
            )

            ideal = np.sort(rel)[::-1]

            ideal_gains = (
                2 ** ideal
            ) - 1

            idcg = np.sum(
                ideal_gains / discounts
            )

            ndcg.append(
                0.0 if idcg == 0
                else dcg / idcg
            )

            labels = g["Label"].to_numpy()
            scores_g = g["Score"].to_numpy()

            correct = 0
            total = 0

            for i in range(len(g)):
                for j in range(i + 1, len(g)):

                    if labels[i] == labels[j]:
                        continue

                    total += 1

                    if (
                        (labels[i] - labels[j])
                        *
                        (scores_g[i] - scores_g[j])
                        > 0
                    ):
                        correct += 1

            pairwise.append(
                np.nan if total == 0
                else correct / total
            )

        rows.append({
            "Trial": trial,
            "Top-1": np.mean(top1),
            "Hit@3": np.mean(hit3),
            "Hit@5": np.mean(hit5),
            "NDCG": np.mean(ndcg),
            "Pairwise": np.nanmean(pairwise)
        })

    return pd.DataFrame(rows)


def run(name, model):

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    start = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_time = time.time() - start

    print(
        f"Training: {fit_time:.2f}s"
    )

    start = time.time()

    scores = model.predict(
        X_eval
    )

    predict_time = time.time() - start

    print(
        f"Prediction: {predict_time:.2f}s"
    )

    scores = np.asarray(
        scores,
        dtype=np.float32
    )

    assert np.isfinite(scores).all()

    result = evaluate(
        scores
    )

    result["Model"] = name
    result["Fit_Seconds"] = fit_time
    result["Predict_Seconds"] = predict_time

    path = os.path.join(
        OUT_DIR,
        f"v3_bakeoff_{name.lower()}_results.csv"
    )

    result.to_csv(
        path,
        index=False
    )

    print(
        result.round(6).to_string(
            index=False
        )
    )

    print(
        "✅ Saved:",
        path
    )

    del model
    del scores
    gc.collect()

    return result


results = []

# ============================================================
# KNN
#
# Scaling is essential.
# distance-weighted KNN, k=10
# ============================================================

knn = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "knn",
        KNeighborsRegressor(
            n_neighbors=10,
            weights="distance",
            algorithm="kd_tree",
            leaf_size=40,
            n_jobs=2
        )
    )
])

results.append(
    run(
        "KNN",
        knn
    )
)

# ============================================================
# Linear Regression
# ============================================================

linear = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "regressor",
        LinearRegression()
    )
])

results.append(
    run(
        "LinearRegression",
        linear
    )
)

# ============================================================
# Ridge
# ============================================================

ridge = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "regressor",
        Ridge(
            alpha=1.0
        )
    )
])

results.append(
    run(
        "Ridge",
        ridge
    )
)

batch = pd.concat(
    results,
    ignore_index=True
)

path = os.path.join(
    OUT_DIR,
    "v3_bakeoff_batch3_results.csv"
)

batch.to_csv(
    path,
    index=False
)

print("\n" + "=" * 100)
print("BATCH 3 COMPLETE 🔥")
print("=" * 100)

print(
    batch.round(6).to_string(
        index=False
    )
)

print("\nSaved:", path)

del X_train, y_train, X_eval
gc.collect()

In [ ]:
# ============================================================
# MODEL BATTLE — BATCH 4
# Logistic Regression + Gaussian Naive Bayes
# ============================================================

import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

DATA_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

EVAL_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

OUT_DIR = os.path.join(
    ROOT, "V3", "Outputs"
)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

print("=" * 100)
print("BATCH 4 — LOGISTIC REGRESSION + GAUSSIAN NB")
print("=" * 100)

df = pd.read_csv(DATA_PATH)
eval_df = pd.read_csv(EVAL_PATH)

holdout_resumes = set(
    eval_df["Resume_Index"].unique()
)

train_df = df[
    ~df["Resume_Index"].isin(
        holdout_resumes
    )
]

X_train = train_df[
    FEATURES
].astype(np.float32)

y_train = train_df[
    "Label"
].astype(int)

X_eval = eval_df[
    FEATURES
].astype(np.float32)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

def evaluate(scores):

    d = eval_df[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label"
        ]
    ].copy()

    d["Score"] = scores

    rows = []

    for trial, tdf in d.groupby(
        "Trial",
        sort=True
    ):

        a = []
        b = []
        c = []
        n = []
        p = []

        for _, g in tdf.groupby(
            "Resume_Index",
            sort=False
        ):

            ranked = g.sort_values(
                "Score",
                ascending=False
            )

            best = g["Label"].max()

            a.append(
                float(
                    ranked.iloc[0]["Label"]
                    == best
                )
            )

            b.append(
                float(
                    (
                        ranked.head(3)["Label"]
                        == best
                    ).any()
                )
            )

            c.append(
                float(
                    (
                        ranked.head(5)["Label"]
                        == best
                    ).any()
                )
            )

            rel = ranked["Label"].to_numpy(float)

            discounts = np.log2(
                np.arange(2, len(rel) + 2)
            )

            gains = (2 ** rel) - 1

            dcg = np.sum(
                gains / discounts
            )

            ideal = np.sort(rel)[::-1]

            ideal_gains = (
                2 ** ideal
            ) - 1

            idcg = np.sum(
                ideal_gains / discounts
            )

            n.append(
                0.0 if idcg == 0
                else dcg / idcg
            )

            labels = g["Label"].to_numpy()
            scores_g = g["Score"].to_numpy()

            correct = 0
            total = 0

            for i in range(len(g)):
                for j in range(i + 1, len(g)):

                    if labels[i] == labels[j]:
                        continue

                    total += 1

                    if (
                        (labels[i] - labels[j])
                        *
                        (scores_g[i] - scores_g[j])
                        > 0
                    ):
                        correct += 1

            p.append(
                np.nan if total == 0
                else correct / total
            )

        rows.append({
            "Trial": trial,
            "Top-1": np.mean(a),
            "Hit@3": np.mean(b),
            "Hit@5": np.mean(c),
            "NDCG": np.mean(n),
            "Pairwise": np.nanmean(p)
        })

    return pd.DataFrame(rows)


def run_classifier(
    name,
    model
):

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    start = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_time = time.time() - start

    print(
        f"Training: {fit_time:.2f}s"
    )

    start = time.time()

    probabilities = model.predict_proba(
        X_eval
    )

    predict_time = time.time() - start

    print(
        f"Prediction: {predict_time:.2f}s"
    )

    classes = model.classes_

    scores = np.zeros(
        len(X_eval),
        dtype=np.float32
    )

    # Expected relevance:
    #
    # E[Y] = Σ class * P(class)

    for idx, class_value in enumerate(
        classes
    ):

        scores += (
            float(class_value)
            *
            probabilities[:, idx]
        )

    result = evaluate(
        scores
    )

    result["Model"] = name
    result["Fit_Seconds"] = fit_time
    result["Predict_Seconds"] = predict_time

    path = os.path.join(
        OUT_DIR,
        f"v3_bakeoff_{name.lower()}_results.csv"
    )

    result.to_csv(
        path,
        index=False
    )

    print(
        result.round(6).to_string(
            index=False
        )
    )

    print(
        "✅ Saved:",
        path
    )

    del model
    del probabilities
    del scores
    gc.collect()

    return result


results = []

# ============================================================
# Logistic Regression
# ============================================================

logistic = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42,
            n_jobs=2
        )
    )
])

results.append(
    run_classifier(
        "LogisticRegression",
        logistic
    )
)

# ============================================================
# Gaussian Naive Bayes
# ============================================================

gnb = GaussianNB()

results.append(
    run_classifier(
        "GaussianNB",
        gnb
    )
)

batch = pd.concat(
    results,
    ignore_index=True
)

path = os.path.join(
    OUT_DIR,
    "v3_bakeoff_batch4_results.csv"
)

batch.to_csv(
    path,
    index=False
)

print("\n" + "=" * 100)
print("BATCH 4 COMPLETE 🔥")
print("=" * 100)

print(
    batch.round(6).to_string(
        index=False
    )
)

print("\nSaved:", path)

del X_train, y_train, X_eval
gc.collect()

In [ ]:
# ============================================================
# V3 FINAL MODEL TUNING — EXTRATREES
#
# Inner validation is used ONLY for hyperparameter selection.
# The existing 25,400 evaluation pairs remain untouched.
# ============================================================

import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

DATA_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

EVAL_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

OUT_DIR = os.path.join(
    ROOT,
    "V3",
    "Outputs"
)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

print("=" * 100)
print("EXTRATREES HYPERPARAMETER TUNING")
print("=" * 100)

# ============================================================
# 1. LOAD
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    usecols=FEATURES + [
        "Label",
        "Resume_Index"
    ]
)

eval_df = pd.read_csv(
    EVAL_PATH
)

# The 308 resumes used by our final evaluation must remain
# completely outside the tuning/training pool.
benchmark_holdout = set(
    eval_df["Resume_Index"].unique()
)

assert len(benchmark_holdout) == 308

training_pool = df[
    ~df["Resume_Index"].isin(
        benchmark_holdout
    )
].copy()

print(
    "Training pool rows:",
    len(training_pool)
)

print(
    "Training pool resumes:",
    training_pool["Resume_Index"].nunique()
)

assert len(training_pool) == 454451
assert training_pool["Resume_Index"].nunique() == 1232

# ============================================================
# 2. INNER RESUME-LEVEL SPLIT
# ============================================================

training_resumes = np.sort(
    training_pool["Resume_Index"].unique()
)

inner_train_resumes, inner_val_resumes = train_test_split(
    training_resumes,
    test_size=0.20,
    random_state=2026
)

inner_train_resumes = np.sort(
    inner_train_resumes
)

inner_val_resumes = np.sort(
    inner_val_resumes
)

print("\nInner split:")
print(
    "Inner train resumes:",
    len(inner_train_resumes)
)

print(
    "Inner validation resumes:",
    len(inner_val_resumes)
)

assert len(inner_train_resumes) == 985
assert len(inner_val_resumes) == 247

inner_train = training_pool[
    training_pool["Resume_Index"].isin(
        inner_train_resumes
    )
].copy()

inner_val = training_pool[
    training_pool["Resume_Index"].isin(
        inner_val_resumes
    )
].copy()

print(
    "Inner train rows:",
    len(inner_train)
)

print(
    "Inner validation rows:",
    len(inner_val)
)

# ============================================================
# 3. DATA MATRICES
# ============================================================

X_inner_train = inner_train[
    FEATURES
].to_numpy(
    dtype=np.float32
)

y_inner_train = inner_train[
    "Label"
].to_numpy(
    dtype=np.float32
)

X_inner_val = inner_val[
    FEATURES
].to_numpy(
    dtype=np.float32
)

y_inner_val = inner_val[
    "Label"
].to_numpy(
    dtype=np.float32
)

# ============================================================
# 4. INNER VALIDATION NDCG
# ============================================================

def validation_ndcg(
    val_df,
    scores
):

    temp = val_df[
        [
            "Resume_Index",
            "Label"
        ]
    ].copy()

    temp["Score"] = scores

    values = []

    for _, g in temp.groupby(
        "Resume_Index",
        sort=False
    ):

        ranked = g.sort_values(
            "Score",
            ascending=False
        )

        rel = ranked[
            "Label"
        ].to_numpy(
            dtype=float
        )

        discounts = np.log2(
            np.arange(
                2,
                len(rel) + 2
            )
        )

        gains = (
            2 ** rel
        ) - 1

        dcg = np.sum(
            gains / discounts
        )

        ideal = np.sort(
            rel
        )[::-1]

        ideal_gains = (
            2 ** ideal
        ) - 1

        idcg = np.sum(
            ideal_gains / discounts
        )

        values.append(
            0.0
            if idcg == 0
            else dcg / idcg
        )

    return float(
        np.mean(values)
    )


# ============================================================
# 5. SMALL PARAMETER SEARCH
#
# Deliberately small to keep runtime reasonable.
# ============================================================

configs = [
    {
        "name": "ET_150_None_1",
        "n_estimators": 150,
        "max_depth": None,
        "min_samples_leaf": 1
    },

    {
        "name": "ET_200_None_1",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1
    },

    {
        "name": "ET_200_20_1",
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_leaf": 1
    },

    {
        "name": "ET_200_None_2",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 2
    }
]

tuning_results = []

# ============================================================
# 6. TUNE
# ============================================================

for cfg in configs:

    print("\n" + "-" * 100)
    print(
        "Testing:",
        cfg["name"]
    )
    print("-" * 100)

    model = ExtraTreesRegressor(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        min_samples_leaf=cfg["min_samples_leaf"],
        random_state=42,
        n_jobs=2,
        max_features=1.0
    )

    start = time.time()

    model.fit(
        X_inner_train,
        y_inner_train
    )

    fit_time = time.time() - start

    scores = model.predict(
        X_inner_val
    )

    score = validation_ndcg(
        inner_val,
        scores
    )

    print(
        f"NDCG = {score:.6f}"
    )

    print(
        f"Fit time = {fit_time:.2f}s"
    )

    tuning_results.append({
        **cfg,
        "Validation_NDCG": score,
        "Fit_Seconds": fit_time
    })

    del model
    del scores
    gc.collect()

# ============================================================
# 7. TUNING TABLE
# ============================================================

tuning_df = pd.DataFrame(
    tuning_results
).sort_values(
    "Validation_NDCG",
    ascending=False
).reset_index(
    drop=True
)

print("\n" + "=" * 100)
print("EXTRATREES TUNING RESULTS")
print("=" * 100)

print(
    tuning_df.round(6).to_string(
        index=False
    )
)

best = tuning_df.iloc[0]

print("\n🏆 BEST CONFIG:")
print(
    best.to_dict()
)

# ============================================================
# 8. RETRAIN BEST CONFIG ON ALL 1,232 TRAINING RESUMES
# ============================================================

X_full_train = training_pool[
    FEATURES
].to_numpy(
    dtype=np.float32
)

y_full_train = training_pool[
    "Label"
].to_numpy(
    dtype=np.float32
)

print("\n" + "=" * 100)
print("RETRAINING BEST EXTRATREES ON ALL 1232 TRAINING RESUMES")
print("=" * 100)

final_model = ExtraTreesRegressor(
    n_estimators=int(
        best["n_estimators"]
    ),
    max_depth=(
        None
        if pd.isna(best["max_depth"])
        else int(best["max_depth"])
    ),
    min_samples_leaf=int(
        best["min_samples_leaf"]
    ),
    random_state=42,
    n_jobs=2,
    max_features=1.0
)

start = time.time()

final_model.fit(
    X_full_train,
    y_full_train
)

final_fit_time = time.time() - start

print(
    f"✅ Final ExtraTrees trained in "
    f"{final_fit_time:.2f}s"
)

# ============================================================
# 9. FINAL EVALUATION ON EXACT 25,400 PAIRS
# ============================================================

X_eval = eval_df[
    FEATURES
].to_numpy(
    dtype=np.float32
)

start = time.time()

pred = final_model.predict(
    X_eval
)

predict_time = time.time() - start

print(
    f"Prediction time: {predict_time:.2f}s"
)

assert len(pred) == 25400
assert np.isfinite(pred).all()

# ============================================================
# 10. FINAL RANKING METRICS
# ============================================================

def calculate_metrics(
    data,
    scores
):

    temp = data[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label"
        ]
    ].copy()

    temp["Score"] = scores

    rows = []

    for trial, trial_df in temp.groupby(
        "Trial",
        sort=True
    ):

        top1 = []
        hit3 = []
        hit5 = []
        ndcg = []
        pairwise = []

        for _, g in trial_df.groupby(
            "Resume_Index",
            sort=False
        ):

            ranked = g.sort_values(
                "Score",
                ascending=False
            )

            best_label = g["Label"].max()

            top1.append(
                float(
                    ranked.iloc[0]["Label"]
                    == best_label
                )
            )

            hit3.append(
                float(
                    (
                        ranked.head(3)["Label"]
                        == best_label
                    ).any()
                )
            )

            hit5.append(
                float(
                    (
                        ranked.head(5)["Label"]
                        == best_label
                    ).any()
                )
            )

            rel = ranked[
                "Label"
            ].to_numpy(
                dtype=float
            )

            discounts = np.log2(
                np.arange(
                    2,
                    len(rel) + 2
                )
            )

            gains = (
                2 ** rel
            ) - 1

            dcg = np.sum(
                gains / discounts
            )

            ideal = np.sort(
                rel
            )[::-1]

            ideal_gains = (
                2 ** ideal
            ) - 1

            idcg = np.sum(
                ideal_gains / discounts
            )

            ndcg.append(
                0.0
                if idcg == 0
                else dcg / idcg
            )

            labels = g[
                "Label"
            ].to_numpy()

            scores_g = g[
                "Score"
            ].to_numpy()

            correct = 0
            total = 0

            for i in range(len(g)):

                for j in range(
                    i + 1,
                    len(g)
                ):

                    if labels[i] == labels[j]:
                        continue

                    total += 1

                    if (
                        (labels[i] - labels[j])
                        *
                        (scores_g[i] - scores_g[j])
                        > 0
                    ):
                        correct += 1

            pairwise.append(
                np.nan
                if total == 0
                else correct / total
            )

        rows.append({
            "Trial": trial,
            "Top-1": np.mean(top1),
            "Hit@3": np.mean(hit3),
            "Hit@5": np.mean(hit5),
            "NDCG": np.mean(ndcg),
            "Pairwise": np.nanmean(pairwise)
        })

    return pd.DataFrame(rows)


final_results = calculate_metrics(
    eval_df,
    pred
)

print("\n" + "=" * 100)
print("FINAL EXTRATREES RESULTS")
print("=" * 100)

print(
    final_results.round(6).to_string(
        index=False
    )
)

# ============================================================
# 11. MEAN
# ============================================================

print("\nMEAN RESULTS:")

print(
    final_results[
        [
            "Top-1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise"
        ]
    ].mean().round(6)
)

# ============================================================
# 12. SAVE
# ============================================================

tuning_path = os.path.join(
    OUT_DIR,
    "v3_extratrees_tuning_results.csv"
)

final_results_path = os.path.join(
    OUT_DIR,
    "v3_extratrees_tuned_10trial_results.csv"
)

model_path = os.path.join(
    OUT_DIR,
    "v3_extratrees_tuned_final.joblib"
)

pred_path = os.path.join(
    OUT_DIR,
    "v3_extratrees_tuned_25400_predictions.csv"
)

tuning_df.to_csv(
    tuning_path,
    index=False
)

final_results.to_csv(
    final_results_path,
    index=False
)

# Use joblib for sklearn model
import joblib

joblib.dump(
    final_model,
    model_path
)

prediction_output = eval_df[
    [
        "Trial",
        "Seed",
        "Resume_Index",
        "Job_Index",
        "Label"
    ]
].copy()

prediction_output[
    "ExtraTrees_Tuned_Score"
] = pred

prediction_output.to_csv(
    pred_path,
    index=False
)

print("\n✅ SAVED:")
print(tuning_path)
print(final_results_path)
print(model_path)
print(pred_path)

del final_model
del pred
del X_full_train
del y_full_train
del X_eval
del X_inner_train
del y_inner_train
del X_inner_val
del y_inner_val
del inner_train
del inner_val
del training_pool
del eval_df
gc.collect()

print("\n" + "=" * 100)
print("EXTRATREES DONE 🔥")
print("=" * 100)

In [ ]:
# ============================================================
# V3 FINAL MODEL TUNING — RANDOM FOREST
# ============================================================

import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

DATA_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

EVAL_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

OUT_DIR = os.path.join(
    ROOT,
    "V3",
    "Outputs"
)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

print("=" * 100)
print("RANDOM FOREST HYPERPARAMETER TUNING")
print("=" * 100)

# ============================================================
# 1. LOAD
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    usecols=FEATURES + [
        "Label",
        "Resume_Index"
    ]
)

eval_df = pd.read_csv(
    EVAL_PATH
)

benchmark_holdout = set(
    eval_df["Resume_Index"].unique()
)

assert len(benchmark_holdout) == 308

training_pool = df[
    ~df["Resume_Index"].isin(
        benchmark_holdout
    )
].copy()

assert len(training_pool) == 454451
assert training_pool["Resume_Index"].nunique() == 1232

# ============================================================
# 2. INNER RESUME-LEVEL VALIDATION
# ============================================================

training_resumes = np.sort(
    training_pool["Resume_Index"].unique()
)

inner_train_resumes, inner_val_resumes = train_test_split(
    training_resumes,
    test_size=0.20,
    random_state=2026
)

inner_train = training_pool[
    training_pool["Resume_Index"].isin(
        inner_train_resumes
    )
].copy()

inner_val = training_pool[
    training_pool["Resume_Index"].isin(
        inner_val_resumes
    )
].copy()

print(
    "Inner train resumes:",
    len(inner_train_resumes)
)

print(
    "Inner validation resumes:",
    len(inner_val_resumes)
)

print(
    "Inner train rows:",
    len(inner_train)
)

print(
    "Inner validation rows:",
    len(inner_val)
)

# ============================================================
# 3. MATRICES
# ============================================================

X_inner_train = inner_train[
    FEATURES
].to_numpy(
    dtype=np.float32
)

y_inner_train = inner_train[
    "Label"
].to_numpy(
    dtype=np.float32
)

X_inner_val = inner_val[
    FEATURES
].to_numpy(
    dtype=np.float32
)

# ============================================================
# 4. INNER VALIDATION NDCG
# ============================================================

def validation_ndcg(
    val_df,
    scores
):

    temp = val_df[
        [
            "Resume_Index",
            "Label"
        ]
    ].copy()

    temp["Score"] = scores

    values = []

    for _, g in temp.groupby(
        "Resume_Index",
        sort=False
    ):

        ranked = g.sort_values(
            "Score",
            ascending=False
        )

        rel = ranked[
            "Label"
        ].to_numpy(
            dtype=float
        )

        discounts = np.log2(
            np.arange(
                2,
                len(rel) + 2
            )
        )

        dcg = np.sum(
            (
                (2 ** rel) - 1
            ) / discounts
        )

        ideal = np.sort(
            rel
        )[::-1]

        idcg = np.sum(
            (
                (2 ** ideal) - 1
            ) / discounts
        )

        values.append(
            0.0
            if idcg == 0
            else dcg / idcg
        )

    return float(
        np.mean(values)
    )

# ============================================================
# 5. SMALL SEARCH
# ============================================================

configs = [

    {
        "name": "RF_150_None_1",
        "n_estimators": 150,
        "max_depth": None,
        "min_samples_leaf": 1
    },

    {
        "name": "RF_200_None_1",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1
    },

    {
        "name": "RF_200_20_1",
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_leaf": 1
    },

    {
        "name": "RF_200_None_2",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 2
    }
]

tuning_results = []

# ============================================================
# 6. SEARCH
# ============================================================

for cfg in configs:

    print("\n" + "-" * 100)
    print(
        "Testing:",
        cfg["name"]
    )
    print("-" * 100)

    model = RandomForestRegressor(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        min_samples_leaf=cfg["min_samples_leaf"],
        random_state=42,
        n_jobs=2,
        max_features=1.0
    )

    start = time.time()

    model.fit(
        X_inner_train,
        y_inner_train
    )

    fit_time = time.time() - start

    scores = model.predict(
        X_inner_val
    )

    score = validation_ndcg(
        inner_val,
        scores
    )

    print(
        f"NDCG = {score:.6f}"
    )

    print(
        f"Fit time = {fit_time:.2f}s"
    )

    tuning_results.append({
        **cfg,
        "Validation_NDCG": score,
        "Fit_Seconds": fit_time
    })

    del model
    del scores
    gc.collect()

# ============================================================
# 7. TUNING RESULTS
# ============================================================

tuning_df = pd.DataFrame(
    tuning_results
).sort_values(
    "Validation_NDCG",
    ascending=False
).reset_index(
    drop=True
)

print("\n" + "=" * 100)
print("RANDOM FOREST TUNING RESULTS")
print("=" * 100)

print(
    tuning_df.round(6).to_string(
        index=False
    )
)

best = tuning_df.iloc[0]

print("\n🏆 BEST CONFIG:")
print(
    best.to_dict()
)

# ============================================================
# 8. RETRAIN BEST CONFIG ON ALL 1,232 TRAINING RESUMES
# ============================================================

X_full_train = training_pool[
    FEATURES
].to_numpy(
    dtype=np.float32
)

y_full_train = training_pool[
    "Label"
].to_numpy(
    dtype=np.float32
)

print("\n" + "=" * 100)
print("RETRAINING BEST RANDOM FOREST ON ALL 1232 TRAINING RESUMES")
print("=" * 100)

final_model = RandomForestRegressor(
    n_estimators=int(
        best["n_estimators"]
    ),
    max_depth=(
        None
        if pd.isna(best["max_depth"])
        else int(best["max_depth"])
    ),
    min_samples_leaf=int(
        best["min_samples_leaf"]
    ),
    random_state=42,
    n_jobs=2,
    max_features=1.0
)

start = time.time()

final_model.fit(
    X_full_train,
    y_full_train
)

final_fit_time = time.time() - start

print(
    f"✅ Final RandomForest trained in "
    f"{final_fit_time:.2f}s"
)

# ============================================================
# 9. FINAL EVALUATION
# ============================================================

X_eval = eval_df[
    FEATURES
].to_numpy(
    dtype=np.float32
)

start = time.time()

pred = final_model.predict(
    X_eval
)

predict_time = time.time() - start

print(
    f"Prediction time: {predict_time:.2f}s"
)

assert len(pred) == 25400
assert np.isfinite(pred).all()

# ============================================================
# 10. METRICS
# ============================================================

def calculate_metrics(
    data,
    scores
):

    temp = data[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label"
        ]
    ].copy()

    temp["Score"] = scores

    rows = []

    for trial, trial_df in temp.groupby(
        "Trial",
        sort=True
    ):

        top1 = []
        hit3 = []
        hit5 = []
        ndcg = []
        pairwise = []

        for _, g in trial_df.groupby(
            "Resume_Index",
            sort=False
        ):

            ranked = g.sort_values(
                "Score",
                ascending=False
            )

            best_label = g["Label"].max()

            top1.append(
                float(
                    ranked.iloc[0]["Label"]
                    == best_label
                )
            )

            hit3.append(
                float(
                    (
                        ranked.head(3)["Label"]
                        == best_label
                    ).any()
                )
            )

            hit5.append(
                float(
                    (
                        ranked.head(5)["Label"]
                        == best_label
                    ).any()
                )
            )

            rel = ranked[
                "Label"
            ].to_numpy(
                dtype=float
            )

            discounts = np.log2(
                np.arange(
                    2,
                    len(rel) + 2
                )
            )

            dcg = np.sum(
                (
                    (2 ** rel) - 1
                ) / discounts
            )

            ideal = np.sort(
                rel
            )[::-1]

            idcg = np.sum(
                (
                    (2 ** ideal) - 1
                ) / discounts
            )

            ndcg.append(
                0.0
                if idcg == 0
                else dcg / idcg
            )

            labels = g["Label"].to_numpy()
            scores_g = g["Score"].to_numpy()

            correct = 0
            total = 0

            for i in range(len(g)):

                for j in range(
                    i + 1,
                    len(g)
                ):

                    if labels[i] == labels[j]:
                        continue

                    total += 1

                    if (
                        (labels[i] - labels[j])
                        *
                        (scores_g[i] - scores_g[j])
                        > 0
                    ):
                        correct += 1

            pairwise.append(
                np.nan
                if total == 0
                else correct / total
            )

        rows.append({
            "Trial": trial,
            "Top-1": np.mean(top1),
            "Hit@3": np.mean(hit3),
            "Hit@5": np.mean(hit5),
            "NDCG": np.mean(ndcg),
            "Pairwise": np.nanmean(pairwise)
        })

    return pd.DataFrame(rows)


final_results = calculate_metrics(
    eval_df,
    pred
)

print("\n" + "=" * 100)
print("FINAL RANDOM FOREST RESULTS")
print("=" * 100)

print(
    final_results.round(6).to_string(
        index=False
    )
)

print("\nMEAN RESULTS:")

print(
    final_results[
        [
            "Top-1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise"
        ]
    ].mean().round(6)
)

# ============================================================
# 11. SAVE
# ============================================================

import joblib

tuning_path = os.path.join(
    OUT_DIR,
    "v3_randomforest_tuning_results.csv"
)

final_results_path = os.path.join(
    OUT_DIR,
    "v3_randomforest_tuned_10trial_results.csv"
)

model_path = os.path.join(
    OUT_DIR,
    "v3_randomforest_tuned_final.joblib"
)

pred_path = os.path.join(
    OUT_DIR,
    "v3_randomforest_tuned_25400_predictions.csv"
)

tuning_df.to_csv(
    tuning_path,
    index=False
)

final_results.to_csv(
    final_results_path,
    index=False
)

joblib.dump(
    final_model,
    model_path
)

prediction_output = eval_df[
    [
        "Trial",
        "Seed",
        "Resume_Index",
        "Job_Index",
        "Label"
    ]
].copy()

prediction_output[
    "RandomForest_Tuned_Score"
] = pred

prediction_output.to_csv(
    pred_path,
    index=False
)

print("\n✅ SAVED:")
print(tuning_path)
print(final_results_path)
print(model_path)
print(pred_path)

del final_model
del pred
del X_full_train
del y_full_train
del X_eval
del X_inner_train
del y_inner_train
del X_inner_val
del inner_train
del inner_val
del training_pool
del eval_df
gc.collect()

print("\n" + "=" * 100)
print("RANDOM FOREST DONE 🔥")
print("=" * 100)

In [ ]:
# ============================================================
# V3 ERROR ANALYSIS — LOAD SAVED ARTIFACTS
# ============================================================

import os
import numpy as np
import pandas as pd

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

V3_DATA = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_supervised_dataset_568210_rich6_semantic.csv"
)

PRED_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_extratrees_tuned_25400_predictions.csv"
)

SAMPLES_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

RESUME_TEXT_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_resume_texts.pkl"
)

JOB_TEXT_PATH = os.path.join(
    ROOT, "V3", "Outputs",
    "v3_job_texts.pkl"
)

OUT_DIR = os.path.join(
    ROOT, "V3", "Outputs"
)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

print("=" * 100)
print("V3 ERROR ANALYSIS — RECOVERY")
print("=" * 100)

# ------------------------------------------------------------
# Load predictions
# ------------------------------------------------------------

pred_df = pd.read_csv(PRED_PATH)

print("\nPredictions:", pred_df.shape)
print(pred_df.columns.tolist())

assert len(pred_df) == 25400

# ------------------------------------------------------------
# Load V3 feature data
# ------------------------------------------------------------

v3_df = pd.read_csv(
    V3_DATA,
    usecols=[
        "Resume_Index",
        "Job_Index",
        "Cosine",
        "Skill_Score",
        "Normalized_Cosine",
        "Required_Skill_Count",
        "Matched_Skill_Count",
        "Role_Similarity",
        "Semantic_Similarity",
        "Label"
    ]
)

print("V3 feature data:", v3_df.shape)

# ------------------------------------------------------------
# Merge prediction + feature information
# ------------------------------------------------------------

analysis_df = pred_df.merge(
    v3_df,
    on=[
        "Resume_Index",
        "Job_Index",
        "Label"
    ],
    how="left"
)

assert len(analysis_df) == 25400

for col in FEATURES:
    assert analysis_df[col].isna().sum() == 0

print("\n✅ Predictions successfully matched to features.")

# ------------------------------------------------------------
# Missing skills
# ------------------------------------------------------------

analysis_df["Missing_Skill_Count"] = (
    analysis_df["Required_Skill_Count"]
    - analysis_df["Matched_Skill_Count"]
)

# ------------------------------------------------------------
# Rank jobs within each Trial + Resume
# ------------------------------------------------------------

analysis_df["Predicted_Rank"] = (
    analysis_df
    .groupby(
        ["Trial", "Resume_Index"]
    )["ExtraTrees_Tuned_Score"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)

analysis_df["Best_Label"] = (
    analysis_df
    .groupby(
        ["Trial", "Resume_Index"]
    )["Label"]
    .transform("max")
)

analysis_df["Is_Best_Label"] = (
    analysis_df["Label"]
    == analysis_df["Best_Label"]
)

# ------------------------------------------------------------
# Determine whether predicted job is Top-1
# ------------------------------------------------------------

analysis_df["Is_Predicted_Top1"] = (
    analysis_df["Predicted_Rank"] == 1
)

top1_df = analysis_df[
    analysis_df["Is_Predicted_Top1"]
].copy()

top1_df["Top1_Is_Correct"] = (
    top1_df["Label"]
    == top1_df["Best_Label"]
)

# ------------------------------------------------------------
# Rank of the best-labelled job
#
# If there are multiple Label=2 jobs, use the highest-ranked
# Label=2 job.
# ------------------------------------------------------------

best_label_rank = (
    analysis_df[
        analysis_df["Is_Best_Label"]
    ]
    .groupby(
        ["Trial", "Resume_Index"]
    )["Predicted_Rank"]
    .min()
    .reset_index(
        name="Best_Label_Rank"
    )
)

top1_df = top1_df.merge(
    best_label_rank,
    on=[
        "Trial",
        "Resume_Index"
    ],
    how="left"
)

# ------------------------------------------------------------
# Score of highest-ranked best-label job
# ------------------------------------------------------------

best_label_score = (
    analysis_df[
        analysis_df["Is_Best_Label"]
    ]
    .sort_values(
        [
            "Trial",
            "Resume_Index",
            "ExtraTrees_Tuned_Score"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .drop_duplicates(
        [
            "Trial",
            "Resume_Index"
        ]
    )[
        [
            "Trial",
            "Resume_Index",
            "ExtraTrees_Tuned_Score"
        ]
    ]
    .rename(
        columns={
            "ExtraTrees_Tuned_Score":
                "Best_Label_Score"
        }
    )
)

top1_df = top1_df.merge(
    best_label_score,
    on=[
        "Trial",
        "Resume_Index"
    ],
    how="left"
)

top1_df["Top1_Score_Margin"] = (
    top1_df["ExtraTrees_Tuned_Score"]
    - top1_df["Best_Label_Score"]
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TOP-1 ERROR SUMMARY")
print("=" * 100)

total_cases = len(top1_df)
correct_cases = top1_df["Top1_Is_Correct"].sum()
wrong_cases = total_cases - correct_cases

print("Resume-trial ranking cases:", total_cases)
print("Correct Top-1:", correct_cases)
print("Wrong Top-1:", wrong_cases)
print(
    "Top-1 accuracy:",
    round(
        correct_cases / total_cases,
        6
    )
)

print(
    "\nBest-label rank distribution:"
)

print(
    top1_df[
        "Best_Label_Rank"
    ]
    .value_counts()
    .sort_index()
    .head(10)
)

# ------------------------------------------------------------
# Wrong Top-1 cases
# ------------------------------------------------------------

wrong_top1 = top1_df[
    ~top1_df["Top1_Is_Correct"]
].copy()

wrong_top1 = wrong_top1.sort_values(
    [
        "Best_Label_Rank",
        "Top1_Score_Margin"
    ],
    ascending=[
        False,
        False
    ]
)

print(
    "\nWorst Top-1 failures:"
)

print(
    wrong_top1[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label",
            "Best_Label",
            "Predicted_Rank",
            "Best_Label_Rank",
            "ExtraTrees_Tuned_Score",
            "Best_Label_Score",
            "Top1_Score_Margin",
            "Cosine",
            "Skill_Score",
            "Normalized_Cosine",
            "Required_Skill_Count",
            "Matched_Skill_Count",
            "Missing_Skill_Count",
            "Role_Similarity",
            "Semantic_Similarity"
        ]
    ]
    .head(30)
    .round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Feature comparison:
# Top-1 selected job vs best-label job
# ------------------------------------------------------------

top1_records = []

for (trial, resume), g in analysis_df.groupby(
    ["Trial", "Resume_Index"],
    sort=False
):

    ranked = g.sort_values(
        "ExtraTrees_Tuned_Score",
        ascending=False
    )

    chosen = ranked.iloc[0]

    best_candidates = g[
        g["Label"] == g["Label"].max()
    ].sort_values(
        "ExtraTrees_Tuned_Score",
        ascending=False
    )

    best = best_candidates.iloc[0]

    record = {
        "Trial": trial,
        "Resume_Index": resume,
        "Top1_Correct": (
            chosen["Label"] == g["Label"].max()
        ),
        "Top1_Label": chosen["Label"],
        "Best_Label": g["Label"].max(),
        "Top1_Job": chosen["Job_Index"],
        "Best_Job": best["Job_Index"],
        "Best_Label_Rank": int(
            best["Predicted_Rank"]
        ),
    }

    for feature in FEATURES:
        record[
            f"{feature}_Top1"
        ] = chosen[feature]

        record[
            f"{feature}_Best"
        ] = best[feature]

        record[
            f"{feature}_Gap"
        ] = (
            best[feature]
            - chosen[feature]
        )

    top1_records.append(record)

resume_error_df = pd.DataFrame(
    top1_records
)

# ------------------------------------------------------------
# Feature averages for wrong Top-1 cases
# ------------------------------------------------------------

wrong_resume = resume_error_df[
    ~resume_error_df["Top1_Correct"]
]

print("\n" + "=" * 100)
print("FEATURE GAPS — WRONG TOP-1 CASES")
print("=" * 100)

gap_cols = [
    f"{f}_Gap"
    for f in FEATURES
]

gap_summary = (
    wrong_resume[gap_cols]
    .mean()
    .sort_values()
)

print(
    gap_summary.round(5).to_string()
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

analysis_path = os.path.join(
    OUT_DIR,
    "v3_error_analysis_pair_level_25400.csv"
)

wrong_path = os.path.join(
    OUT_DIR,
    "v3_error_analysis_wrong_top1.csv"
)

resume_path = os.path.join(
    OUT_DIR,
    "v3_error_analysis_resume_level.csv"
)

analysis_df.to_csv(
    analysis_path,
    index=False
)

wrong_top1.to_csv(
    wrong_path,
    index=False
)

resume_error_df.to_csv(
    resume_path,
    index=False
)

print("\n✅ SAVED:")
print(analysis_path)
print(wrong_path)
print(resume_path)

print("\n" + "=" * 100)
print("RECOVERY + FIRST ERROR ANALYSIS COMPLETE 🔥")
print("=" * 100)

In [ ]:
# ============================================================
# V3 ERROR ANALYSIS — DEEP DIVE
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd

ROOT = "/content/drive/MyDrive/Resume_Model_Project"

WRONG_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_error_analysis_wrong_top1.csv"
)

ANALYSIS_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_error_analysis_pair_level_25400.csv"
)

RESUME_TEXT_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_resume_texts.pkl"
)

JOB_TEXT_PATH = os.path.join(
    ROOT,
    "V3",
    "Outputs",
    "v3_job_texts.pkl"
)

wrong = pd.read_csv(
    WRONG_PATH
)

analysis_df = pd.read_csv(
    ANALYSIS_PATH
)

print("=" * 100)
print("V3 ERROR ANALYSIS — DEEP DIVE")
print("=" * 100)

print(
    "\nTotal wrong Top-1 cases:",
    len(wrong)
)

# ------------------------------------------------------------
# Most severe:
# best relevant job ranked furthest down
# ------------------------------------------------------------

worst = wrong.sort_values(
    [
        "Best_Label_Rank",
        "Top1_Score_Margin"
    ],
    ascending=[
        False,
        False
    ]
).head(20)

print("\n" + "=" * 100)
print("20 WORST TOP-1 FAILURES")
print("=" * 100)

print(
    worst[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label",
            "Best_Label",
            "Predicted_Rank",
            "Best_Label_Rank",
            "ExtraTrees_Tuned_Score",
            "Best_Label_Score",
            "Top1_Score_Margin",
            "Cosine",
            "Skill_Score",
            "Required_Skill_Count",
            "Matched_Skill_Count",
            "Role_Similarity",
            "Semantic_Similarity"
        ]
    ]
    .round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# How often was a strong job just barely missed?
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("BEST-LABEL RANK BUCKETS")
print("=" * 100)

buckets = pd.cut(
    wrong["Best_Label_Rank"],
    bins=[
        0,
        1,
        3,
        5,
        10,
        np.inf
    ],
    labels=[
        "Rank 1",
        "Rank 2-3",
        "Rank 4-5",
        "Rank 6-10",
        "Rank 11+"
    ]
)

print(
    buckets.value_counts(
        sort=False
    )
)

# ------------------------------------------------------------
# Close calls
# ------------------------------------------------------------

close_calls = wrong[
    wrong["Top1_Score_Margin"] < 0.05
].copy()

print(
    "\nClose Top-1 mistakes "
    "(score margin < 0.05):",
    len(close_calls),
    "/",
    len(wrong)
)

# ------------------------------------------------------------
# Feature patterns in wrong predictions
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("AVERAGE FEATURES — WRONG TOP-1")
print("=" * 100)

feature_cols = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Missing_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

print(
    analysis_df[
        analysis_df["Predicted_Rank"] == 1
    ][
        feature_cols
    ]
    .describe()
    .T[
        [
            "mean",
            "std",
            "min",
            "50%",
            "max"
        ]
    ]
    .round(5)
    .to_string()
)

# ------------------------------------------------------------
# Load text maps if available
# ------------------------------------------------------------

def load_pickle_map(path):

    if not os.path.exists(path):
        return None

    with open(
        path,
        "rb"
    ) as f:
        obj = pickle.load(f)

    return obj


resume_texts = load_pickle_map(
    RESUME_TEXT_PATH
)

job_texts = load_pickle_map(
    JOB_TEXT_PATH
)

print(
    "\nResume text object:",
    type(resume_texts)
)

print(
    "Job text object:",
    type(job_texts)
)

# ------------------------------------------------------------
# Helper for list/dict
# ------------------------------------------------------------

def get_text(
    obj,
    index
):

    if obj is None:
        return "[TEXT NOT LOADED]"

    try:

        if isinstance(obj, dict):
            return str(
                obj.get(
                    int(index),
                    obj.get(
                        str(index),
                        "[TEXT NOT FOUND]"
                    )
                )
            )

        return str(
            obj[int(index)]
        )

    except Exception:
        return "[TEXT NOT FOUND]"


# ------------------------------------------------------------
# Print a few actual failures
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ACTUAL RESUME / JOB FAILURE EXAMPLES")
print("=" * 100)

for _, row in worst.head(10).iterrows():

    trial = int(row["Trial"])
    resume_idx = int(row["Resume_Index"])
    predicted_job = int(row["Job_Index"])

    best_row = analysis_df[
        (
            analysis_df["Trial"] == trial
        )
        &
        (
            analysis_df["Resume_Index"]
            == resume_idx
        )
        &
        (
            analysis_df["Label"]
            == analysis_df[
                (
                    analysis_df["Trial"]
                    == trial
                )
                &
                (
                    analysis_df["Resume_Index"]
                    == resume_idx
                )
            ]["Label"].max()
        )
    ].sort_values(
        "ExtraTrees_Tuned_Score",
        ascending=False
    ).iloc[0]

    best_job = int(
        best_row["Job_Index"]
    )

    print("\n" + "-" * 100)

    print(
        f"Trial={trial} | Resume={resume_idx}"
    )

    print(
        f"Predicted Job={predicted_job} "
        f"(label={int(row['Label'])}, "
        f"rank={int(row['Predicted_Rank'])})"
    )

    print(
        f"Best-label Job={best_job} "
        f"(label={int(best_row['Label'])}, "
        f"rank={int(best_row['Predicted_Rank'])})"
    )

    print(
        "\nRESUME:"
    )

    print(
        get_text(
            resume_texts,
            resume_idx
        )[:1500]
    )

    print(
        "\nPREDICTED JOB:"
    )

    print(
        get_text(
            job_texts,
            predicted_job
        )[:1000]
    )

    print(
        "\nBEST-LABEL JOB:"
    )

    print(
        get_text(
            job_texts,
            best_job
        )[:1000]
    )

print("\n" + "=" * 100)
print("DEEP DIVE COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# V3 LABEL-QUALITY AUDIT — CORRECTED
# 452 WRONG TOP-1 CASES
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

WRONG_PATH = os.path.join(
    BASE, "v3_error_analysis_wrong_top1.csv"
)

PAIR_PATH = os.path.join(
    BASE, "v3_error_analysis_pair_level_25400.csv"
)

DATA_PATH = os.path.join(
    BASE, "v3_supervised_dataset_568210_rich6_semantic.csv"
)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

wrong = pd.read_csv(WRONG_PATH)
pair = pd.read_csv(PAIR_PATH)

print("Wrong Top-1 shape:", wrong.shape)
print("Pair-level shape:", pair.shape)

# Actual V3 feature columns
feature_cols = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

use_cols = [
    "Resume_Index",
    "Job_Index",
    "Label"
] + feature_cols

full = pd.read_csv(
    DATA_PATH,
    usecols=use_cols
)

print("Full V3 dataset:", full.shape)

# ------------------------------------------------------------
# 2. CREATE FAST LOOKUP KEY
# ------------------------------------------------------------

full["_key"] = (
    full["Resume_Index"].astype(str)
    + "_"
    + full["Job_Index"].astype(str)
)

pair["_key"] = (
    pair["Resume_Index"].astype(str)
    + "_"
    + pair["Job_Index"].astype(str)
)

# ------------------------------------------------------------
# 3. MERGE FEATURES INTO PAIR-LEVEL DATA
# ------------------------------------------------------------

pair_small_cols = [
    "Trial",
    "Seed",
    "Resume_Index",
    "Job_Index",
    "Label",
    "ExtraTrees_Tuned_Score",
    "Missing_Skill_Count"
]

pair_small = pair[pair_small_cols].copy()

pair_small["_key"] = (
    pair_small["Resume_Index"].astype(str)
    + "_"
    + pair_small["Job_Index"].astype(str)
)

pair_small = pair_small.merge(
    full.drop_duplicates("_key"),
    on="_key",
    how="left",
    suffixes=("", "_full")
)

print("Merged pair data:", pair_small.shape)

# ------------------------------------------------------------
# 4. CHECK MERGE
# ------------------------------------------------------------

missing_features = pair_small[feature_cols].isna().any(axis=1).sum()

print("Rows with missing feature data:", missing_features)

if missing_features > 0:
    print("\nWARNING: Some feature rows did not match.")

# ------------------------------------------------------------
# 5. RANK ALL 10 JOBS
# ------------------------------------------------------------

pair_small["rank"] = (
    pair_small
    .groupby(["Trial", "Resume_Index"])
    ["ExtraTrees_Tuned_Score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# ------------------------------------------------------------
# 6. TOP-1 PREDICTION FOR EACH RESUME/TRIAL
# ------------------------------------------------------------

top1 = pair_small[
    pair_small["rank"] == 1
].copy()

# ------------------------------------------------------------
# 7. BEST LABEL-2 JOB
# ------------------------------------------------------------

label2 = pair_small[
    pair_small["Label"] == 2
].copy()

label2 = label2.sort_values(
    ["Trial", "Resume_Index", "ExtraTrees_Tuned_Score"],
    ascending=[True, True, False]
)

best_label2 = (
    label2
    .groupby(
        ["Trial", "Resume_Index"],
        as_index=False
    )
    .first()
)

# Rename label-2 columns
best_label2 = best_label2.rename(
    columns={
        "Job_Index": "Best_Label2_Job_Index",
        "Label": "Best_Label2",
        "ExtraTrees_Tuned_Score": "Best_Label2_Score",
        "rank": "Best_Label2_Rank"
    }
)

# Add prefix to feature columns
for f in feature_cols:
    best_label2 = best_label2.rename(
        columns={f: f"Best_Label2_{f}"}
    )

best_label2 = best_label2.rename(
    columns={
        "Missing_Skill_Count":
            "Best_Label2_Missing_Skill_Count"
    }
)

# ------------------------------------------------------------
# 8. MERGE TOP-1 WITH BEST LABEL-2
# ------------------------------------------------------------

audit = top1.merge(
    best_label2[
        [
            "Trial",
            "Resume_Index",
            "Best_Label2_Job_Index",
            "Best_Label2",
            "Best_Label2_Score",
            "Best_Label2_Rank"
        ]
        + [f"Best_Label2_{f}" for f in feature_cols]
        + ["Best_Label2_Missing_Skill_Count"]
    ],
    on=["Trial", "Resume_Index"],
    how="inner"
)

# Keep actual Top-1 mistakes only
audit = audit[
    audit["Label"] != 2
].copy()

print("\nWrong Top-1 cases recovered:", len(audit))

# ------------------------------------------------------------
# 9. FEATURE GAPS
#
# Positive  = label-2 job has stronger feature
# Negative  = predicted Top-1 has stronger feature
# ------------------------------------------------------------

for f in feature_cols:
    audit[f"{f}_Gap"] = (
        audit[f"Best_Label2_{f}"] - audit[f]
    )

audit["Missing_Skill_Count_Gap"] = (
    audit["Best_Label2_Missing_Skill_Count"]
    - audit["Missing_Skill_Count"]
)

audit["Model_Score_Gap"] = (
    audit["Best_Label2_Score"]
    - audit["ExtraTrees_Tuned_Score"]
)

# ------------------------------------------------------------
# 10. COUNT WHICH JOB WINS ON FEATURES
# ------------------------------------------------------------

gap_cols = [
    f"{f}_Gap"
    for f in feature_cols
]

audit["Features_Favoring_Label2"] = (
    audit[gap_cols] > 0
).sum(axis=1)

audit["Features_Favoring_Top1"] = (
    audit[gap_cols] < 0
).sum(axis=1)

audit["Feature_Ties"] = (
    audit[gap_cols].abs() < 1e-8
).sum(axis=1)

# ------------------------------------------------------------
# 11. AUDIT CATEGORY
# ------------------------------------------------------------

def classify_case(row):

    label2_wins = row["Features_Favoring_Label2"]
    top1_wins = row["Features_Favoring_Top1"]
    score_gap = row["Model_Score_Gap"]

    # Strong indication that model-selected job
    # has better feature evidence than the label-2 job
    if top1_wins >= 6 and score_gap < -0.05:
        return "HIGH_LABEL_NOISE_SUSPICION"

    if top1_wins >= 5 and score_gap < -0.03:
        return "MODERATE_LABEL_NOISE_SUSPICION"

    # Label-2 job has stronger feature evidence
    if label2_wins >= 5 and score_gap > 0:
        return "LIKELY_MODEL_ERROR"

    return "AMBIGUOUS"


audit["Audit_Category"] = audit.apply(
    classify_case,
    axis=1
)

# ------------------------------------------------------------
# 12. SUMMARY
# ------------------------------------------------------------

summary = (
    audit["Audit_Category"]
    .value_counts()
    .rename_axis("Audit_Category")
    .reset_index(name="Count")
)

summary["Percentage"] = (
    summary["Count"]
    / len(audit)
    * 100
)

summary = summary.sort_values(
    "Count",
    ascending=False
)

print("\n" + "=" * 75)
print("LABEL QUALITY AUDIT SUMMARY")
print("=" * 75)

print(
    summary.to_string(index=False)
)

# ------------------------------------------------------------
# 13. FEATURE EVIDENCE
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("AVERAGE FEATURE GAPS")
print("=" * 75)

gap_summary = (
    audit[
        gap_cols
        + [
            "Missing_Skill_Count_Gap",
            "Model_Score_Gap"
        ]
    ]
    .mean()
    .sort_values()
)

print(
    gap_summary.to_string()
)

# ------------------------------------------------------------
# 14. MOST SUSPICIOUS CASES
# ------------------------------------------------------------

suspicious = audit[
    audit["Audit_Category"].isin([
        "HIGH_LABEL_NOISE_SUSPICION",
        "MODERATE_LABEL_NOISE_SUSPICION"
    ])
].copy()

suspicious["Suspicion_Score"] = (
    suspicious["Features_Favoring_Top1"] * 0.5
    + (
        -suspicious["Model_Score_Gap"]
    ).clip(lower=0) * 10
)

suspicious = suspicious.sort_values(
    "Suspicion_Score",
    ascending=False
)

print("\n" + "=" * 75)
print("TOP 25 POSSIBLE LABEL-NOISE CASES")
print("=" * 75)

display(
    suspicious[
        [
            "Trial",
            "Resume_Index",

            "Job_Index",
            "Label",
            "ExtraTrees_Tuned_Score",

            "Best_Label2_Job_Index",
            "Best_Label2",
            "Best_Label2_Score",
            "Best_Label2_Rank",

            "Features_Favoring_Top1",
            "Features_Favoring_Label2",
            "Model_Score_Gap",

            "Cosine_Gap",
            "Skill_Score_Gap",
            "Normalized_Cosine_Gap",
            "Required_Skill_Count_Gap",
            "Matched_Skill_Count_Gap",
            "Role_Similarity_Gap",
            "Semantic_Similarity_Gap",

            "Audit_Category"
        ]
    ].head(25)
)

# ------------------------------------------------------------
# 15. HOW MANY ARE CLEARLY MODEL-SIDED VS LABEL-SIDED?
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FEATURE-WIN DISTRIBUTION")
print("=" * 75)

print(
    audit["Features_Favoring_Top1"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 16. SAVE
# ------------------------------------------------------------

AUDIT_PATH = os.path.join(
    BASE,
    "v3_label_quality_audit_452_wrong_top1.csv"
)

SUMMARY_PATH = os.path.join(
    BASE,
    "v3_label_quality_audit_summary.csv"
)

SUSPICIOUS_PATH = os.path.join(
    BASE,
    "v3_label_quality_audit_suspicious_cases.csv"
)

audit.to_csv(
    AUDIT_PATH,
    index=False
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

suspicious.to_csv(
    SUSPICIOUS_PATH,
    index=False
)

print("\n" + "=" * 75)
print("SAVED")
print("=" * 75)

print(AUDIT_PATH)
print(SUMMARY_PATH)
print(SUSPICIOUS_PATH)

print("\n✅ LABEL-QUALITY AUDIT COMPLETE")

In [ ]:
# ============================================================
# RECONCILE ORIGINAL 452 WRONG CASES VS RECOMPUTED CASES
# ============================================================

import os
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

WRONG_PATH = os.path.join(
    BASE, "v3_error_analysis_wrong_top1.csv"
)

PAIR_PATH = os.path.join(
    BASE, "v3_error_analysis_pair_level_25400.csv"
)

wrong = pd.read_csv(WRONG_PATH)
pair = pd.read_csv(PAIR_PATH)

# ------------------------------------------------------------
# 1. Create unique evaluation-case keys
# ------------------------------------------------------------

wrong["_case"] = (
    wrong["Trial"].astype(str)
    + "_"
    + wrong["Resume_Index"].astype(str)
)

pair["_case"] = (
    pair["Trial"].astype(str)
    + "_"
    + pair["Resume_Index"].astype(str)
)

# ------------------------------------------------------------
# 2. ORIGINAL WRONG CASES
# ------------------------------------------------------------

original_wrong_cases = set(
    wrong["_case"]
)

print("=" * 75)
print("ORIGINAL ERROR ANALYSIS")
print("=" * 75)

print("Original wrong cases:", len(original_wrong_cases))

# ------------------------------------------------------------
# 3. RECOMPUTE RANKS
# ------------------------------------------------------------

pair["Recomputed_Rank"] = (
    pair
    .groupby(["Trial", "Resume_Index"])
    ["ExtraTrees_Tuned_Score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

recomputed_top1 = pair[
    pair["Recomputed_Rank"] == 1
].copy()

recomputed_wrong = recomputed_top1[
    recomputed_top1["Label"] != 2
].copy()

recomputed_wrong_cases = set(
    recomputed_wrong["_case"]
)

print("\nRecomputed wrong cases:", len(recomputed_wrong_cases))

# ------------------------------------------------------------
# 4. COMPARE
# ------------------------------------------------------------

missing_from_recomputed = (
    original_wrong_cases
    - recomputed_wrong_cases
)

new_wrong_cases = (
    recomputed_wrong_cases
    - original_wrong_cases
)

intersection = (
    original_wrong_cases
    & recomputed_wrong_cases
)

print("\n" + "=" * 75)
print("RECONCILIATION")
print("=" * 75)

print("Original wrong:", len(original_wrong_cases))
print("Recomputed wrong:", len(recomputed_wrong_cases))
print("Intersection:", len(intersection))
print("Original but NOT recomputed:", len(missing_from_recomputed))
print("Recomputed but NOT original:", len(new_wrong_cases))

# ------------------------------------------------------------
# 5. SHOW THE 53 DISCREPANCIES
# ------------------------------------------------------------

if missing_from_recomputed:

    discrepancy = wrong[
        wrong["_case"].isin(
            missing_from_recomputed
        )
    ].copy()

    print("\n" + "=" * 75)
    print("ORIGINAL WRONG CASES THAT ARE NOT RECOMPUTED AS WRONG")
    print("=" * 75)

    display(
        discrepancy[
            [
                "Trial",
                "Resume_Index",
                "Job_Index",
                "Label",
                "Best_Label",
                "Predicted_Rank",
                "Best_Label_Rank",
                "ExtraTrees_Tuned_Score",
                "Best_Label_Score"
            ]
        ].sort_values(
            ["Trial", "Resume_Index"]
        )
    )

# ------------------------------------------------------------
# 6. CHECK WHETHER THESE CASES STILL EXIST IN PAIR DATA
# ------------------------------------------------------------

if missing_from_recomputed:

    check = pair[
        pair["_case"].isin(
            missing_from_recomputed
        )
    ].copy()

    print("\n" + "=" * 75)
    print("WHY WERE THEY NOT CLASSIFIED AS WRONG?")
    print("=" * 75)

    debug = (
        check
        .groupby(
            ["Trial", "Resume_Index"]
        )
        .agg(
            Top1_Score=(
                "ExtraTrees_Tuned_Score",
                "max"
            ),
            Max_Label=(
                "Label",
                "max"
            ),
            Num_Label2=(
                "Label",
                lambda x: (x == 2).sum()
            ),
            Num_Jobs=(
                "Job_Index",
                "count"
            )
        )
        .reset_index()
    )

    display(
        debug.head(100)
    )

# ------------------------------------------------------------
# 7. CHECK LABEL-2 AVAILABILITY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("LABEL-2 COVERAGE")
print("=" * 75)

case_stats = (
    pair
    .groupby(
        ["Trial", "Resume_Index"]
    )
    .agg(
        Num_Jobs=("Job_Index", "count"),
        Num_Label2=(
            "Label",
            lambda x: (x == 2).sum()
        ),
        Top1_Label=(
            "Label",
            lambda x: x.iloc[
                np.argmax(
                    pair.loc[
                        x.index,
                        "ExtraTrees_Tuned_Score"
                    ].values
                )
            ]
        )
    )
    .reset_index()
)

print(
    case_stats["Num_Label2"]
    .value_counts()
    .sort_index()
)

print(
    "\nCases with zero label-2 jobs:",
    (case_stats["Num_Label2"] == 0).sum()
)

# ------------------------------------------------------------
# 8. SCORE TIE CHECK
# ------------------------------------------------------------

pair["Score_Rank_Count"] = (
    pair
    .groupby(["Trial", "Resume_Index"])
    ["ExtraTrees_Tuned_Score"]
    .transform(
        lambda x: x.duplicated(
            keep=False
        ).sum()
    )
)

ties = pair[
    pair["Score_Rank_Count"] > 0
]

print("\n" + "=" * 75)
print("SCORE TIES")
print("=" * 75)

print(
    "Cases containing tied prediction scores:",
    ties[
        ["Trial", "Resume_Index"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\n✅ Reconciliation complete.")

In [ ]:
# ============================================================
# FINAL RECONCILIATION OF THE 452 TOP-1 ERRORS
# ============================================================

import os
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

PAIR_PATH = os.path.join(
    BASE,
    "v3_error_analysis_pair_level_25400.csv"
)

WRONG_PATH = os.path.join(
    BASE,
    "v3_error_analysis_wrong_top1.csv"
)

pair = pd.read_csv(PAIR_PATH)
wrong = pd.read_csv(WRONG_PATH)

# ------------------------------------------------------------
# 1. Rank within each evaluation case
# ------------------------------------------------------------

pair["rank"] = (
    pair
    .groupby(["Trial", "Resume_Index"])
    ["ExtraTrees_Tuned_Score"]
    .rank(
        method="first",
        ascending=False
    )
)

top1 = pair[pair["rank"] == 1].copy()

# ------------------------------------------------------------
# 2. Number of label-2 jobs per case
# ------------------------------------------------------------

case_info = (
    pair
    .groupby(["Trial", "Resume_Index"])
    .agg(
        Num_Jobs=("Job_Index", "count"),

        Num_Label2=(
            "Label",
            lambda x: (x == 2).sum()
        ),

        Top1_Label=(
            "Label",
            lambda x: x.loc[
                pair.loc[
                    x.index,
                    "ExtraTrees_Tuned_Score"
                ].idxmax()
            ]
        ),

        Num_NaN_Scores=(
            "ExtraTrees_Tuned_Score",
            lambda x: x.isna().sum()
        ),

        Max_Score=(
            "ExtraTrees_Tuned_Score",
            "max"
        ),

        Min_Score=(
            "ExtraTrees_Tuned_Score",
            "min"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. Original error cases
# ------------------------------------------------------------

wrong["_case"] = (
    wrong["Trial"].astype(str)
    + "_"
    + wrong["Resume_Index"].astype(str)
)

case_info["_case"] = (
    case_info["Trial"].astype(str)
    + "_"
    + case_info["Resume_Index"].astype(str)
)

original_wrong_cases = set(
    wrong["_case"]
)

# ------------------------------------------------------------
# 4. THE IMPORTANT GROUPS
# ------------------------------------------------------------

zero_label2 = case_info[
    case_info["Num_Label2"] == 0
].copy()

has_label2 = case_info[
    case_info["Num_Label2"] > 0
].copy()

print("=" * 80)
print("CASE-LEVEL RECONCILIATION")
print("=" * 80)

print("Total cases:", len(case_info))
print("Cases with label-2:", len(has_label2))
print("Cases without label-2:", len(zero_label2))

print("\nTop-1 label distribution:")
print(
    case_info["Top1_Label"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nZero-label-2 cases — Top-1 label distribution:")
print(
    zero_label2["Top1_Label"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nZero-label-2 cases with NaN prediction scores:",
    zero_label2["Num_NaN_Scores"].gt(0).sum()
)

print(
    "Zero-label-2 cases with all scores NaN:",
    zero_label2["Max_Score"].isna().sum()
)

# ------------------------------------------------------------
# 5. FIND THE 53 CASES
# ------------------------------------------------------------

# Cases with NO label 2 that are NOT in original errors
zero_label2_not_original = zero_label2[
    ~zero_label2["_case"].isin(original_wrong_cases)
].copy()

print("\n" + "=" * 80)
print("ZERO-LABEL-2 CASES NOT IN ORIGINAL ERROR FILE")
print("=" * 80)

print(
    "Count:",
    len(zero_label2_not_original)
)

display(
    zero_label2_not_original[
        [
            "Trial",
            "Resume_Index",
            "Num_Jobs",
            "Num_Label2",
            "Top1_Label",
            "Num_NaN_Scores",
            "Max_Score",
            "Min_Score"
        ]
    ].head(100)
)

# ------------------------------------------------------------
# 6. CHECK SCORE TIES IN THESE 53
# ------------------------------------------------------------

zero_label2_not_original = zero_label2_not_original.copy()

def count_top_ties(case_df):

    scores = case_df[
        "ExtraTrees_Tuned_Score"
    ].dropna()

    if len(scores) == 0:
        return np.nan

    max_score = scores.max()

    return (
        np.isclose(
            scores.values,
            max_score,
            rtol=1e-9,
            atol=1e-9
        ).sum()
    )

tie_counts = (
    pair[
        pair.set_index(
            ["Trial", "Resume_Index"]
        ).index.isin(
            zero_label2_not_original.set_index(
                ["Trial", "Resume_Index"]
            ).index
        )
    ]
    .groupby(
        ["Trial", "Resume_Index"]
    )
    .apply(
        count_top_ties
    )
    .reset_index(
        name="Top_Score_Tie_Count"
    )
)

print("\n" + "=" * 80)
print("TOP-SCORE TIE CHECK FOR THE 53 CASES")
print("=" * 80)

print(
    tie_counts["Top_Score_Tie_Count"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# 7. LABEL DISTRIBUTION OF ALL 2540 CASES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMBER OF LABEL-2 JOBS PER CASE")
print("=" * 80)

print(
    case_info["Num_Label2"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 8. RECONSTRUCT THE ORIGINAL EVALUATION LOGIC
# ------------------------------------------------------------

# A case is evaluated ONLY when at least one label-2 job exists.
#
# Correct = model's top ranked job has Label 2
# Wrong   = model's top ranked job is NOT Label 2

evaluated = case_info[
    case_info["Num_Label2"] > 0
].copy()

reconstructed_correct = (
    evaluated["Top1_Label"] == 2
).sum()

reconstructed_wrong = (
    evaluated["Top1_Label"] != 2
).sum()

print("\n" + "=" * 80)
print("RECONSTRUCTED ORIGINAL METRIC")
print("=" * 80)

print("Evaluated cases:", len(evaluated))
print("Correct:", reconstructed_correct)
print("Wrong:", reconstructed_wrong)

print(
    "Top-1:",
    reconstructed_correct / len(evaluated)
)

print(
    "\nOriginal reported:",
    "2088 correct / 452 wrong"
)

print(
    "Original Top-1:",
    2088 / 2540
)

print("\n✅ RECONCILIATION FINISHED")

In [ ]:
# ============================================================
# DIAGNOSE THE 55 NO-LABEL-2 / TOP-1-LABEL-0 CASES
# ============================================================

import os
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

PAIR_PATH = os.path.join(
    BASE,
    "v3_error_analysis_pair_level_25400.csv"
)

WRONG_PATH = os.path.join(
    BASE,
    "v3_error_analysis_wrong_top1.csv"
)

pair = pd.read_csv(PAIR_PATH)
wrong = pd.read_csv(WRONG_PATH)

# ------------------------------------------------------------
# Recompute rank using the evaluation scores
# ------------------------------------------------------------

pair["Rank_Recomputed"] = (
    pair
    .groupby(["Trial", "Resume_Index"])
    ["ExtraTrees_Tuned_Score"]
    .rank(
        method="first",
        ascending=False
    )
)

# ------------------------------------------------------------
# Find cases with:
#   no label 2
#   top1 label = 0
# ------------------------------------------------------------

top1 = pair[
    pair["Rank_Recomputed"] == 1
].copy()

case_label2 = (
    pair
    .groupby(["Trial", "Resume_Index"])["Label"]
    .apply(lambda x: (x == 2).sum())
    .reset_index(name="Num_Label2")
)

zero2_top1_0 = top1[
    top1["Label"] == 0
].merge(
    case_label2,
    on=["Trial", "Resume_Index"]
)

zero2_top1_0 = zero2_top1_0[
    zero2_top1_0["Num_Label2"] == 0
].copy()

# ------------------------------------------------------------
# Original error cases
# ------------------------------------------------------------

wrong["_case"] = (
    wrong["Trial"].astype(str)
    + "_"
    + wrong["Resume_Index"].astype(str)
)

zero2_top1_0["_case"] = (
    zero2_top1_0["Trial"].astype(str)
    + "_"
    + zero2_top1_0["Resume_Index"].astype(str)
)

zero2_top1_0["Original_Error"] = (
    zero2_top1_0["_case"].isin(
        set(wrong["_case"])
    )
)

print("=" * 75)
print("NO-LABEL-2 / TOP-1-LABEL-0 DIAGNOSTIC")
print("=" * 75)

print(
    "Total such cases:",
    len(zero2_top1_0)
)

print(
    "Present in original 452 errors:",
    zero2_top1_0["Original_Error"].sum()
)

print(
    "NOT present in original 452 errors:",
    (~zero2_top1_0["Original_Error"]).sum()
)

# ------------------------------------------------------------
# Inspect all 55
# ------------------------------------------------------------

display(
    zero2_top1_0[
        [
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label",
            "ExtraTrees_Tuned_Score",
            "Num_Label2",
            "Original_Error"
        ]
    ]
    .sort_values(
        ["Original_Error", "Trial", "Resume_Index"],
        ascending=[False, True, True]
    )
)

# ------------------------------------------------------------
# Check whether the two exceptions have score ties
# ------------------------------------------------------------

exceptions = zero2_top1_0[
    ~zero2_top1_0["Original_Error"]
]

print("\n" + "=" * 75)
print("THE 2 EXCEPTIONS")
print("=" * 75)

for _, row in exceptions.iterrows():

    t = row["Trial"]
    r = row["Resume_Index"]

    g = pair[
        (pair["Trial"] == t) &
        (pair["Resume_Index"] == r)
    ].copy()

    max_score = g["ExtraTrees_Tuned_Score"].max()

    tied = g[
        np.isclose(
            g["ExtraTrees_Tuned_Score"],
            max_score,
            rtol=1e-9,
            atol=1e-9
        )
    ]

    print(f"\nTrial={t}, Resume={r}")
    print("Max score:", max_score)
    print("Top-score ties:", len(tied))

    display(
        tied[
            [
                "Job_Index",
                "Label",
                "ExtraTrees_Tuned_Score"
            ]
        ]
        .sort_values("Job_Index")
    )

In [ ]:
# ============================================================
# V3 — FINAL 452-CASE LABEL QUALITY AUDIT
# USE ORIGINAL ERROR FILE AS SOURCE OF TRUTH
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

WRONG_PATH = os.path.join(
    BASE,
    "v3_error_analysis_wrong_top1.csv"
)

PAIR_PATH = os.path.join(
    BASE,
    "v3_error_analysis_pair_level_25400.csv"
)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

wrong = pd.read_csv(WRONG_PATH)
pair = pd.read_csv(PAIR_PATH)

print("=" * 80)
print("FINAL 452-CASE LABEL QUALITY AUDIT")
print("=" * 80)

print("Original wrong cases:", len(wrong))
print("Pair-level rows:", len(pair))

assert len(wrong) == 452, \
    f"Expected 452 wrong cases, got {len(wrong)}"

# ------------------------------------------------------------
# 2. RANK THE 10 JOBS IN EACH TRIAL/RESUME
# ------------------------------------------------------------

pair["Computed_Rank"] = (
    pair
    .groupby(["Trial", "Resume_Index"])
    ["ExtraTrees_Tuned_Score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

# ------------------------------------------------------------
# 3. GET BEST-LABEL JOB
#
# IMPORTANT:
# We reproduce the original concept:
# highest available label in each 10-job set.
#
# Prefer label 2.
# Otherwise label 1.
# Otherwise label 0.
# ------------------------------------------------------------

pair["Label"] = pair["Label"].astype(int)

best_label = (
    pair
    .sort_values(
        [
            "Trial",
            "Resume_Index",
            "Label",
            "ExtraTrees_Tuned_Score"
        ],
        ascending=[
            True,
            True,
            False,
            False
        ]
    )
    .groupby(
        ["Trial", "Resume_Index"],
        as_index=False
    )
    .first()
)

best_label = best_label.rename(
    columns={
        "Job_Index": "Best_Label_Job_Index",
        "Label": "Best_Label",
        "ExtraTrees_Tuned_Score":
            "Best_Label_Model_Score",
        "Computed_Rank":
            "Best_Label_Rank_Computed"
    }
)

# ------------------------------------------------------------
# 4. FEATURE COLUMN NAMES
# ------------------------------------------------------------

features = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

# ------------------------------------------------------------
# 5. MERGE BEST-LABEL JOB INTO ORIGINAL 452
# ------------------------------------------------------------

audit = wrong.merge(
    best_label[
        [
            "Trial",
            "Resume_Index",
            "Best_Label_Job_Index",
            "Best_Label",
            "Best_Label_Model_Score",
            "Best_Label_Rank_Computed"
        ]
        + features
    ],
    on=[
        "Trial",
        "Resume_Index"
    ],
    how="left",
    suffixes=(
        "",
        "_BestLabel"
    )
)

print(
    "\nRows after merging best-label job:",
    len(audit)
)

# ------------------------------------------------------------
# 6. VALIDATE
# ------------------------------------------------------------

print("\nMissing best-label jobs:",
      audit["Best_Label_Job_Index"].isna().sum())

print(
    "Best_Label mismatches with original:",
    (
        audit["Best_Label_y"]
        if "Best_Label_y" in audit.columns
        else pd.Series(dtype=float)
    ).shape[0]
)

# Handle possible suffix names
if "Best_Label_y" in audit.columns:
    audit["Best_Label_Final"] = audit["Best_Label_y"]
else:
    audit["Best_Label_Final"] = audit["Best_Label"]

# ------------------------------------------------------------
# 7. FEATURE GAPS
#
# Positive = best-label job stronger
# Negative = predicted Top-1 stronger
# ------------------------------------------------------------

for f in features:

    best_col = f"{f}_BestLabel"

    if best_col not in audit.columns:
        # because merge may have kept original feature names
        best_col = f"{f}_y"

    if best_col not in audit.columns:
        raise ValueError(
            f"Could not find best-label feature column for {f}"
        )

    audit[f"{f}_Gap"] = (
        audit[best_col] - audit[f]
    )

# ------------------------------------------------------------
# 8. MODEL SCORE GAP
# ------------------------------------------------------------

audit["Model_Score_Gap"] = (
    audit["Best_Label_Model_Score"]
    - audit["ExtraTrees_Tuned_Score"]
)

# ------------------------------------------------------------
# 9. FEATURE WINS
# ------------------------------------------------------------

gap_cols = [
    f"{f}_Gap"
    for f in features
]

audit["Features_Favoring_Top1"] = (
    audit[gap_cols] < 0
).sum(axis=1)

audit["Features_Favoring_BestLabel"] = (
    audit[gap_cols] > 0
).sum(axis=1)

audit["Feature_Ties"] = (
    audit[gap_cols].abs() < 1e-8
).sum(axis=1)

# ------------------------------------------------------------
# 10. FEATURE AGREEMENT SCORE
#
# How many of the 7 features agree with the model?
# ------------------------------------------------------------

audit["Top1_Feature_Win_Ratio"] = (
    audit["Features_Favoring_Top1"] / len(features)
)

audit["BestLabel_Feature_Win_Ratio"] = (
    audit["Features_Favoring_BestLabel"] / len(features)
)

# ------------------------------------------------------------
# 11. CLASSIFY
# ------------------------------------------------------------

def classify(row):

    top1_wins = row["Features_Favoring_Top1"]
    best_wins = row["Features_Favoring_BestLabel"]
    score_gap = row["Model_Score_Gap"]

    # Very strong evidence that the model-selected
    # job is more feature-compatible than the label.
    if top1_wins >= 6 and score_gap < -0.05:
        return "STRONG_LABEL_NOISE"

    if top1_wins >= 5 and score_gap < -0.03:
        return "POSSIBLE_LABEL_NOISE"

    # Strong evidence that the labelled-best job
    # is genuinely better and the model missed it.
    if best_wins >= 6 and score_gap > 0.05:
        return "STRONG_MODEL_ERROR"

    if best_wins >= 5 and score_gap > 0.03:
        return "POSSIBLE_MODEL_ERROR"

    return "AMBIGUOUS"

audit["Audit_Category"] = audit.apply(
    classify,
    axis=1
)

# ------------------------------------------------------------
# 12. SUMMARY
# ------------------------------------------------------------

summary = (
    audit["Audit_Category"]
    .value_counts()
    .rename_axis("Audit_Category")
    .reset_index(name="Count")
)

summary["Percentage"] = (
    summary["Count"] / len(audit) * 100
)

print("\n" + "=" * 80)
print("AUDIT SUMMARY — ALL 452 ORIGINAL ERRORS")
print("=" * 80)

print(
    summary.to_string(index=False)
)

# ------------------------------------------------------------
# 13. FEATURE GAP SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AVERAGE FEATURE GAPS")
print("=" * 80)

gap_summary = (
    audit[
        gap_cols + ["Model_Score_Gap"]
    ]
    .mean()
    .sort_values()
)

print(
    gap_summary.to_string()
)

# ------------------------------------------------------------
# 14. FEATURE-WIN DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP-1 FEATURE-WIN DISTRIBUTION")
print("=" * 80)

print(
    audit["Features_Favoring_Top1"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 15. STRONGEST POSSIBLE LABEL-NOISE CASES
# ------------------------------------------------------------

noise = audit[
    audit["Audit_Category"].isin([
        "STRONG_LABEL_NOISE",
        "POSSIBLE_LABEL_NOISE"
    ])
].copy()

noise["Noise_Evidence"] = (
    noise["Features_Favoring_Top1"] * 10
    + (-noise["Model_Score_Gap"]).clip(lower=0) * 10
)

noise = noise.sort_values(
    "Noise_Evidence",
    ascending=False
)

print("\n" + "=" * 80)
print("TOP 30 POSSIBLE LABEL-NOISE CASES")
print("=" * 80)

display(
    noise[
        [
            "Trial",
            "Resume_Index",

            "Job_Index",
            "Label",
            "ExtraTrees_Tuned_Score",

            "Best_Label_Job_Index",
            "Best_Label",
            "Best_Label_Model_Score",
            "Best_Label_Rank_Computed",

            "Features_Favoring_Top1",
            "Features_Favoring_BestLabel",
            "Model_Score_Gap",

            "Cosine_Gap",
            "Skill_Score_Gap",
            "Normalized_Cosine_Gap",
            "Required_Skill_Count_Gap",
            "Matched_Skill_Count_Gap",
            "Role_Similarity_Gap",
            "Semantic_Similarity_Gap",

            "Audit_Category"
        ]
    ].head(30)
)

# ------------------------------------------------------------
# 16. STRONGEST MODEL-ERROR CASES
# ------------------------------------------------------------

model_errors = audit[
    audit["Audit_Category"].isin([
        "STRONG_MODEL_ERROR",
        "POSSIBLE_MODEL_ERROR"
    ])
].copy()

model_errors["Model_Error_Evidence"] = (
    model_errors["Features_Favoring_BestLabel"] * 10
    + model_errors["Model_Score_Gap"].clip(lower=0) * 10
)

model_errors = model_errors.sort_values(
    "Model_Error_Evidence",
    ascending=False
)

print("\n" + "=" * 80)
print("TOP 30 POSSIBLE MODEL-ERROR CASES")
print("=" * 80)

display(
    model_errors[
        [
            "Trial",
            "Resume_Index",

            "Job_Index",
            "Label",
            "ExtraTrees_Tuned_Score",

            "Best_Label_Job_Index",
            "Best_Label",
            "Best_Label_Model_Score",
            "Best_Label_Rank_Computed",

            "Features_Favoring_Top1",
            "Features_Favoring_BestLabel",
            "Model_Score_Gap",

            "Cosine_Gap",
            "Skill_Score_Gap",
            "Normalized_Cosine_Gap",
            "Required_Skill_Count_Gap",
            "Matched_Skill_Count_Gap",
            "Role_Similarity_Gap",
            "Semantic_Similarity_Gap",

            "Audit_Category"
        ]
    ].head(30)
)

# ------------------------------------------------------------
# 17. SAVE
# ------------------------------------------------------------

AUDIT_PATH = os.path.join(
    BASE,
    "v3_final_452_label_quality_audit.csv"
)

SUMMARY_PATH = os.path.join(
    BASE,
    "v3_final_452_label_quality_audit_summary.csv"
)

NOISE_PATH = os.path.join(
    BASE,
    "v3_final_452_possible_label_noise.csv"
)

MODEL_ERROR_PATH = os.path.join(
    BASE,
    "v3_final_452_possible_model_errors.csv"
)

audit.to_csv(
    AUDIT_PATH,
    index=False
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

noise.to_csv(
    NOISE_PATH,
    index=False
)

model_errors.to_csv(
    MODEL_ERROR_PATH,
    index=False
)

print("\n" + "=" * 80)
print("SAVED")
print("=" * 80)

print(AUDIT_PATH)
print(SUMMARY_PATH)
print(NOISE_PATH)
print(MODEL_ERROR_PATH)

print("\n✅ FINAL 452-CASE AUDIT COMPLETE")

In [ ]:
# ============================================================
# BLIND RE-REVIEW DATASET FOR SUSPECTED LABEL-NOISE CASES
# ============================================================

import os
import pickle
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

AUDIT_PATH = os.path.join(
    BASE,
    "v3_final_452_label_quality_audit.csv"
)

RESUME_TEXT_PATH = os.path.join(
    BASE,
    "v3_resume_texts.pkl"
)

JOB_TEXT_PATH = os.path.join(
    BASE,
    "v3_job_texts.pkl"
)

audit = pd.read_csv(AUDIT_PATH)

print("Audit rows:", len(audit))

# ------------------------------------------------------------
# 1. Keep only suspicious cases
# ------------------------------------------------------------

suspicious = audit[
    audit["Audit_Category"].isin([
        "STRONG_LABEL_NOISE",
        "POSSIBLE_LABEL_NOISE"
    ])
].copy()

print("Suspicious cases:", len(suspicious))

# ------------------------------------------------------------
# 2. LOAD CACHED TEXTS
# ------------------------------------------------------------

with open(RESUME_TEXT_PATH, "rb") as f:
    resume_texts = pickle.load(f)

with open(JOB_TEXT_PATH, "rb") as f:
    job_texts = pickle.load(f)

print("Resume texts:", len(resume_texts))
print("Job texts:", len(job_texts))

# ------------------------------------------------------------
# 3. ADD TEXT
# ------------------------------------------------------------

suspicious["Resume_Text"] = suspicious[
    "Resume_Index"
].map(
    lambda i: resume_texts[int(i)]
)

suspicious["Predicted_Job_Text"] = suspicious[
    "Job_Index"
].map(
    lambda i: job_texts[int(i)]
)

suspicious["Best_Label_Job_Text"] = suspicious[
    "Best_Label_Job_Index"
].map(
    lambda i: job_texts[int(i)]
)

# ------------------------------------------------------------
# 4. CREATE BLIND PAIR IDENTIFIER
# ------------------------------------------------------------

suspicious["Review_ID"] = np.arange(
    1,
    len(suspicious) + 1
)

# ------------------------------------------------------------
# 5. DON'T EXPOSE ORIGINAL LABELS IN BLIND FILE
# ------------------------------------------------------------

blind = suspicious[
    [
        "Review_ID",
        "Trial",
        "Resume_Index",

        "Job_Index",
        "Best_Label_Job_Index",

        "Resume_Text",
        "Predicted_Job_Text",
        "Best_Label_Job_Text",

        # feature evidence
        "Features_Favoring_Top1",
        "Features_Favoring_BestLabel",

        "Cosine_Gap",
        "Skill_Score_Gap",
        "Normalized_Cosine_Gap",
        "Required_Skill_Count_Gap",
        "Matched_Skill_Count_Gap",
        "Role_Similarity_Gap",
        "Semantic_Similarity_Gap",

        "Audit_Category"
    ]
].copy()

# ------------------------------------------------------------
# 6. BLIND JOB NAMES
#
# Randomly call them A/B so the reviewer doesn't know
# which one was originally label-2.
# ------------------------------------------------------------

rng = np.random.default_rng(20261008)

blind["Swap"] = rng.integers(
    0,
    2,
    size=len(blind)
)

def assign_jobs(row):

    if row["Swap"] == 0:
        return pd.Series({
            "Job_A": row["Predicted_Job_Text"],
            "Job_B": row["Best_Label_Job_Text"],
            "Original_Model_Job_Position": "A"
        })

    else:
        return pd.Series({
            "Job_A": row["Best_Label_Job_Text"],
            "Job_B": row["Predicted_Job_Text"],
            "Original_Model_Job_Position": "B"
        })

positions = blind.apply(
    assign_jobs,
    axis=1
)

blind = pd.concat(
    [
        blind.drop(
            columns=[
                "Predicted_Job_Text",
                "Best_Label_Job_Text"
            ]
        ),
        positions
    ],
    axis=1
)

# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

BLIND_PATH = os.path.join(
    BASE,
    "v3_blind_re_review_127_suspicious_cases.csv"
)

blind.to_csv(
    BLIND_PATH,
    index=False
)

print("\n" + "=" * 75)
print("BLIND RE-REVIEW DATASET CREATED")
print("=" * 75)

print("Cases:", len(blind))
print("Columns:", len(blind.columns))
print("\nSaved:")
print(BLIND_PATH)

display(
    blind[
        [
            "Review_ID",
            "Trial",
            "Resume_Index",
            "Job_A",
            "Job_B",
            "Audit_Category"
        ]
    ].head(10)
)

print(
    "\n✅ Original labels are NOT included in the blind review file."
)

In [ ]:
# ============================================================
# TRUE BLIND RE-REVIEW DATASET
# REMOVE ALL INFORMATION THAT CAN REVEAL THE ORIGINAL LABEL
# ============================================================

import os
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

AUDIT_PATH = os.path.join(
    BASE,
    "v3_final_452_label_quality_audit.csv"
)

RESUME_TEXT_PATH = os.path.join(
    BASE,
    "v3_resume_texts.pkl"
)

JOB_TEXT_PATH = os.path.join(
    BASE,
    "v3_job_texts.pkl"
)

audit = pd.read_csv(AUDIT_PATH)

# ------------------------------------------------------------
# 1. ONLY suspicious cases
# ------------------------------------------------------------

sus = audit[
    audit["Audit_Category"].isin([
        "STRONG_LABEL_NOISE",
        "POSSIBLE_LABEL_NOISE"
    ])
].copy()

# ------------------------------------------------------------
# 2. LOAD TEXTS
# ------------------------------------------------------------

import pickle

with open(RESUME_TEXT_PATH, "rb") as f:
    resume_texts = pickle.load(f)

with open(JOB_TEXT_PATH, "rb") as f:
    job_texts = pickle.load(f)

# ------------------------------------------------------------
# 3. ADD TEXT
# ------------------------------------------------------------

sus["Resume_Text"] = sus["Resume_Index"].map(
    lambda x: resume_texts[int(x)]
)

sus["Original_Job_Text"] = sus["Job_Index"].map(
    lambda x: job_texts[int(x)]
)

sus["BestLabel_Job_Text"] = sus["Best_Label_Job_Index"].map(
    lambda x: job_texts[int(x)]
)

# ------------------------------------------------------------
# 4. RANDOMLY ASSIGN A/B
# ------------------------------------------------------------

rng = np.random.default_rng(20261008)

swap = rng.integers(
    0,
    2,
    size=len(sus)
)

rows = []

for i, (_, row) in enumerate(sus.iterrows()):

    if swap[i] == 0:
        job_a = row["Original_Job_Text"]
        job_b = row["BestLabel_Job_Text"]

    else:
        job_a = row["BestLabel_Job_Text"]
        job_b = row["Original_Job_Text"]

    rows.append({
        "Review_ID": i + 1,
        "Resume_Text": row["Resume_Text"],
        "Job_A": job_a,
        "Job_B": job_b
    })

blind_true = pd.DataFrame(rows)

# ------------------------------------------------------------
# 5. SANITY CHECK
# ------------------------------------------------------------

print("=" * 75)
print("TRUE BLIND REVIEW DATASET")
print("=" * 75)

print("Rows:", len(blind_true))
print("Columns:", list(blind_true.columns))

# These MUST NOT exist
for forbidden in [
    "Label",
    "Best_Label",
    "Audit_Category",
    "Trial",
    "Resume_Index",
    "Job_Index",
    "Original_Job",
    "BestLabel_Job",
    "Model_Score_Gap",
    "Features_Favoring_Top1"
]:
    assert forbidden not in blind_true.columns, \
        f"LEAKAGE DETECTED: {forbidden}"

print("\n✅ No original labels")
print("✅ No audit category")
print("✅ No feature-gap information")
print("✅ No model score")
print("✅ No job IDs")
print("✅ Only Resume + Job A + Job B")

# ------------------------------------------------------------
# 6. SAVE
# ------------------------------------------------------------

TRUE_BLIND_PATH = os.path.join(
    BASE,
    "v3_TRUE_BLIND_REVIEW_127.csv"
)

blind_true.to_csv(
    TRUE_BLIND_PATH,
    index=False
)

print("\nSaved:")
print(TRUE_BLIND_PATH)

display(
    blind_true.head(5)
)

print("\n🔥 TRUE BLIND DATASET READY")

In [ ]:
# ============================================================
# V3 — BLIND REVIEW RESULTS + LABEL CONSISTENCY ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

AUDIT_PATH = os.path.join(
    BASE,
    "v3_final_452_label_quality_audit.csv"
)

audit = pd.read_csv(AUDIT_PATH)

# ------------------------------------------------------------
# 1. BLIND REVIEW RESPONSES
#
# Exact 127 responses supplied by the independent reviewer.
# M = MEDIUM, L = LOW, H = HIGH
# ------------------------------------------------------------

decision_string = (
    "AAABABBBBAABTABAAAABABABBAAABBBBABBAAAABBABBBAAABBBABABBABBBBBB"
    "ABBABAABBBTBBBABABABBAABBBBABBTBBABABTBBBAAAABAABBTATBABABAABABB"
)

confidence_string = (
    "MMLMMMHMHLLMLMHLLMMLMMLLMMLLLHLMHMLMMLLLLMLMLLMLLHHLMLMHLMLLLH"
    "HLHHMLLLLLLLLMLLHLLMMMHLHMHLHLLLLLLLLLLHLLLHLLLLLHLLLLMLLHLLLLLLL"
)

assert len(decision_string) == 127
assert len(confidence_string) == 127

# ------------------------------------------------------------
# 2. LOAD THE SAME SUSPICIOUS CASE ORDER USED TO CREATE
#    THE BLIND FILE
# ------------------------------------------------------------

suspicious = audit[
    audit["Audit_Category"].isin([
        "STRONG_LABEL_NOISE",
        "POSSIBLE_LABEL_NOISE"
    ])
].copy().reset_index(drop=True)

assert len(suspicious) == 127

review = pd.DataFrame({
    "Review_ID": np.arange(1, 128),
    "Decision": list(decision_string),
    "Confidence": list(confidence_string)
})

# ------------------------------------------------------------
# 3. RECREATE THE EXACT A/B SHUFFLE
# ------------------------------------------------------------

rng = np.random.default_rng(20261008)

swap = rng.integers(
    0,
    2,
    size=127
)

# swap=0:
#   Job A = ExtraTrees job
#   Job B = original label-2 job
#
# swap=1:
#   Job A = original label-2 job
#   Job B = ExtraTrees job

review["Swap"] = swap

review["ExtraTrees_Position"] = np.where(
    swap == 0,
    "A",
    "B"
)

review["Original_Label2_Position"] = np.where(
    swap == 0,
    "B",
    "A"
)

# ------------------------------------------------------------
# 4. INTERPRET REVIEW DECISION
# ------------------------------------------------------------

def interpret(row):

    if row["Decision"] == "T":
        return "TIE"

    if row["Decision"] == row["ExtraTrees_Position"]:
        return "REVIEW_FAVORS_EXTRATREES"

    return "REVIEW_FAVORS_ORIGINAL_LABEL2"


review["Review_Outcome"] = review.apply(
    interpret,
    axis=1
)

# ------------------------------------------------------------
# 5. MERGE WITH ORIGINAL AUDIT
# ------------------------------------------------------------

reviewed = pd.concat(
    [
        suspicious.reset_index(drop=True),
        review.reset_index(drop=True)
    ],
    axis=1
)

# ------------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------------

print("=" * 80)
print("BLIND RE-REVIEW SUMMARY")
print("=" * 80)

print("\nOverall:")
print(
    reviewed["Review_Outcome"]
    .value_counts()
)

print("\nConfidence:")
print(
    reviewed["Confidence"]
    .value_counts()
)

print("\nOutcome × Confidence:")
print(
    pd.crosstab(
        reviewed["Review_Outcome"],
        reviewed["Confidence"]
    )
)

# ------------------------------------------------------------
# 7. NON-TIE AGREEMENT
# ------------------------------------------------------------

non_ties = reviewed[
    reviewed["Review_Outcome"] != "TIE"
]

favors_model = (
    non_ties["Review_Outcome"]
    == "REVIEW_FAVORS_EXTRATREES"
).sum()

favors_label = (
    non_ties["Review_Outcome"]
    == "REVIEW_FAVORS_ORIGINAL_LABEL2"
).sum()

print("\n" + "=" * 80)
print("NON-TIE RESULTS")
print("=" * 80)

print("Non-ties:", len(non_ties))
print("Reviewer favors ExtraTrees:", favors_model)
print("Reviewer favors original label-2:", favors_label)

print(
    "ExtraTrees share among non-ties:",
    round(
        favors_model / len(non_ties) * 100,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# 8. HIGH-CONFIDENCE ONLY
# ------------------------------------------------------------

high = reviewed[
    reviewed["Confidence"] == "H"
]

high_non_tie = high[
    high["Review_Outcome"] != "TIE"
]

print("\n" + "=" * 80)
print("HIGH-CONFIDENCE RESULTS")
print("=" * 80)

print("High-confidence reviews:", len(high_non_tie))

print(
    high_non_tie["Review_Outcome"]
    .value_counts()
)

if len(high_non_tie) > 0:

    model_high = (
        high_non_tie["Review_Outcome"]
        == "REVIEW_FAVORS_EXTRATREES"
    ).sum()

    print(
        "ExtraTrees share:",
        round(
            model_high
            / len(high_non_tie)
            * 100,
            2
        ),
        "%"
    )

# ------------------------------------------------------------
# 9. HIGH + MEDIUM CONFIDENCE
# ------------------------------------------------------------

hm = reviewed[
    reviewed["Confidence"].isin(["H", "M"])
]

hm_non_tie = hm[
    hm["Review_Outcome"] != "TIE"
]

print("\n" + "=" * 80)
print("HIGH + MEDIUM CONFIDENCE RESULTS")
print("=" * 80)

print("Cases:", len(hm_non_tie))

print(
    hm_non_tie["Review_Outcome"]
    .value_counts()
)

if len(hm_non_tie) > 0:

    model_hm = (
        hm_non_tie["Review_Outcome"]
        == "REVIEW_FAVORS_EXTRATREES"
    ).sum()

    print(
        "ExtraTrees share:",
        round(
            model_hm
            / len(hm_non_tie)
            * 100,
            2
        ),
        "%"
    )

# ------------------------------------------------------------
# 10. COMPARE WITH ORIGINAL AUDIT CATEGORY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AUDIT CATEGORY × BLIND REVIEW")
print("=" * 80)

category_table = pd.crosstab(
    reviewed["Audit_Category"],
    reviewed["Review_Outcome"]
)

print(category_table)

# ------------------------------------------------------------
# 11. STRONG LABEL-NOISE CASES
# ------------------------------------------------------------

strong = reviewed[
    reviewed["Audit_Category"]
    == "STRONG_LABEL_NOISE"
]

print("\n" + "=" * 80)
print("STRONG LABEL-NOISE CASES")
print("=" * 80)

print(
    strong["Review_Outcome"]
    .value_counts()
)

# ------------------------------------------------------------
# 12. CASES WHERE HIGH-CONFIDENCE REVIEW REJECTS ORIGINAL
# ------------------------------------------------------------

high_model_favored = reviewed[
    (reviewed["Confidence"] == "H") &
    (
        reviewed["Review_Outcome"]
        == "REVIEW_FAVORS_EXTRATREES"
    )
].copy()

print("\n" + "=" * 80)
print("HIGH-CONFIDENCE CASES FAVORING EXTRATREES")
print("=" * 80)

display(
    high_model_favored[
        [
            "Review_ID",
            "Trial",
            "Resume_Index",
            "Job_Index",
            "Label",
            "Best_Label_Job_Index",
            "Best_Label",
            "Audit_Category",
            "Decision",
            "Confidence",
            "Best_Label_Rank_Computed",
            "Features_Favoring_Top1",
            "Features_Favoring_BestLabel",
            "Model_Score_Gap"
        ]
    ]
)

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

RESULT_PATH = os.path.join(
    BASE,
    "v3_blind_review_127_results.csv"
)

SUMMARY_PATH = os.path.join(
    BASE,
    "v3_blind_review_127_summary.csv"
)

reviewed.to_csv(
    RESULT_PATH,
    index=False
)

summary_rows = []

for outcome, count in (
    reviewed["Review_Outcome"]
    .value_counts()
    .items()
):

    summary_rows.append({
        "Measure": "Review Outcome",
        "Category": outcome,
        "Count": count,
        "Percentage": count / len(reviewed) * 100
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

print("\n" + "=" * 80)
print("SAVED")
print("=" * 80)

print(RESULT_PATH)
print(SUMMARY_PATH)

print("\n🔥 BLIND REVIEW ANALYSIS COMPLETE")

In [ ]:
# ============================================================
# V3 — LABEL-NOISE SENSITIVITY ANALYSIS
#
# IMPORTANT:
# This DOES NOT modify the real training data or final model.
#
# It creates hypothetical evaluation labels using the
# INDEPENDENT blind review and measures how benchmark metrics
# would change.
# ============================================================

import os
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

PAIR_PATH = os.path.join(
    BASE,
    "v3_error_analysis_pair_level_25400.csv"
)

REVIEW_PATH = os.path.join(
    BASE,
    "v3_blind_review_127_results.csv"
)

pair = pd.read_csv(PAIR_PATH)
reviewed = pd.read_csv(REVIEW_PATH)

print("=" * 80)
print("V3 LABEL-NOISE SENSITIVITY ANALYSIS")
print("=" * 80)

print("Evaluation rows:", len(pair))
print("Blind-review rows:", len(reviewed))

assert len(pair) == 25400
assert len(reviewed) == 127


# ============================================================
# 1. IDENTIFY WHICH JOB IS EXTRATREES AND WHICH IS LABEL-2
# ============================================================

# In the reviewed file:
#
# ExtraTrees_Position = A/B
# Original_Label2_Position = A/B
#
# We recover the actual Job_Index using:
#   Job_Index
#   Best_Label_Job_Index

reviewed = reviewed.copy()

# ============================================================
# 2. BUILD A CORRECTION TABLE
# ============================================================

corrections = []

for _, row in reviewed.iterrows():

    decision = row["Decision"]
    confidence = row["Confidence"]

    # A / B / TIE
    # We only care when reviewer selected one job.

    if decision == "T":
        continue

    # Determine whether reviewer selected ExtraTrees' job
    if decision == row["ExtraTrees_Position"]:
        outcome = "EXTRATREES"
    else:
        outcome = "LABEL2"

    # Original jobs
    et_job = int(row["Job_Index"])
    label2_job = int(row["Best_Label_Job_Index"])

    corrections.append({
        "Review_ID": int(row["Review_ID"]),
        "Trial": int(row["Trial"]),
        "Resume_Index": int(row["Resume_Index"]),
        "ExtraTrees_Job_Index": et_job,
        "Original_Label2_Job_Index": label2_job,
        "Review_Outcome": outcome,
        "Confidence": confidence
    })

corrections = pd.DataFrame(corrections)

print("\nNon-tie reviewed cases:",
      len(corrections))


# ============================================================
# 3. FUNCTION TO APPLY HYPOTHETICAL LABEL CORRECTIONS
# ============================================================

def apply_corrections(
    pair_df,
    corrections_df,
    allowed_confidence
):

    result = pair_df.copy()

    selected = corrections_df[
        corrections_df["Confidence"].isin(
            allowed_confidence
        )
        &
        (
            corrections_df["Review_Outcome"]
            == "EXTRATREES"
        )
    ].copy()

    print(
        f"\nCases corrected "
        f"({'+'.join(allowed_confidence)}):",
        len(selected)
    )

    # --------------------------------------------------------
    # Swap the labels between ExtraTrees job and original
    # label-2 job.
    #
    # This preserves the original label distribution within
    # each 10-job evaluation set.
    # --------------------------------------------------------

    for _, row in selected.iterrows():

        t = row["Trial"]
        r = row["Resume_Index"]

        et_job = row["ExtraTrees_Job_Index"]
        label2_job = row["Original_Label2_Job_Index"]

        mask_et = (
            (result["Trial"] == t)
            &
            (result["Resume_Index"] == r)
            &
            (result["Job_Index"] == et_job)
        )

        mask_l2 = (
            (result["Trial"] == t)
            &
            (result["Resume_Index"] == r)
            &
            (result["Job_Index"] == label2_job)
        )

        if mask_et.sum() != 1:
            raise ValueError(
                f"Expected exactly 1 ExtraTrees row "
                f"for Trial={t}, Resume={r}, Job={et_job}"
            )

        if mask_l2.sum() != 1:
            raise ValueError(
                f"Expected exactly 1 label-2 row "
                f"for Trial={t}, Resume={r}, Job={label2_job}"
            )

        original_et_label = result.loc[
            mask_et,
            "Label"
        ].iloc[0]

        original_l2_label = result.loc[
            mask_l2,
            "Label"
        ].iloc[0]

        # Should normally be 0/1 versus 2
        # because this came from an original Top-1 error.
        result.loc[
            mask_et,
            "Label"
        ] = original_l2_label

        result.loc[
            mask_l2,
            "Label"
        ] = original_et_label

    return result, selected


# ============================================================
# 4. METRIC FUNCTIONS
# ============================================================

def top1_metric(g):

    g = g.sort_values(
        "ExtraTrees_Tuned_Score",
        ascending=False
    )

    # Highest label available in this 10-job set
    best_label = g["Label"].max()

    return int(g.iloc[0]["Label"] == best_label)


def hit_at_k(g, k):

    g = g.sort_values(
        "ExtraTrees_Tuned_Score",
        ascending=False
    )

    best_label = g["Label"].max()

    return int(
        (g.head(k)["Label"] == best_label).any()
    )


def ndcg_metric(g):

    g = g.sort_values(
        "ExtraTrees_Tuned_Score",
        ascending=False
    )

    labels = g["Label"].to_numpy(
        dtype=float
    )

    gains = (
        2 ** labels
        - 1
    )

    discounts = np.log2(
        np.arange(2, len(labels) + 2)
    )

    dcg = np.sum(
        gains / discounts
    )

    ideal_labels = np.sort(
        labels
    )[::-1]

    ideal_gains = (
        2 ** ideal_labels
        - 1
    )

    idcg = np.sum(
        ideal_gains / discounts
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg


def pairwise_metric(g):

    g = g.sort_values(
        "ExtraTrees_Tuned_Score",
        ascending=False
    )

    labels = g["Label"].to_numpy()

    total = 0
    correct = 0

    for i in range(len(labels)):

        for j in range(i + 1, len(labels)):

            if labels[i] == labels[j]:
                continue

            total += 1

            if labels[i] > labels[j]:
                correct += 1

    if total == 0:
        return 0.0

    return correct / total


def evaluate(df):

    grouped = df.groupby(
        ["Trial", "Resume_Index"]
    )

    metrics = {
        "Top1": grouped.apply(
            top1_metric
        ).mean(),

        "Hit@3": grouped.apply(
            lambda g: hit_at_k(g, 3)
        ).mean(),

        "Hit@5": grouped.apply(
            lambda g: hit_at_k(g, 5)
        ).mean(),

        "NDCG": grouped.apply(
            ndcg_metric
        ).mean(),

        "Pairwise": grouped.apply(
            pairwise_metric
        ).mean()
    }

    return metrics


# ============================================================
# 5. ORIGINAL
# ============================================================

original_metrics = evaluate(pair)

print("\n" + "=" * 80)
print("ORIGINAL BENCHMARK")
print("=" * 80)

for k, v in original_metrics.items():
    print(f"{k:10s}: {v:.6f}")


# ============================================================
# 6. CONSERVATIVE SCENARIO
#    HIGH confidence only
# ============================================================

high_pair, high_corrections = apply_corrections(
    pair,
    corrections,
    ["H"]
)

high_metrics = evaluate(
    high_pair
)

print("\n" + "=" * 80)
print("SCENARIO A — HIGH CONFIDENCE ONLY")
print("=" * 80)

for k, v in high_metrics.items():
    print(f"{k:10s}: {v:.6f}")


# ============================================================
# 7. BROADER SCENARIO
#    HIGH + MEDIUM
# ============================================================

hm_pair, hm_corrections = apply_corrections(
    pair,
    corrections,
    ["H", "M"]
)

hm_metrics = evaluate(
    hm_pair
)

print("\n" + "=" * 80)
print("SCENARIO B — HIGH + MEDIUM CONFIDENCE")
print("=" * 80)

for k, v in hm_metrics.items():
    print(f"{k:10s}: {v:.6f}")


# ============================================================
# 8. IMPROVEMENT TABLE
# ============================================================

rows = []

for metric in original_metrics:

    original = original_metrics[metric]
    high = high_metrics[metric]
    hm = hm_metrics[metric]

    rows.append({
        "Metric": metric,

        "Original": original,

        "High_Only": high,
        "High_Only_Change": high - original,

        "High_Medium": hm,
        "High_Medium_Change": hm - original
    })

comparison = pd.DataFrame(rows)

print("\n" + "=" * 80)
print("LABEL-NOISE SENSITIVITY")
print("=" * 80)

display(
    comparison.round(6)
)


# ============================================================
# 9. WHICH CASES WERE CORRECTED?
# ============================================================

print("\n" + "=" * 80)
print("CORRECTION COUNTS")
print("=" * 80)

print(
    "HIGH confidence corrections:",
    len(high_corrections)
)

print(
    "HIGH+MEDIUM corrections:",
    len(hm_corrections)
)


# ============================================================
# 10. IMPORTANT WARNING
# ============================================================

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)

print("""
These are sensitivity analyses, NOT new unbiased benchmark scores.

The 127 cases were selected because they were suspicious
model-error cases.

Therefore:

- Original metrics remain the primary reported benchmark.
- High-only / High+Medium results show how sensitive the
  benchmark is to plausible label corrections.
- These corrected labels must NOT be added to model training.
- Do not claim the corrected metrics as the official final score.
""")


# ============================================================
# 11. SAVE
# ============================================================

comparison_path = os.path.join(
    BASE,
    "v3_label_noise_sensitivity_metrics.csv"
)

high_pair_path = os.path.join(
    BASE,
    "v3_eval_labels_high_confidence_corrected.csv"
)

hm_pair_path = os.path.join(
    BASE,
    "v3_eval_labels_high_medium_corrected.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

high_pair.to_csv(
    high_pair_path,
    index=False
)

hm_pair.to_csv(
    hm_pair_path,
    index=False
)

print("\nSaved:")
print(comparison_path)
print(high_pair_path)
print(hm_pair_path)

print("\n🔥 LABEL-NOISE SENSITIVITY COMPLETE")

In [ ]:
# ============================================================
# V3 — GENUINE MODEL ERROR ANALYSIS
# ============================================================
#
# Goal:
#   Understand WHY ExtraTrees makes its 452 original Top-1
#   mistakes and identify systematic weaknesses for V4.
#
# IMPORTANT:
#   We use the ORIGINAL 452 error file as the source of truth
#   for "wrong Top-1".
#
#   This analysis does NOT modify labels or the model.
# ============================================================

import os
import re
import pickle
import joblib
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

WRONG_PATH = os.path.join(
    BASE,
    "v3_error_analysis_wrong_top1.csv"
)

PAIR_PATH = os.path.join(
    BASE,
    "v3_error_analysis_pair_level_25400.csv"
)

RESUME_TEXT_PATH = os.path.join(
    BASE,
    "v3_resume_texts.pkl"
)

JOB_TEXT_PATH = os.path.join(
    BASE,
    "v3_job_texts.pkl"
)

MODEL_PATH = os.path.join(
    BASE,
    "v3_extratrees_tuned_final.joblib"
)

# ============================================================
# 1. LOAD DATA
# ============================================================

wrong = pd.read_csv(WRONG_PATH)
pair = pd.read_csv(PAIR_PATH)

with open(RESUME_TEXT_PATH, "rb") as f:
    resume_texts = pickle.load(f)

with open(JOB_TEXT_PATH, "rb") as f:
    job_texts = pickle.load(f)

print("=" * 80)
print("V3 GENUINE ERROR ANALYSIS")
print("=" * 80)

print("Wrong cases:", len(wrong))
print("Pair rows:", len(pair))
print("Resumes:", len(resume_texts))
print("Jobs:", len(job_texts))

assert len(wrong) == 452
assert len(pair) == 25400
assert len(resume_texts) == 1540
assert len(job_texts) == 434


# ============================================================
# 2. FEATURE NAMES
# ============================================================

features = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]


# ============================================================
# 3. CREATE RANKING WITHIN EVERY TRIAL/RESUME
# ============================================================

pair["Rank"] = (
    pair
    .groupby(["Trial", "Resume_Index"])
    ["ExtraTrees_Tuned_Score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# ============================================================
# 4. IDENTIFY TOP-1 FOR EVERY CASE
# ============================================================

top1 = pair[
    pair["Rank"] == 1
].copy()

print("\nEvaluation cases:", len(top1))


# ============================================================
# 5. FIND BEST LABEL JOB
#
# Highest available relevance label.
# ============================================================

best_label = (
    pair
    .sort_values(
        [
            "Trial",
            "Resume_Index",
            "Label",
            "ExtraTrees_Tuned_Score"
        ],
        ascending=[
            True,
            True,
            False,
            False
        ]
    )
    .groupby(
        ["Trial", "Resume_Index"],
        as_index=False
    )
    .first()
)

best_label = best_label.rename(
    columns={
        "Job_Index": "Best_Job_Index",
        "Label": "Best_Label",
        "ExtraTrees_Tuned_Score":
            "Best_Job_Score",
        "Rank":
            "Best_Job_Rank"
    }
)


# ============================================================
# 6. CASE-LEVEL DATASET
# ============================================================

case_df = top1.merge(
    best_label[
        [
            "Trial",
            "Resume_Index",
            "Best_Job_Index",
            "Best_Label",
            "Best_Job_Score",
            "Best_Job_Rank"
        ]
    ],
    on=[
        "Trial",
        "Resume_Index"
    ],
    suffixes=(
        "",
        "_best"
    )
)

# Correct according to original benchmark logic
case_df["Correct"] = (
    case_df["Label"] == case_df["Best_Label"]
)

case_df["Error"] = ~case_df["Correct"]

# Distance between model top1 and best-label ranking
case_df["Rank_Gap"] = (
    case_df["Best_Job_Rank"] - 1
)

# Score margin
case_df["Top1_vs_Best_Score_Margin"] = (
    case_df["ExtraTrees_Tuned_Score"]
    - case_df["Best_Job_Score"]
)

print(
    "Correct cases:",
    case_df["Correct"].sum()
)

print(
    "Wrong cases:",
    case_df["Error"].sum()
)


# ============================================================
# 7. ADD FEATURE VALUES FOR BEST-LABEL JOB
# ============================================================

best_feature_cols = []

for f in features:

    best_col = f"{f}_BEST"

    temp = best_label[
        [
            "Trial",
            "Resume_Index",
            f
        ]
    ].rename(
        columns={
            f: best_col
        }
    )

    case_df = case_df.merge(
        temp,
        on=[
            "Trial",
            "Resume_Index"
        ],
        how="left"
    )

    best_feature_cols.append(best_col)


# ============================================================
# 8. FEATURE GAPS
#
# Positive = best-label job has larger feature
# Negative = ExtraTrees Top-1 has larger feature
# ============================================================

for f in features:

    case_df[f"{f}_Gap"] = (
        case_df[f"{f}_BEST"]
        - case_df[f]
    )


# ============================================================
# 9. PARSE JOB TITLES FROM JOB TEXT
# ============================================================

def extract_job_title(text):

    text = str(text)

    m = re.search(
        r"\btitle\s*:\s*(.*?)(?:\s+skills\b|\s+responsibilities\b|\s+keywords\b|$)",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1).strip()

    m = re.search(
        r"\btitle\s+(.*?)(?:\s+skills\b|\s+responsibilities\b|\s+keywords\b|$)",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1).strip()

    return text[:100].strip()


job_titles = {
    i: extract_job_title(
        job_texts[i]
    )
    for i in range(len(job_texts))
}


# ============================================================
# 10. PARSE RESUME ROLE
# ============================================================

def extract_resume_role(text):

    text = str(text)

    # First try experience title
    m = re.search(
        r"\bexperience title\s*:\s*([^:]+?)(?=\s+experience level\b)",
        text,
        flags=re.IGNORECASE
    )

    if m:
        return m.group(1).strip()

    # Fall back to summary
    m = re.search(
        r"\bsummary\s*:\s*(.*?)(?:\s+skills\b|\s+experience title\b|$)",
        text,
        flags=re.IGNORECASE
    )

    if m:
        words = m.group(1).strip().split()
        return " ".join(words[:12])

    return "unknown"


resume_roles = {
    i: extract_resume_role(
        resume_texts[i]
    )
    for i in range(len(resume_texts))
}

case_df["Resume_Role"] = case_df[
    "Resume_Index"
].map(resume_roles)

case_df["Predicted_Job_Title"] = case_df[
    "Job_Index"
].map(job_titles)

case_df["Best_Job_Title"] = case_df[
    "Best_Job_Index"
].map(job_titles)


# ============================================================
# 11. BROAD ROLE-FAMILY CLASSIFIER
#
# HEURISTIC ONLY — used to discover patterns.
# ============================================================

ROLE_PATTERNS = {

    "Data / Analytics": [
        "data analyst",
        "data engineer",
        "bi analyst",
        "business intelligence",
        "data scientist",
        "analytics",
        "database administrator",
        "database engineer",
        "nosql"
    ],

    "Backend / Software": [
        "backend",
        "software engineer",
        "software developer",
        "java developer",
        "python developer",
        "c developer",
        "cpp developer",
        "full stack",
        "developer"
    ],

    "Frontend / Web": [
        "frontend",
        "front end",
        "web developer",
        "ui developer",
        "react developer",
        "angular developer",
        "vue developer"
    ],

    "Mobile": [
        "flutter",
        "mobile developer",
        "android",
        "ios",
        "react native"
    ],

    "Cloud / DevOps": [
        "cloud engineer",
        "cloud architect",
        "devops",
        "site reliability",
        " sre ",
        "kubernetes",
        "platform engineer"
    ],

    "Cybersecurity": [
        "security analyst",
        "cybersecurity",
        "penetration tester",
        "security engineer",
        "infosec",
        "network security",
        "graduate security"
    ],

    "Networking": [
        "network engineer",
        "network administrator",
        "networking"
    ],

    "QA / Testing": [
        "software tester",
        "qa",
        "quality assurance",
        "test engineer",
        "testing"
    ],

    "ML / AI": [
        "machine learning",
        "ml engineer",
        "ai engineer",
        "artificial intelligence",
        "deep learning"
    ],

    "Other": []
}


def role_family(title):

    t = " " + str(title).lower() + " "

    for family, patterns in ROLE_PATTERNS.items():

        for p in patterns:

            if p.strip() in t:
                return family

    return "Other"


case_df["Predicted_Family"] = (
    case_df["Predicted_Job_Title"]
    .map(role_family)
)

case_df["Best_Family"] = (
    case_df["Best_Job_Title"]
    .map(role_family)
)


# ============================================================
# 12. ERROR-ONLY DATASET
# ============================================================

errors = case_df[
    case_df["Error"]
].copy()

print("\nActual errors:", len(errors))


# ============================================================
# 13. ERROR DIFFICULTY
# ============================================================

def margin_bucket(x):

    x = abs(x)

    if x < 0.05:
        return "<0.05"

    if x < 0.10:
        return "0.05–0.10"

    if x < 0.25:
        return "0.10–0.25"

    if x < 0.50:
        return "0.25–0.50"

    return ">=0.50"


errors["Margin_Bucket"] = (
    errors["Top1_vs_Best_Score_Margin"]
    .abs()
    .map(margin_bucket)
)

print("\n" + "=" * 80)
print("ERROR DIFFICULTY — SCORE MARGIN")
print("=" * 80)

margin_summary = (
    errors["Margin_Bucket"]
    .value_counts()
    .reindex(
        [
            "<0.05",
            "0.05–0.10",
            "0.10–0.25",
            "0.25–0.50",
            ">=0.50"
        ]
    )
    .fillna(0)
    .astype(int)
    .rename_axis("Margin_Bucket")
    .reset_index(name="Count")
)

margin_summary["Percentage"] = (
    margin_summary["Count"]
    / len(errors)
    * 100
)

print(
    margin_summary.to_string(index=False)
)


# ============================================================
# 14. BEST-LABEL RANK BUCKET
# ============================================================

def rank_bucket(rank):

    if rank <= 3:
        return "2–3"

    if rank <= 5:
        return "4–5"

    return "6–10"


errors["Best_Label_Rank_Bucket"] = (
    errors["Best_Job_Rank"]
    .map(rank_bucket)
)

print("\n" + "=" * 80)
print("ERROR DIFFICULTY — BEST-LABEL RANK")
print("=" * 80)

print(
    errors["Best_Label_Rank_Bucket"]
    .value_counts()
    .reindex(["2–3", "4–5", "6–10"])
    .fillna(0)
    .astype(int)
)


# ============================================================
# 15. CORRECT VS WRONG FEATURE PROFILE
# ============================================================

profile_cols = features

profile = (
    case_df
    .groupby("Correct")[profile_cols]
    .mean()
    .T
)

profile.columns = [
    "Wrong",
    "Correct"
] if list(profile.columns) == [False, True] else profile.columns

print("\n" + "=" * 80)
print("CORRECT VS WRONG TOP-1 FEATURE PROFILE")
print("=" * 80)

display(
    profile.round(5)
)


# ============================================================
# 16. FEATURE WIN RATE AMONG ERRORS
# ============================================================

print("\n" + "=" * 80)
print("WHICH FEATURES FAVOR THE WRONG TOP-1?")
print("=" * 80)

feature_win_rows = []

for f in features:

    gap = errors[f"{f}_Gap"]

    feature_win_rows.append({
        "Feature": f,
        "Top1_Wins": int(
            (gap < 0).sum()
        ),
        "BestLabel_Wins": int(
            (gap > 0).sum()
        ),
        "Ties": int(
            (gap.abs() < 1e-8).sum()
        )
    })

feature_wins = pd.DataFrame(
    feature_win_rows
)

feature_wins["Top1_Win_%"] = (
    feature_wins["Top1_Wins"]
    / len(errors)
    * 100
)

feature_wins["BestLabel_Win_%"] = (
    feature_wins["BestLabel_Wins"]
    / len(errors)
    * 100
)

display(
    feature_wins.sort_values(
        "Top1_Win_%",
        ascending=False
    ).round(2)
)


# ============================================================
# 17. TARGET ROLE FAMILY ERROR RATES
#
# For every evaluation case:
#   Best_Family = family of the best available job
#
# Then calculate Top-1 success/failure per family.
# ============================================================

family_stats = (
    case_df
    .groupby("Best_Family")
    .agg(
        Cases=("Correct", "size"),
        Correct=("Correct", "sum")
    )
)

family_stats["Errors"] = (
    family_stats["Cases"]
    - family_stats["Correct"]
)

family_stats["Error_Rate"] = (
    family_stats["Errors"]
    / family_stats["Cases"]
)

family_stats["Accuracy"] = (
    family_stats["Correct"]
    / family_stats["Cases"]
)

family_stats = family_stats.sort_values(
    "Cases",
    ascending=False
)

print("\n" + "=" * 80)
print("ERROR RATE BY TARGET ROLE FAMILY")
print("=" * 80)

display(
    family_stats.round(4)
)


# ============================================================
# 18. ROLE-FAMILY CONFUSION — ERRORS ONLY
# ============================================================

confusion = pd.crosstab(
    errors["Best_Family"],
    errors["Predicted_Family"]
)

print("\n" + "=" * 80)
print("ROLE-FAMILY CONFUSION — 452 ERRORS")
print("=" * 80)

display(confusion)


# ============================================================
# 19. MOST COMMON CONFUSIONS
# ============================================================

confusion_pairs = (
    errors
    .groupby(
        [
            "Best_Family",
            "Predicted_Family"
        ]
    )
    .size()
    .reset_index(
        name="Count"
    )
    .sort_values(
        "Count",
        ascending=False
    )
)

confusion_pairs["Percentage_of_Errors"] = (
    confusion_pairs["Count"]
    / len(errors)
    * 100
)

print("\n" + "=" * 80)
print("TOP ROLE-FAMILY CONFUSIONS")
print("=" * 80)

display(
    confusion_pairs.head(20).round(2)
)


# ============================================================
# 20. MOST COMMON WRONG PREDICTION TITLES
# ============================================================

wrong_pred_titles = (
    errors["Predicted_Job_Title"]
    .value_counts()
    .rename_axis("Predicted_Job_Title")
    .reset_index(name="Error_Count")
)

wrong_pred_titles["Percentage_of_Errors"] = (
    wrong_pred_titles["Error_Count"]
    / len(errors)
    * 100
)

print("\n" + "=" * 80)
print("MOST COMMON WRONG PREDICTED JOB TITLES")
print("=" * 80)

display(
    wrong_pred_titles.head(20)
)


# ============================================================
# 21. MOST COMMON MISSED BEST-LABEL TITLES
# ============================================================

missed_titles = (
    errors["Best_Job_Title"]
    .value_counts()
    .rename_axis("Best_Job_Title")
    .reset_index(name="Miss_Count")
)

missed_titles["Percentage_of_Errors"] = (
    missed_titles["Miss_Count"]
    / len(errors)
    * 100
)

print("\n" + "=" * 80)
print("MOST COMMON MISSED BEST-LABEL JOB TITLES")
print("=" * 80)

display(
    missed_titles.head(20)
)


# ============================================================
# 22. HIGH-SEMANTIC BUT WRONG
#
# Cases where semantic similarity is relatively high,
# but model still misses the best-labelled job.
# ============================================================

semantic_threshold = errors[
    "Semantic_Similarity"
].quantile(0.75)

high_semantic_errors = errors[
    errors["Semantic_Similarity"]
    >= semantic_threshold
].copy()

high_semantic_errors = (
    high_semantic_errors
    .sort_values(
        "Semantic_Similarity",
        ascending=False
    )
)

print("\n" + "=" * 80)
print(
    "HIGH-SEMANTIC-SIMILARITY ERRORS"
)
print("=" * 80)

print(
    "75th percentile threshold:",
    round(semantic_threshold, 4)
)

display(
    high_semantic_errors[
        [
            "Trial",
            "Resume_Index",
            "Predicted_Job_Title",
            "Best_Job_Title",
            "Label",
            "Best_Label",
            "ExtraTrees_Tuned_Score",
            "Best_Job_Score",
            "Semantic_Similarity",
            "Role_Similarity",
            "Skill_Score",
            "Normalized_Cosine",
            "Best_Job_Rank"
        ]
    ].head(30)
)


# ============================================================
# 23. LOW-EVIDENCE ERRORS
#
# Cases where both explicit skills and semantic similarity
# are weak.
# ============================================================

low_evidence = errors[
    (
        errors["Skill_Score"] <=
        errors["Skill_Score"].quantile(0.25)
    )
    &
    (
        errors["Semantic_Similarity"] <=
        errors["Semantic_Similarity"].quantile(0.25)
    )
].copy()

print("\n" + "=" * 80)
print("LOW-EVIDENCE ERRORS")
print("=" * 80)

print(
    "Cases:",
    len(low_evidence)
)

display(
    low_evidence[
        [
            "Trial",
            "Resume_Index",
            "Predicted_Job_Title",
            "Best_Job_Title",
            "Skill_Score",
            "Semantic_Similarity",
            "Role_Similarity",
            "Cosine",
            "Normalized_Cosine",
            "Best_Job_Rank"
        ]
    ].head(30)
)


# ============================================================
# 24. EXTRA TREES FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 80)
print("EXTRATREES FEATURE IMPORTANCE")
print("=" * 80)

try:

    model = joblib.load(
        MODEL_PATH
    )

    # Handle possible pipeline/object wrappers
    if hasattr(model, "feature_importances_"):

        importances = model.feature_importances_

    elif hasattr(
        model,
        "named_steps"
    ):

        final_estimator = (
            list(
                model.named_steps.values()
            )[-1]
        )

        importances = (
            final_estimator
            .feature_importances_
        )

    else:
        importances = None

    if importances is not None:

        importance_df = pd.DataFrame({
            "Feature": features,
            "Importance": importances
        }).sort_values(
            "Importance",
            ascending=False
        )

        print(
            importance_df.to_string(
                index=False
            )
        )

    else:

        print(
            "Could not extract feature_importances_."
        )

except Exception as e:

    importance_df = pd.DataFrame()

    print(
        "Could not load ExtraTrees model:",
        repr(e)
    )


# ============================================================
# 25. SYSTEMATIC ERROR FLAGS
# ============================================================

errors["High_Semantic"] = (
    errors["Semantic_Similarity"]
    >= errors["Semantic_Similarity"].quantile(0.75)
)

errors["High_Skill"] = (
    errors["Skill_Score"]
    >= errors["Skill_Score"].quantile(0.75)
)

errors["Low_Skill"] = (
    errors["Skill_Score"]
    <= errors["Skill_Score"].quantile(0.25)
)

errors["Low_Semantic"] = (
    errors["Semantic_Similarity"]
    <= errors["Semantic_Similarity"].quantile(0.25)
)

errors["Close_Margin"] = (
    errors["Top1_vs_Best_Score_Margin"].abs()
    < 0.05
)


print("\n" + "=" * 80)
print("SYSTEMATIC ERROR FLAGS")
print("=" * 80)

flags = pd.DataFrame({
    "Pattern": [
        "Close score margin (<0.05)",
        "High semantic similarity",
        "High skill score",
        "Low skill score",
        "Low semantic similarity"
    ],
    "Count": [
        errors["Close_Margin"].sum(),
        errors["High_Semantic"].sum(),
        errors["High_Skill"].sum(),
        errors["Low_Skill"].sum(),
        errors["Low_Semantic"].sum()
    ]
})

flags["Percentage"] = (
    flags["Count"]
    / len(errors)
    * 100
)

display(
    flags.round(2)
)


# ============================================================
# 26. SAVE CASE-LEVEL ERROR ANALYSIS
# ============================================================

CASE_PATH = os.path.join(
    BASE,
    "v3_genuine_error_analysis_case_level.csv"
)

FAMILY_PATH = os.path.join(
    BASE,
    "v3_genuine_error_analysis_role_family_stats.csv"
)

CONFUSION_PATH = os.path.join(
    BASE,
    "v3_genuine_error_analysis_role_confusion.csv"
)

FEATURE_PATH = os.path.join(
    BASE,
    "v3_genuine_error_analysis_feature_wins.csv"
)

MARGIN_PATH = os.path.join(
    BASE,
    "v3_genuine_error_analysis_margin_buckets.csv"
)

case_df.to_csv(
    CASE_PATH,
    index=False
)

family_stats.to_csv(
    FAMILY_PATH
)

confusion_pairs.to_csv(
    CONFUSION_PATH,
    index=False
)

feature_wins.to_csv(
    FEATURE_PATH,
    index=False
)

margin_summary.to_csv(
    MARGIN_PATH,
    index=False
)

print("\n" + "=" * 80)
print("SAVED")
print("=" * 80)

print(CASE_PATH)
print(FAMILY_PATH)
print(CONFUSION_PATH)
print(FEATURE_PATH)
print(MARGIN_PATH)

print("\n🔥 GENUINE MODEL ERROR ANALYSIS COMPLETE")

In [ ]:
# ============================================================
# V3 ENHANCEMENT — CELL 1
# LOAD EXISTING ARTIFACTS + VERIFY
# ============================================================

import os
import gc
import re
import json
import joblib
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Resume_Model_Project"
V3_DIR = os.path.join(PROJECT_DIR, "V3")
V3_OUT = os.path.join(V3_DIR, "Outputs")

V2_DIR = os.path.join(PROJECT_DIR, "V2_100k")
V2_INPUT = os.path.join(V2_DIR, "Inputs")
V2_OUT = os.path.join(V2_DIR, "Outputs")

os.makedirs(V3_OUT, exist_ok=True)

# ------------------------------------------------------------
# Existing source data
# ------------------------------------------------------------

candidate_df = pd.read_pickle(
    os.path.join(V2_INPUT, "candidate_df.pkl")
)

entry_job_df = pd.read_pickle(
    os.path.join(V2_INPUT, "entry_job_df.pkl")
)

match_vectorizer = joblib.load(
    os.path.join(V2_INPUT, "tfidf_vectorizer.pkl")
)

# ------------------------------------------------------------
# Existing V3 feature dataset
# ------------------------------------------------------------

v3_path = os.path.join(
    V3_OUT,
    "v3_supervised_dataset_568210_rich6_semantic.pkl"
)

v3_df = pd.read_pickle(v3_path)

# ------------------------------------------------------------
# Existing matrices
# ------------------------------------------------------------

role_similarity_matrix = np.load(
    os.path.join(V2_OUT, "role_similarity_matrix.npy")
)

semantic_similarity_matrix = np.load(
    os.path.join(V3_OUT, "v3_semantic_similarity_matrix.npy")
)

# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

print("=" * 70)
print("V3 ENHANCEMENT — ARTIFACT CHECK")
print("=" * 70)

print("Candidates:", len(candidate_df))
print("Jobs:", len(entry_job_df))
print("TF-IDF features:", len(match_vectorizer.get_feature_names_out()))

print("\nCandidate columns:")
print(candidate_df.columns.tolist())

print("\nJob columns:")
print(entry_job_df.columns.tolist())

print("\nV3 dataset shape:", v3_df.shape)
print("V3 columns:")
print(v3_df.columns.tolist())

print("\nRole matrix:", role_similarity_matrix.shape)
print("Semantic matrix:", semantic_similarity_matrix.shape)

assert len(candidate_df) == 1540
assert len(entry_job_df) == 434
assert len(match_vectorizer.get_feature_names_out()) == 44046

assert role_similarity_matrix.shape == (434, 1540)
assert semantic_similarity_matrix.shape == (1540, 434)

assert len(v3_df) == 568210

print("\n✅ All core artifacts loaded successfully.")
print("✅ V3 baseline remains untouched.")

In [ ]:
# ============================================================
# V3 ENHANCEMENT — CELL 2
# EXTRACT CURRENT/LAST EXPERIENCE TITLES
# ============================================================

import re
import pandas as pd
import numpy as np

def extract_experience_titles(text):
    """
    Extract all values following 'experience title:' from the
    structured resume_text representation.
    """
    if not isinstance(text, str):
        return []

    pattern = r"experience title:\s*(.*?)(?=\s+experience level:|\s+responsibilities:|$)"
    matches = re.findall(pattern, text, flags=re.IGNORECASE)

    return [m.strip() for m in matches if m.strip()]


resume_role_records = []

for idx, text_value in enumerate(candidate_df["resume_text"]):
    titles = extract_experience_titles(text_value)

    current_title = titles[0] if titles else ""
    last_title = titles[-1] if titles else ""

    # Explicit internship experience
    has_intern_experience = any(
        "intern" in title.lower()
        for title in titles
    )

    resume_role_records.append({
        "Resume_Index": idx,
        "Current_Title": current_title,
        "Last_Title": last_title,
        "Experience_Title_Count": len(titles),
        "Has_Intern_Experience": has_intern_experience
    })

resume_roles_df = pd.DataFrame(resume_role_records)

print("=" * 70)
print("ROLE TITLE EXTRACTION AUDIT")
print("=" * 70)

print("\nRows:", len(resume_roles_df))
print(
    "Candidates with >=1 experience title:",
    (resume_roles_df["Experience_Title_Count"] > 0).sum()
)
print(
    "Candidates with explicit internship experience:",
    resume_roles_df["Has_Intern_Experience"].sum()
)

print("\nExperience title count distribution:")
print(resume_roles_df["Experience_Title_Count"].value_counts().sort_index())

print("\nFirst 15 extracted roles:")
print(
    resume_roles_df[
        [
            "Resume_Index",
            "Current_Title",
            "Last_Title",
            "Experience_Title_Count",
            "Has_Intern_Experience"
        ]
    ].head(15).to_string(index=False)
)

# ------------------------------------------------------------
# Job internship detection
# ------------------------------------------------------------

job_role_df = entry_job_df[["Title"]].copy()

job_role_df["Job_Index"] = np.arange(len(job_role_df))

job_role_df["Is_Intern_Job"] = (
    job_role_df["Title"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains(r"\bintern\b", regex=True)
)

print("\nIntern jobs:", job_role_df["Is_Intern_Job"].sum())
print("Non-intern jobs:", (~job_role_df["Is_Intern_Job"]).sum())

print("\nInternship job titles:")
print(
    job_role_df.loc[
        job_role_df["Is_Intern_Job"],
        ["Job_Index", "Title"]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Sanity assertions
# ------------------------------------------------------------

assert len(resume_roles_df) == 1540
assert len(job_role_df) == 434

print("\n✅ Role extraction complete.")
print("✅ Existing V3 data has not been modified.")

In [ ]:
# ============================================================
# V3 ENHANCEMENT — CELL 3
# ROLE-TITLE SEMANTIC SIMILARITY
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Load MiniLM safely without overwriting the ExtraTrees model
# ------------------------------------------------------------

if (
    "semantic_model" in globals()
    and hasattr(semantic_model, "encode")
):
    print("✅ Existing semantic_model found.")

elif (
    "model" in globals()
    and hasattr(model, "encode")
):
    # Only use `model` if it is actually a sentence-transformer
    semantic_model = model
    print("✅ Existing MiniLM found in `model`; assigned to `semantic_model`.")

else:
    from sentence_transformers import SentenceTransformer

    print("Loading MiniLM...")
    semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
    print("✅ MiniLM loaded into `semantic_model`.")

# ------------------------------------------------------------
# Prepare title texts
# ------------------------------------------------------------

current_titles = (
    resume_roles_df["Current_Title"]
    .fillna("")
    .astype(str)
    .tolist()
)

last_titles = (
    resume_roles_df["Last_Title"]
    .fillna("")
    .astype(str)
    .tolist()
)

job_titles = (
    entry_job_df["Title"]
    .fillna("")
    .astype(str)
    .tolist()
)

# ------------------------------------------------------------
# Encode current role titles
# ------------------------------------------------------------

print("=" * 70)
print("ENCODING RESUME ROLE TITLES")
print("=" * 70)

current_role_embeddings = semantic_model.encode(
    current_titles,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Current role embeddings:",
    current_role_embeddings.shape
)

# ------------------------------------------------------------
# Encode last role titles
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ENCODING LAST ROLE TITLES")
print("=" * 70)

last_role_embeddings = semantic_model.encode(
    last_titles,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Last role embeddings:",
    last_role_embeddings.shape
)

# ------------------------------------------------------------
# Encode job titles
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ENCODING JOB TITLES")
print("=" * 70)

job_title_embeddings = semantic_model.encode(
    job_titles,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Job title embeddings:",
    job_title_embeddings.shape
)

# ------------------------------------------------------------
# Similarity
#
# Embeddings are normalized, so dot product = cosine similarity
#
# Shape:
# job x resume = (434, 1540)
# ------------------------------------------------------------

current_title_similarity = (
    job_title_embeddings @ current_role_embeddings.T
)

last_title_similarity = (
    job_title_embeddings @ last_role_embeddings.T
)

# ------------------------------------------------------------
# SAME ROLE-SELECTION LOGIC AS EXISTING ROLE_SIMILARITY
#
# NORMAL JOB:
#     current role
#
# INTERN + explicit internship experience:
#     last role
#
# INTERN + no explicit internship experience:
#     max(current, last)
# ------------------------------------------------------------

is_intern_job = (
    entry_job_df["Title"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains(r"\bintern\b", regex=True)
    .to_numpy()
)

has_intern_experience = (
    resume_roles_df["Has_Intern_Experience"]
    .to_numpy()
)

role_title_semantic_matrix = current_title_similarity.copy()

# Internship + explicit internship experience
intern_exp_mask = (
    is_intern_job[:, None]
    & has_intern_experience[None, :]
)

role_title_semantic_matrix[intern_exp_mask] = (
    last_title_similarity[intern_exp_mask]
)

# Internship + no explicit internship experience
intern_fallback_mask = (
    is_intern_job[:, None]
    & (~has_intern_experience[None, :])
)

role_title_semantic_matrix[intern_fallback_mask] = np.maximum(
    current_title_similarity[intern_fallback_mask],
    last_title_similarity[intern_fallback_mask]
)

# ------------------------------------------------------------
# Clean numeric values
# ------------------------------------------------------------

role_title_semantic_matrix = np.asarray(
    role_title_semantic_matrix,
    dtype=np.float32
)

role_title_semantic_matrix = np.clip(
    role_title_semantic_matrix,
    -1.0,
    1.0
)

# ------------------------------------------------------------
# Save matrix
# ------------------------------------------------------------

role_title_matrix_path = os.path.join(
    V3_OUT,
    "v3_role_title_semantic_similarity_matrix.npy"
)

np.save(
    role_title_matrix_path,
    role_title_semantic_matrix
)

# ------------------------------------------------------------
# Save embeddings
# ------------------------------------------------------------

np.save(
    os.path.join(
        V3_OUT,
        "v3_resume_current_role_title_embeddings.npy"
    ),
    current_role_embeddings.astype(np.float32)
)

np.save(
    os.path.join(
        V3_OUT,
        "v3_resume_last_role_title_embeddings.npy"
    ),
    last_role_embeddings.astype(np.float32)
)

np.save(
    os.path.join(
        V3_OUT,
        "v3_job_title_embeddings.npy"
    ),
    job_title_embeddings.astype(np.float32)
)

# ------------------------------------------------------------
# Metadata
# ------------------------------------------------------------

metadata = {
    "model": "all-MiniLM-L6-v2",
    "feature": "Role_Title_Semantic_Similarity",
    "resume_count": 1540,
    "job_count": 434,
    "embedding_dimension": 384,
    "matrix_shape": [434, 1540],
    "selection_logic": {
        "normal_job": "current_role",
        "intern_with_explicit_internship": "last_role",
        "intern_without_explicit_internship": "max(current_role, last_role)"
    }
}

metadata_path = os.path.join(
    V3_OUT,
    "v3_role_title_semantic_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ROLE-TITLE SEMANTIC SIMILARITY AUDIT")
print("=" * 70)

print(
    "Matrix shape:",
    role_title_semantic_matrix.shape
)

print(
    "dtype:",
    role_title_semantic_matrix.dtype
)

print(
    "Min:",
    float(role_title_semantic_matrix.min())
)

print(
    "Max:",
    float(role_title_semantic_matrix.max())
)

print(
    "Mean:",
    float(role_title_semantic_matrix.mean())
)

print(
    "Median:",
    float(np.median(role_title_semantic_matrix))
)

# ------------------------------------------------------------
# Concrete examples
# ------------------------------------------------------------

examples = [
    ("python developer", "python developer"),
    ("java developer", "java developer"),
    (
        "machine learning engineer intern",
        "ml engineer intern"
    ),
    (
        "backend developer",
        "backend developer - entry level"
    ),
    (
        "backend developer",
        "python developer"
    )
]

print("\nExample title similarities:")

for resume_title, job_title in examples:

    r_idx = next(
        (
            i for i, x in enumerate(current_titles)
            if x.lower() == resume_title.lower()
        ),
        None
    )

    j_idx = next(
        (
            i for i, x in enumerate(job_titles)
            if x.lower() == job_title.lower()
        ),
        None
    )

    if r_idx is not None and j_idx is not None:
        print(
            f"  {resume_title!r} <-> {job_title!r}: "
            f"{role_title_semantic_matrix[j_idx, r_idx]:.6f}"
        )

# ------------------------------------------------------------
# Cleanup large temporary matrices
# ------------------------------------------------------------

del current_title_similarity
del last_title_similarity

gc.collect()

print("\n✅ Role-title semantic feature generated successfully.")
print("✅ Saved:", role_title_matrix_path)
print("✅ ExtraTrees `model` was NOT overwritten.")

In [ ]:
# Safe cleanup after weighted-skill computation

if "weighted_skill_flat" in locals():
    del weighted_skill_flat

gc.collect()

print("✅ Cleanup complete.")
print("✅ Weighted skill matrix is already saved.")
print(
    "✅ Saved:",
    weighted_skill_matrix_path
)

In [ ]:
# ============================================================
# V3 ENHANCEMENT — CELL 5
# BUILD 9-FEATURE V3 DATASET
# ============================================================

import os
import gc
import numpy as np
import pandas as pd

print("=" * 70)
print("BUILDING 9-FEATURE V3 DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Existing V3 feature columns
# ------------------------------------------------------------

base_features = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

# New features
new_features = [
    "Role_Title_Semantic_Similarity",
    "Weighted_Skill_Score"
]

all_features = base_features + new_features

print("\nExisting features:")
for i, f in enumerate(base_features, 1):
    print(f"  {i}. {f}")

print("\nNew features:")
for i, f in enumerate(new_features, 8):
    print(f"  {i}. {f}")

# ------------------------------------------------------------
# Verify source matrices
# ------------------------------------------------------------

assert role_title_semantic_matrix.shape == (434, 1540)
assert weighted_skill_matrix.shape == (434, 1540)

print("\n✅ Role-title matrix verified:", role_title_semantic_matrix.shape)
print("✅ Weighted-skill matrix verified:", weighted_skill_matrix.shape)

# ------------------------------------------------------------
# Add new feature columns to the existing V3 dataset
#
# IMPORTANT:
# Existing V3 rows use:
#     Job_Index
#     Resume_Index
#
# New matrices use:
#     [Job_Index, Resume_Index]
# ------------------------------------------------------------

v3_enhanced_df = v3_df.copy()

v3_enhanced_df["Role_Title_Semantic_Similarity"] = (
    v3_enhanced_df.apply(
        lambda row: role_title_semantic_matrix[
            int(row["Job_Index"]),
            int(row["Resume_Index"])
        ],
        axis=1
    ).astype(np.float32)
)

v3_enhanced_df["Weighted_Skill_Score"] = (
    v3_enhanced_df.apply(
        lambda row: weighted_skill_matrix[
            int(row["Job_Index"]),
            int(row["Resume_Index"])
        ],
        axis=1
    ).astype(np.float32)
)

# ------------------------------------------------------------
# Verify shape
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET CHECK")
print("=" * 70)

print("Original V3 shape:", v3_df.shape)
print("Enhanced V3 shape:", v3_enhanced_df.shape)

assert len(v3_enhanced_df) == 568210

# 11 original columns + 2 new = 13
assert len(v3_enhanced_df.columns) == 13

# ------------------------------------------------------------
# Verify feature columns
# ------------------------------------------------------------

print("\nEnhanced feature columns:")

for i, feature in enumerate(all_features, 1):
    print(f"  {i}. {feature}")

for feature in all_features:
    assert feature in v3_enhanced_df.columns

# ------------------------------------------------------------
# Check missing values
# ------------------------------------------------------------

print("\nMissing values by feature:")

missing_counts = (
    v3_enhanced_df[all_features]
    .isna()
    .sum()
)

print(missing_counts.to_string())

assert missing_counts.sum() == 0

# ------------------------------------------------------------
# Check value ranges
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE RANGE CHECK")
print("=" * 70)

range_rows = []

for feature in all_features:

    values = v3_enhanced_df[feature].to_numpy()

    range_rows.append({
        "Feature": feature,
        "Min": float(np.min(values)),
        "Max": float(np.max(values)),
        "Mean": float(np.mean(values)),
        "NaN": int(np.isnan(values).sum())
    })

range_df = pd.DataFrame(range_rows)

print(
    range_df.to_string(index=False)
)

# ------------------------------------------------------------
# Validate the two new features against their original matrices
#
# Random sample check prevents accidental indexing mistakes.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INDEX ALIGNMENT CHECK")
print("=" * 70)

rng = np.random.default_rng(2026)

sample_indices = rng.choice(
    len(v3_enhanced_df),
    size=1000,
    replace=False
)

sample = v3_enhanced_df.iloc[sample_indices]

expected_role_title = np.array([
    role_title_semantic_matrix[
        int(job_idx),
        int(resume_idx)
    ]
    for job_idx, resume_idx in zip(
        sample["Job_Index"],
        sample["Resume_Index"]
    )
])

expected_weighted_skill = np.array([
    weighted_skill_matrix[
        int(job_idx),
        int(resume_idx)
    ]
    for job_idx, resume_idx in zip(
        sample["Job_Index"],
        sample["Resume_Index"]
    )
])

role_diff = np.max(
    np.abs(
        sample["Role_Title_Semantic_Similarity"].to_numpy()
        - expected_role_title
    )
)

weighted_diff = np.max(
    np.abs(
        sample["Weighted_Skill_Score"].to_numpy()
        - expected_weighted_skill
    )
)

print(
    "Role-title max abs diff:",
    float(role_diff)
)

print(
    "Weighted-skill max abs diff:",
    float(weighted_diff)
)

assert role_diff < 1e-7
assert weighted_diff < 1e-7

# ------------------------------------------------------------
# Save enhanced dataset
# ------------------------------------------------------------

enhanced_pkl_path = os.path.join(
    V3_OUT,
    "v3_supervised_dataset_568210_9features.pkl"
)

enhanced_csv_path = os.path.join(
    V3_OUT,
    "v3_supervised_dataset_568210_9features.csv"
)

# Pickle is the main artifact because it is much faster to reload.
v3_enhanced_df.to_pickle(
    enhanced_pkl_path
)

# CSV is provided for inspection/interoperability.
v3_enhanced_df.to_csv(
    enhanced_csv_path,
    index=False
)

# ------------------------------------------------------------
# Save feature manifest
# ------------------------------------------------------------

manifest = pd.DataFrame({
    "Feature_Index": range(1, 10),
    "Feature_Name": all_features,
    "Source": [
        "Existing V3",
        "Existing V3",
        "Existing V3",
        "Existing V3",
        "Existing V3",
        "Existing V3",
        "Existing V3",
        "New V3 enhancement",
        "New V3 enhancement"
    ]
})

manifest_path = os.path.join(
    V3_OUT,
    "v3_9feature_manifest.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL 9-FEATURE DATASET")
print("=" * 70)

print("Rows:", len(v3_enhanced_df))
print("Columns:", len(v3_enhanced_df.columns))
print("ML features:", len(all_features))

print("\nFeature order:")
print(all_features)

print("\nLabel distribution:")
print(
    v3_enhanced_df["Label"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n✅ 9-feature dataset built successfully.")
print("✅ Alignment checks passed.")
print("✅ Existing 7 features were preserved.")
print("✅ Labels were not modified.")

print("\nSaved:")
print("  ", enhanced_pkl_path)
print("  ", enhanced_csv_path)
print("  ", manifest_path)

In [ ]:
# ============================================================
# V3 ENHANCEMENT — CELL 6
# BENCHMARK + BASELINE VERIFICATION — CORRECTED
# ============================================================

import os
import gc
import joblib
import numpy as np
import pandas as pd

print("=" * 70)
print("V3 ENHANCEMENT — BENCHMARK + BASELINE VERIFICATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. FEATURE DEFINITIONS
# ------------------------------------------------------------

FEATURES_7 = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

FEATURE_ROLE = FEATURES_7 + [
    "Role_Title_Semantic_Similarity"
]

FEATURE_SKILL = FEATURES_7 + [
    "Weighted_Skill_Score"
]

FEATURES_9 = FEATURES_7 + [
    "Role_Title_Semantic_Similarity",
    "Weighted_Skill_Score"
]

print("\n7-feature baseline:")
print(FEATURES_7)

print("\n8-feature role variant:")
print(FEATURE_ROLE)

print("\n8-feature weighted-skill variant:")
print(FEATURE_SKILL)

print("\n9-feature combined variant:")
print(FEATURES_9)


# ------------------------------------------------------------
# 2. LOAD THE EXISTING BENCHMARK
# ------------------------------------------------------------

benchmark_path = os.path.join(
    V3_OUT,
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

benchmark_eval = pd.read_csv(
    benchmark_path
)

print("\n" + "=" * 70)
print("BENCHMARK CHECK")
print("=" * 70)

print("Benchmark shape:", benchmark_eval.shape)

print("Columns:")
print(benchmark_eval.columns.tolist())

assert len(benchmark_eval) == 25400

required_cols = {
    "Pair_ID",
    "Trial",
    "Seed",
    "Resume_Index",
    "Job_Index",
    "Label"
}

assert required_cols.issubset(
    benchmark_eval.columns
)

print("\nUnique trials:", benchmark_eval["Trial"].nunique())

print(
    "Unique resumes:",
    benchmark_eval["Resume_Index"].nunique()
)

print(
    "Unique pairs:",
    benchmark_eval[
        ["Resume_Index", "Job_Index"]
    ].drop_duplicates().shape[0]
)

print("\nRows per trial:")
print(
    benchmark_eval
    .groupby("Trial")
    .size()
    .to_string()
)

assert benchmark_eval["Trial"].nunique() == 10

assert (
    benchmark_eval
    .groupby("Trial")
    .size()
    .eq(2540)
    .all()
)

assert benchmark_eval["Resume_Index"].nunique() == 308

print("\n✅ Benchmark structure verified.")
print("✅ Labels already present — no merge required.")


# ------------------------------------------------------------
# 3. ADD THE TWO NEW FEATURES DIRECTLY FROM MATRICES
#
# This is safer than merging because the benchmark contains
# the same resume-job pairs across multiple trials.
# ------------------------------------------------------------

job_indices = (
    benchmark_eval["Job_Index"]
    .to_numpy(dtype=np.int32)
)

resume_indices = (
    benchmark_eval["Resume_Index"]
    .to_numpy(dtype=np.int32)
)

benchmark_eval["Role_Title_Semantic_Similarity"] = (
    role_title_semantic_matrix[
        job_indices,
        resume_indices
    ]
)

benchmark_eval["Weighted_Skill_Score"] = (
    weighted_skill_matrix[
        job_indices,
        resume_indices
    ]
)

benchmark_eval[
    "Role_Title_Semantic_Similarity"
] = benchmark_eval[
    "Role_Title_Semantic_Similarity"
].astype(np.float32)

benchmark_eval[
    "Weighted_Skill_Score"
] = benchmark_eval[
    "Weighted_Skill_Score"
].astype(np.float32)


# ------------------------------------------------------------
# 4. VERIFY NEW FEATURES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ENHANCED BENCHMARK FEATURE CHECK")
print("=" * 70)

print(
    "Role-title missing:",
    benchmark_eval[
        "Role_Title_Semantic_Similarity"
    ].isna().sum()
)

print(
    "Weighted-skill missing:",
    benchmark_eval[
        "Weighted_Skill_Score"
    ].isna().sum()
)

assert (
    benchmark_eval[
        "Role_Title_Semantic_Similarity"
    ].isna().sum() == 0
)

assert (
    benchmark_eval[
        "Weighted_Skill_Score"
    ].isna().sum() == 0
)

print(
    "\nRole-title range:",
    benchmark_eval[
        "Role_Title_Semantic_Similarity"
    ].min(),
    "→",
    benchmark_eval[
        "Role_Title_Semantic_Similarity"
    ].max()
)

print(
    "Weighted-skill range:",
    benchmark_eval[
        "Weighted_Skill_Score"
    ].min(),
    "→",
    benchmark_eval[
        "Weighted_Skill_Score"
    ].max()
)

print("✅ New features attached successfully.")


# ------------------------------------------------------------
# 5. BUILD FEATURE MATRICES
# ------------------------------------------------------------

X_benchmark_7 = (
    benchmark_eval[
        FEATURES_7
    ].to_numpy(dtype=np.float32)
)

X_benchmark_role = (
    benchmark_eval[
        FEATURE_ROLE
    ].to_numpy(dtype=np.float32)
)

X_benchmark_skill = (
    benchmark_eval[
        FEATURE_SKILL
    ].to_numpy(dtype=np.float32)
)

X_benchmark_9 = (
    benchmark_eval[
        FEATURES_9
    ].to_numpy(dtype=np.float32)
)

y_benchmark = (
    benchmark_eval["Label"]
    .to_numpy(dtype=np.int8)
)

print("\n" + "=" * 70)
print("FEATURE MATRIX SHAPES")
print("=" * 70)

print("7-feature:", X_benchmark_7.shape)
print("Role 8-feature:", X_benchmark_role.shape)
print("Skill 8-feature:", X_benchmark_skill.shape)
print("9-feature:", X_benchmark_9.shape)

assert X_benchmark_7.shape == (25400, 7)
assert X_benchmark_role.shape == (25400, 8)
assert X_benchmark_skill.shape == (25400, 8)
assert X_benchmark_9.shape == (25400, 9)


# ------------------------------------------------------------
# 6. LOAD EXISTING V3 CHAMPION
# ------------------------------------------------------------

baseline_model_path = os.path.join(
    V3_OUT,
    "v3_extratrees_tuned_final.joblib"
)

baseline_model = joblib.load(
    baseline_model_path
)

print("\n" + "=" * 70)
print("EXISTING V3 CHAMPION")
print("=" * 70)

print(
    "Model:",
    type(baseline_model).__name__
)

print(
    "Number of features:",
    baseline_model.n_features_in_
)

assert baseline_model.n_features_in_ == 7

print("✅ Existing 7-feature champion loaded.")


# ------------------------------------------------------------
# 7. BASELINE PREDICTIONS
# ------------------------------------------------------------

baseline_predictions = baseline_model.predict(
    X_benchmark_7
)

assert len(baseline_predictions) == 25400

benchmark_eval[
    "Enhancement_Baseline_Score"
] = baseline_predictions

print(
    "\n✅ Baseline predictions generated:",
    len(baseline_predictions)
)


# ------------------------------------------------------------
# 8. EVALUATION FUNCTIONS
# ------------------------------------------------------------

def rank_group(group, score_column):

    return group.sort_values(
        score_column,
        ascending=False,
        kind="mergesort"
    )


def top1_metric(group, score_column):

    ranked = rank_group(
        group,
        score_column
    )

    best_label = group["Label"].max()

    return float(
        ranked.iloc[0]["Label"] == best_label
    )


def hit_at_k_metric(
    group,
    score_column,
    k
):

    ranked = rank_group(
        group,
        score_column
    )

    best_label = group["Label"].max()

    return float(
        (
            ranked.head(k)["Label"]
            == best_label
        ).any()
    )


def ndcg_metric(
    group,
    score_column
):

    ranked = rank_group(
        group,
        score_column
    )

    relevance = (
        ranked["Label"]
        .to_numpy(dtype=np.float64)
    )

    gains = (
        np.power(2.0, relevance) - 1.0
    )

    discounts = np.log2(
        np.arange(
            2,
            len(relevance) + 2
        )
    )

    dcg = np.sum(
        gains / discounts
    )

    ideal_relevance = np.sort(
        relevance
    )[::-1]

    ideal_gains = (
        np.power(2.0, ideal_relevance)
        - 1.0
    )

    ideal_dcg = np.sum(
        ideal_gains / discounts
    )

    if ideal_dcg == 0:
        return 0.0

    return float(
        dcg / ideal_dcg
    )


def pairwise_metric(
    group,
    score_column
):

    scores = (
        group[score_column]
        .to_numpy(dtype=np.float64)
    )

    labels = (
        group["Label"]
        .to_numpy(dtype=np.int8)
    )

    correct = 0.0
    total = 0

    n = len(group)

    for i in range(n):

        for j in range(i + 1, n):

            if labels[i] == labels[j]:
                continue

            total += 1

            if scores[i] == scores[j]:

                correct += 0.5

            elif (
                scores[i] > scores[j]
                and labels[i] > labels[j]
            ) or (
                scores[i] < scores[j]
                and labels[i] < labels[j]
            ):

                correct += 1.0

    if total == 0:
        return 0.0

    return correct / total


# ------------------------------------------------------------
# 9. EVALUATE BASELINE
# ------------------------------------------------------------

baseline_results = []

for (
    trial,
    resume_idx
), group in benchmark_eval.groupby(
    ["Trial", "Resume_Index"],
    sort=False
):

    baseline_results.append({

        "Trial": trial,

        "Resume_Index": resume_idx,

        "Top-1": top1_metric(
            group,
            "Enhancement_Baseline_Score"
        ),

        "Hit@3": hit_at_k_metric(
            group,
            "Enhancement_Baseline_Score",
            3
        ),

        "Hit@5": hit_at_k_metric(
            group,
            "Enhancement_Baseline_Score",
            5
        ),

        "NDCG": ndcg_metric(
            group,
            "Enhancement_Baseline_Score"
        ),

        "Pairwise": pairwise_metric(
            group,
            "Enhancement_Baseline_Score"
        )
    })


baseline_case_metrics = pd.DataFrame(
    baseline_results
)

baseline_metrics = (
    baseline_case_metrics[
        [
            "Top-1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise"
        ]
    ]
    .mean()
    .to_dict()
)


# ------------------------------------------------------------
# 10. DISPLAY BASELINE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXISTING V3 BASELINE — BENCHMARK RESULT")
print("=" * 70)

for metric, value in baseline_metrics.items():

    print(
        f"{metric:<10}: {value:.6f}"
    )


# ------------------------------------------------------------
# 11. COMPARE MAIN METRICS TO TRUSTED V3 NUMBERS
#
# Pairwise is intentionally treated separately because the
# earlier audit found a small implementation discrepancy.
# ------------------------------------------------------------

official_baseline = {
    "Top-1": 0.822047,
    "Hit@3": 0.961417,
    "Hit@5": 0.986220,
    "NDCG": 0.941930
}

print("\n" + "=" * 70)
print("TRUSTED V3 BASELINE COMPARISON")
print("=" * 70)

for metric, official_value in official_baseline.items():

    measured = baseline_metrics[metric]

    diff = measured - official_value

    print(
        f"{metric:<10} "
        f"Trusted={official_value:.6f}  "
        f"Measured={measured:.6f}  "
        f"Diff={diff:+.6f}"
    )


# We only require the reproduction to be very close.
for metric, official_value in official_baseline.items():

    assert abs(
        baseline_metrics[metric]
        - official_value
    ) < 0.002


print(
    "\n✅ Existing V3 baseline reproduced within tolerance."
)


# ------------------------------------------------------------
# 12. SAVE BASELINE BENCHMARK PREDICTIONS
# ------------------------------------------------------------

baseline_predictions_path = os.path.join(
    V3_OUT,
    "v3_enhancement_baseline_benchmark_predictions.csv"
)

benchmark_eval[
    [
        "Pair_ID",
        "Trial",
        "Seed",
        "Resume_Index",
        "Job_Index",
        "Label",
        "Enhancement_Baseline_Score",
        "Role_Title_Semantic_Similarity",
        "Weighted_Skill_Score"
    ]
].to_csv(
    baseline_predictions_path,
    index=False
)

print(
    "\n✅ Baseline benchmark predictions saved:"
)

print(
    baseline_predictions_path
)


# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------

gc.collect()

print("\n" + "=" * 70)
print("CELL 6 COMPLETE ✅")
print("=" * 70)

print(
    "Next: train V3-R, V3-S and V3-RS using the same "
    "training split and ExtraTrees configuration."
)

In [ ]:
# ============================================================
# COLAB RUNTIME RECOVERY
# ============================================================

import os
import gc
import joblib
import numpy as np
import pandas as pd

from google.colab import drive

drive.mount("/content/drive")

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

V3_OUT = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

V2_OUT = "/content/drive/MyDrive/Resume_Model_Project/V2_100k/Outputs"

print("V3 output directory:")
print(V3_OUT)

print("\nFiles available:")
print("\n".join(sorted(os.listdir(V3_OUT))))


# ------------------------------------------------------------
# LOAD BENCHMARK
# ------------------------------------------------------------

benchmark_path = os.path.join(
    V3_OUT,
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

benchmark_eval = pd.read_csv(
    benchmark_path
)

print("\nBenchmark shape:", benchmark_eval.shape)


# ------------------------------------------------------------
# LOAD NEW FEATURE MATRICES
# ------------------------------------------------------------

role_title_semantic_matrix = np.load(
    os.path.join(
        V3_OUT,
        "v3_role_title_semantic_similarity_matrix.npy"
    ),
    mmap_mode="r"
)

weighted_skill_score_matrix = np.load(
    os.path.join(
        V3_OUT,
        "v3_weighted_skill_score_matrix.npy"
    ),
    mmap_mode="r"
)

print(
    "Role-title matrix:",
    role_title_semantic_matrix.shape
)

print(
    "Weighted-skill matrix:",
    weighted_skill_score_matrix.shape
)


# ------------------------------------------------------------
# ATTACH NEW FEATURES TO BENCHMARK
# ------------------------------------------------------------

job_indices = benchmark_eval[
    "Job_Index"
].to_numpy(dtype=np.int32)

resume_indices = benchmark_eval[
    "Resume_Index"
].to_numpy(dtype=np.int32)


benchmark_eval[
    "Role_Title_Semantic_Similarity"
] = role_title_semantic_matrix[
    job_indices,
    resume_indices
]

benchmark_eval[
    "Weighted_Skill_Score"
] = weighted_skill_score_matrix[
    job_indices,
    resume_indices
]


# ------------------------------------------------------------
# FEATURE DEFINITIONS
# ------------------------------------------------------------

FEATURES_7 = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

FEATURE_ROLE = [
    *FEATURES_7,
    "Role_Title_Semantic_Similarity"
]

FEATURE_SKILL = [
    *FEATURES_7,
    "Weighted_Skill_Score"
]

FEATURES_9 = [
    *FEATURES_7,
    "Role_Title_Semantic_Similarity",
    "Weighted_Skill_Score"
]


# ------------------------------------------------------------
# FEATURE MATRICES
# ------------------------------------------------------------

X_benchmark_7 = (
    benchmark_eval[FEATURES_7]
    .to_numpy(dtype=np.float32)
)

X_benchmark_role = (
    benchmark_eval[FEATURE_ROLE]
    .to_numpy(dtype=np.float32)
)

X_benchmark_skill = (
    benchmark_eval[FEATURE_SKILL]
    .to_numpy(dtype=np.float32)
)

X_benchmark_9 = (
    benchmark_eval[FEATURES_9]
    .to_numpy(dtype=np.float32)
)

y_benchmark = (
    benchmark_eval["Label"]
    .to_numpy(dtype=np.int8)
)


# ------------------------------------------------------------
# CHECKS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RECOVERY CHECK")
print("=" * 70)

print("Benchmark:", benchmark_eval.shape)
print("7-feature:", X_benchmark_7.shape)
print("Role 8-feature:", X_benchmark_role.shape)
print("Skill 8-feature:", X_benchmark_skill.shape)
print("9-feature:", X_benchmark_9.shape)

print(
    "Role-title missing:",
    benchmark_eval["Role_Title_Semantic_Similarity"].isna().sum()
)

print(
    "Weighted-skill missing:",
    benchmark_eval["Weighted_Skill_Score"].isna().sum()
)

assert benchmark_eval.shape[0] == 25400
assert X_benchmark_7.shape == (25400, 7)
assert X_benchmark_role.shape == (25400, 8)
assert X_benchmark_skill.shape == (25400, 8)
assert X_benchmark_9.shape == (25400, 9)

assert benchmark_eval[
    "Role_Title_Semantic_Similarity"
].isna().sum() == 0

assert benchmark_eval[
    "Weighted_Skill_Score"
].isna().sum() == 0

print("\n✅ Runtime recovered.")
print("✅ Benchmark rebuilt.")
print("✅ All feature matrices rebuilt.")

In [ ]:
# ============================================================
# V3 ENHANCEMENT — CELL 6 CONTINUATION
# EXISTING V3 CHAMPION VERIFICATION
# ============================================================

import os
import gc
import joblib
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. LOAD EXISTING V3 CHAMPION
# ------------------------------------------------------------

baseline_model_path = os.path.join(
    V3_OUT,
    "v3_extratrees_tuned_final.joblib"
)

baseline_model = joblib.load(
    baseline_model_path
)

print("\n" + "=" * 70)
print("EXISTING V3 CHAMPION")
print("=" * 70)

print(
    "Model:",
    type(baseline_model).__name__
)

print(
    "Number of features:",
    baseline_model.n_features_in_
)

print(
    "Parameters:",
    baseline_model.get_params()
)

assert baseline_model.n_features_in_ == 7

print("\n✅ Existing 7-feature champion loaded.")


# ------------------------------------------------------------
# 2. PREDICT ON EXACT BENCHMARK
# ------------------------------------------------------------

baseline_predictions = baseline_model.predict(
    X_benchmark_7
)

assert len(baseline_predictions) == 25400

benchmark_eval["Enhancement_Baseline_Score"] = (
    baseline_predictions
)

print(
    "\n✅ Baseline predictions generated:",
    len(baseline_predictions)
)


# ------------------------------------------------------------
# 3. GENERIC RANKING FUNCTIONS
# ------------------------------------------------------------

def rank_group(group, score_column):
    return group.sort_values(
        score_column,
        ascending=False,
        kind="mergesort"
    )


def top1_metric(group, score_column):

    ranked = rank_group(
        group,
        score_column
    )

    best_label = group["Label"].max()

    return float(
        ranked.iloc[0]["Label"] == best_label
    )


def hit_at_k_metric(group, score_column, k):

    ranked = rank_group(
        group,
        score_column
    )

    best_label = group["Label"].max()

    return float(
        (
            ranked.head(k)["Label"]
            == best_label
        ).any()
    )


def ndcg_metric(group, score_column):

    ranked = rank_group(
        group,
        score_column
    )

    relevance = ranked["Label"].to_numpy(
        dtype=np.float64
    )

    gains = (
        np.power(2.0, relevance) - 1.0
    )

    discounts = np.log2(
        np.arange(
            2,
            len(relevance) + 2
        )
    )

    dcg = np.sum(
        gains / discounts
    )

    ideal_relevance = np.sort(
        relevance
    )[::-1]

    ideal_gains = (
        np.power(2.0, ideal_relevance)
        - 1.0
    )

    ideal_dcg = np.sum(
        ideal_gains / discounts
    )

    if ideal_dcg == 0:
        return 0.0

    return float(
        dcg / ideal_dcg
    )


def pairwise_metric(group, score_column):

    scores = group[score_column].to_numpy(
        dtype=np.float64
    )

    labels = group["Label"].to_numpy(
        dtype=np.int8
    )

    correct = 0.0
    total = 0

    n = len(group)

    for i in range(n):

        for j in range(i + 1, n):

            if labels[i] == labels[j]:
                continue

            total += 1

            if scores[i] == scores[j]:
                correct += 0.5

            elif (
                scores[i] > scores[j]
                and labels[i] > labels[j]
            ) or (
                scores[i] < scores[j]
                and labels[i] < labels[j]
            ):
                correct += 1.0

    if total == 0:
        return 0.0

    return correct / total


# ------------------------------------------------------------
# 4. EVALUATE BASELINE
# ------------------------------------------------------------

baseline_results = []

for (
    trial,
    resume_idx
), group in benchmark_eval.groupby(
    ["Trial", "Resume_Index"],
    sort=False
):

    baseline_results.append({

        "Trial": trial,

        "Resume_Index": resume_idx,

        "Top-1": top1_metric(
            group,
            "Enhancement_Baseline_Score"
        ),

        "Hit@3": hit_at_k_metric(
            group,
            "Enhancement_Baseline_Score",
            3
        ),

        "Hit@5": hit_at_k_metric(
            group,
            "Enhancement_Baseline_Score",
            5
        ),

        "NDCG": ndcg_metric(
            group,
            "Enhancement_Baseline_Score"
        ),

        "Pairwise": pairwise_metric(
            group,
            "Enhancement_Baseline_Score"
        )
    })


baseline_case_metrics = pd.DataFrame(
    baseline_results
)

baseline_metrics = (
    baseline_case_metrics[
        [
            "Top-1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise"
        ]
    ]
    .mean()
    .to_dict()
)


# ------------------------------------------------------------
# 5. DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXISTING V3 BASELINE — REPRODUCED")
print("=" * 70)

for metric, value in baseline_metrics.items():

    print(
        f"{metric:<10}: {value:.6f}"
    )


# ------------------------------------------------------------
# 6. COMPARE WITH TRUSTED NUMBERS
# ------------------------------------------------------------

official_baseline = {
    "Top-1": 0.822047,
    "Hit@3": 0.961417,
    "Hit@5": 0.986220,
    "NDCG": 0.941930
}

print("\n" + "=" * 70)
print("TRUSTED V3 BASELINE COMPARISON")
print("=" * 70)

for metric, official_value in official_baseline.items():

    measured = baseline_metrics[metric]

    diff = measured - official_value

    print(
        f"{metric:<10} "
        f"Trusted={official_value:.6f}  "
        f"Measured={measured:.6f}  "
        f"Diff={diff:+.6f}"
    )


# ------------------------------------------------------------
# 7. ASSERT MAIN METRICS
# ------------------------------------------------------------

for metric, official_value in official_baseline.items():

    assert abs(
        baseline_metrics[metric]
        - official_value
    ) < 0.002, (
        f"{metric} differs too much from trusted baseline."
    )

print(
    "\n✅ Existing V3 baseline reproduced within tolerance."
)


# ------------------------------------------------------------
# 8. CHECK BENCHMARK'S SAVED V3 SCORE TOO
# ------------------------------------------------------------

if "V3_Score" in benchmark_eval.columns:

    score_diff = np.max(
        np.abs(
            benchmark_eval["V3_Score"].to_numpy(
                dtype=np.float64
            )
            - baseline_predictions.astype(np.float64)
        )
    )

    print("\nSaved benchmark V3 score comparison:")
    print(
        "Maximum absolute difference:",
        float(score_diff)
    )

    print(
        "✅ This confirms the benchmark's stored V3 "
        "scores against the saved champion."
    )


# ------------------------------------------------------------
# 9. SAVE BASELINE PREDICTIONS
# ------------------------------------------------------------

baseline_predictions_path = os.path.join(
    V3_OUT,
    "v3_enhancement_baseline_benchmark_predictions.csv"
)

benchmark_eval[
    [
        "Pair_ID",
        "Trial",
        "Seed",
        "Resume_Index",
        "Job_Index",
        "Label",
        "Enhancement_Baseline_Score",
        "Role_Title_Semantic_Similarity",
        "Weighted_Skill_Score"
    ]
].to_csv(
    baseline_predictions_path,
    index=False
)

print(
    "\n✅ Saved:",
    baseline_predictions_path
)

print("\n" + "=" * 70)
print("CELL 6 COMPLETE ✅")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 7 — V3 ENHANCEMENT ABLATION
# Train:
#   V3-R  = 8 features + Role_Title_Semantic_Similarity
#   V3-S  = 8 features + Weighted_Skill_Score
#   V3-RS = 9 features + both
# ============================================================

import os
import gc
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor


# ============================================================
# 1. LOAD FULL ENHANCED DATASET
# ============================================================

enhanced_pkl = os.path.join(
    V3_OUT,
    "v3_supervised_dataset_568210_9features.pkl"
)

v3_enhanced_df = pd.read_pickle(
    enhanced_pkl
)

print("=" * 70)
print("FULL V3 ENHANCED DATASET")
print("=" * 70)

print("Shape:", v3_enhanced_df.shape)

print(
    "Columns:",
    list(v3_enhanced_df.columns)
)


# ============================================================
# 2. RECREATE EXACT RESUME-LEVEL TRAIN/HOLDOUT SPLIT
# ============================================================

all_resume_indices = np.arange(1540)

train_resumes, holdout_resumes = train_test_split(
    all_resume_indices,
    test_size=0.20,
    random_state=42
)

train_resumes = np.sort(train_resumes)
holdout_resumes = np.sort(holdout_resumes)

print("\n" + "=" * 70)
print("RESUME SPLIT")
print("=" * 70)

print("Training resumes :", len(train_resumes))
print("Holdout resumes  :", len(holdout_resumes))

assert len(train_resumes) == 1232
assert len(holdout_resumes) == 308


# ============================================================
# 3. SELECT ONLY TRAINING-RESUME ROWS
# ============================================================

train_mask = v3_enhanced_df[
    "Resume_Index"
].isin(train_resumes)

train_df = v3_enhanced_df.loc[
    train_mask
].copy()

print("\nTraining pair rows:", len(train_df))

assert len(train_df) == 454451

assert not train_df[
    "Resume_Index"
].isin(holdout_resumes).any()

print("✅ No holdout resumes leaked into training.")


# ============================================================
# 4. MODEL CONFIGURATION
# ============================================================

ET_PARAMS = {
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_leaf": 2,
    "random_state": 42,
    "n_jobs": -1
}


# ============================================================
# 5. METRIC FUNCTIONS
# ============================================================

def evaluate_scores(df, score_column):

    results = []

    for (
        trial,
        resume_idx
    ), group in df.groupby(
        ["Trial", "Resume_Index"],
        sort=False
    ):

        ranked = group.sort_values(
            score_column,
            ascending=False,
            kind="mergesort"
        )

        labels = ranked["Label"].to_numpy(
            dtype=np.int8
        )

        scores = ranked[
            score_column
        ].to_numpy(
            dtype=np.float64
        )

        best_label = labels.max()

        # --------------------------
        # Top-1
        # --------------------------

        top1 = float(
            labels[0] == best_label
        )

        # --------------------------
        # Hit@3
        # --------------------------

        hit3 = float(
            (labels[:3] == best_label).any()
        )

        # --------------------------
        # Hit@5
        # --------------------------

        hit5 = float(
            (labels[:5] == best_label).any()
        )

        # --------------------------
        # NDCG
        # --------------------------

        relevance = labels.astype(
            np.float64
        )

        gains = (
            np.power(2.0, relevance) - 1.0
        )

        discounts = np.log2(
            np.arange(
                2,
                len(relevance) + 2
            )
        )

        dcg = np.sum(
            gains / discounts
        )

        ideal_relevance = np.sort(
            relevance
        )[::-1]

        ideal_gains = (
            np.power(2.0, ideal_relevance)
            - 1.0
        )

        ideal_dcg = np.sum(
            ideal_gains / discounts
        )

        ndcg = (
            dcg / ideal_dcg
            if ideal_dcg > 0
            else 0.0
        )

        # --------------------------
        # Pairwise
        # --------------------------

        pair_correct = 0.0
        pair_total = 0

        n = len(labels)

        for i in range(n):

            for j in range(i + 1, n):

                if labels[i] == labels[j]:
                    continue

                pair_total += 1

                if scores[i] == scores[j]:
                    pair_correct += 0.5

                elif (
                    scores[i] > scores[j]
                    and labels[i] > labels[j]
                ) or (
                    scores[i] < scores[j]
                    and labels[i] < labels[j]
                ):
                    pair_correct += 1.0

        pairwise = (
            pair_correct / pair_total
            if pair_total > 0
            else 0.0
        )

        results.append({
            "Trial": trial,
            "Resume_Index": resume_idx,
            "Top-1": top1,
            "Hit@3": hit3,
            "Hit@5": hit5,
            "NDCG": ndcg,
            "Pairwise": pairwise
        })

    return pd.DataFrame(results)


# ============================================================
# 6. TRAINING FUNCTION
# ============================================================

def train_and_evaluate_variant(
    name,
    features,
    prediction_column,
    model_filename
):

    print("\n" + "=" * 70)
    print(f"TRAINING {name}")
    print("=" * 70)

    print("Features:", features)

    # --------------------------------------------------------
    # Training arrays
    # --------------------------------------------------------

    X_train = train_df[
        features
    ].to_numpy(
        dtype=np.float32
    )

    y_train = train_df[
        "Label"
    ].to_numpy(
        dtype=np.float32
    )

    print(
        "X_train shape:",
        X_train.shape
    )

    print(
        "y_train shape:",
        y_train.shape
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    model = ExtraTreesRegressor(
        **ET_PARAMS
    )

    print("\nFitting ExtraTrees...")

    model.fit(
        X_train,
        y_train
    )

    print("✅ Training complete.")

    # --------------------------------------------------------
    # Save model
    # --------------------------------------------------------

    model_path = os.path.join(
        V3_OUT,
        model_filename
    )

    joblib.dump(
        model,
        model_path
    )

    print(
        "✅ Saved model:",
        model_path
    )

    # --------------------------------------------------------
    # Benchmark prediction
    # --------------------------------------------------------

    print("\nGenerating benchmark predictions...")

    X_benchmark = (
        benchmark_eval[features]
        .to_numpy(dtype=np.float32)
    )

    predictions = model.predict(
        X_benchmark
    )

    assert len(predictions) == 25400

    benchmark_eval[
        prediction_column
    ] = predictions

    print(
        "✅ Predictions:",
        len(predictions)
    )

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    print("\nEvaluating...")

    case_metrics = evaluate_scores(
        benchmark_eval,
        prediction_column
    )

    mean_metrics = (
        case_metrics[
            [
                "Top-1",
                "Hit@3",
                "Hit@5",
                "NDCG",
                "Pairwise"
            ]
        ]
        .mean()
        .to_dict()
    )

    print("\n" + "-" * 70)
    print(f"{name} RESULTS")
    print("-" * 70)

    for metric, value in mean_metrics.items():

        print(
            f"{metric:<10}: {value:.6f}"
        )

    # --------------------------------------------------------
    # Cleanup training arrays
    # --------------------------------------------------------

    del X_train
    del y_train
    del X_benchmark
    del model

    gc.collect()

    return case_metrics, mean_metrics


# ============================================================
# 7. TRAIN V3-R
# ============================================================

role_case_metrics, role_metrics = (
    train_and_evaluate_variant(
        name="V3-R",
        features=FEATURE_ROLE,
        prediction_column="V3R_Score",
        model_filename="v3_extratrees_role8.joblib"
    )
)


# ============================================================
# 8. TRAIN V3-S
# ============================================================

skill_case_metrics, skill_metrics = (
    train_and_evaluate_variant(
        name="V3-S",
        features=FEATURE_SKILL,
        prediction_column="V3S_Score",
        model_filename="v3_extratrees_skill8.joblib"
    )
)


# ============================================================
# 9. TRAIN V3-RS
# ============================================================

role_skill_case_metrics, role_skill_metrics = (
    train_and_evaluate_variant(
        name="V3-RS",
        features=FEATURES_9,
        prediction_column="V3RS_Score",
        model_filename="v3_extratrees_role_skill9.joblib"
    )
)


# ============================================================
# 10. BASELINE METRICS FROM CELL 6
# ============================================================

baseline_summary = {
    "Variant": "V3-Baseline",
    **baseline_metrics
}

role_summary = {
    "Variant": "V3-R",
    **role_metrics
}

skill_summary = {
    "Variant": "V3-S",
    **skill_metrics
}

role_skill_summary = {
    "Variant": "V3-RS",
    **role_skill_metrics
}


# ============================================================
# 11. COMPARISON TABLE
# ============================================================

ablation_summary = pd.DataFrame([
    baseline_summary,
    role_summary,
    skill_summary,
    role_skill_summary
])


print("\n" + "=" * 70)
print("V3 ENHANCEMENT ABLATION SUMMARY")
print("=" * 70)

print(
    ablation_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 12. DELTAS VS BASELINE
# ============================================================

for metric in [
    "Top-1",
    "Hit@3",
    "Hit@5",
    "NDCG",
    "Pairwise"
]:

    baseline_value = baseline_metrics[
        metric
    ]

    ablation_summary[
        f"{metric}_Delta"
    ] = (
        ablation_summary[metric]
        - baseline_value
    )


print("\n" + "=" * 70)
print("DELTA VS V3 BASELINE")
print("=" * 70)

delta_columns = [
    "Variant",
    "Top-1_Delta",
    "Hit@3_Delta",
    "Hit@5_Delta",
    "NDCG_Delta",
    "Pairwise_Delta"
]

print(
    ablation_summary[
        delta_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:+.6f}"
    )
)


# ============================================================
# 13. SAVE RESULTS
# ============================================================

summary_path = os.path.join(
    V3_OUT,
    "v3_enhancement_ablation_summary.csv"
)

ablation_summary.to_csv(
    summary_path,
    index=False
)


# Combine case-level results

baseline_case_metrics = baseline_case_metrics.copy()
baseline_case_metrics["Variant"] = "V3-Baseline"

role_case_metrics = role_case_metrics.copy()
role_case_metrics["Variant"] = "V3-R"

skill_case_metrics = skill_case_metrics.copy()
skill_case_metrics["Variant"] = "V3-S"

role_skill_case_metrics = role_skill_case_metrics.copy()
role_skill_case_metrics["Variant"] = "V3-RS"


case_metrics_all = pd.concat(
    [
        baseline_case_metrics,
        role_case_metrics,
        skill_case_metrics,
        role_skill_case_metrics
    ],
    ignore_index=True
)

case_path = os.path.join(
    V3_OUT,
    "v3_enhancement_ablation_case_metrics.csv"
)

case_metrics_all.to_csv(
    case_path,
    index=False
)


# Trial-level results

trial_results = (
    case_metrics_all
    .groupby(
        ["Variant", "Trial"],
        as_index=False
    )[
        [
            "Top-1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise"
        ]
    ]
    .mean()
)

trial_path = os.path.join(
    V3_OUT,
    "v3_enhancement_ablation_trial_results.csv"
)

trial_results.to_csv(
    trial_path,
    index=False
)


# Predictions

prediction_columns = [
    "Pair_ID",
    "Job_Index",
    "Resume_Index",
    "Label",
    "Trial",
    "Seed",
    "Enhancement_Baseline_Score",
    "V3R_Score",
    "V3S_Score",
    "V3RS_Score",
    "Role_Title_Semantic_Similarity",
    "Weighted_Skill_Score"
]

prediction_path = os.path.join(
    V3_OUT,
    "v3_enhancement_ablation_predictions.csv"
)

benchmark_eval[
    prediction_columns
].to_csv(
    prediction_path,
    index=False
)


print("\n" + "=" * 70)
print("SAVED")
print("=" * 70)

print(summary_path)
print(case_path)
print(trial_path)
print(prediction_path)

print("\n" + "=" * 70)
print("CELL 7 COMPLETE ✅")
print("=" * 70)

In [ ]:
import os
import glob

V3_OUT = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

print("=" * 70)
print("CHECKING FROM THE ORIGINAL TRAINING RUNTIME")
print("=" * 70)

for filename in [
    "v3_extratrees_role8.joblib",
    "v3_extratrees_skill8.joblib",
    "v3_extratrees_role_skill9.joblib",
    "v3_enhancement_ablation_summary.csv",
]:
    path = os.path.join(V3_OUT, filename)

    print(f"\n{filename}")
    print("Path:", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print(
            "Size:",
            os.path.getsize(path) / (1024**2),
            "MB"
        )

print("\n" + "=" * 70)
print("ALL V3 FILES MATCHING EXTRATREES")
print("=" * 70)

for path in glob.glob(
    os.path.join(V3_OUT, "v3_extratrees*")
):
    print(
        os.path.basename(path),
        "->",
        os.path.getsize(path) / (1024**2),
        "MB"
    )

print("\n" + "=" * 70)
print("RECENT FILES")
print("=" * 70)

files = sorted(
    glob.glob(os.path.join(V3_OUT, "*")),
    key=os.path.getmtime,
    reverse=True
)

for path in files[:20]:
    print(
        os.path.basename(path),
        "->",
        os.path.getsize(path) / (1024**2),
        "MB"
    )

In [ ]:
import os

V3_OUT = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

files_to_delete = [
    "v3_extratrees_role8.joblib",
    "v3_extratrees_skill8.joblib",
]

for filename in files_to_delete:
    path = os.path.join(V3_OUT, filename)

    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)
        os.remove(path)
        print(f"✅ Deleted: {filename} ({size_gb:.2f} GB)")
    else:
        print(f"❌ Not found: {filename}")

print("\nCleanup complete.")

In [ ]:
# ============================================================
# CELL 8 — V3-RS CONSISTENCY CHECK
# ============================================================

import os
import pandas as pd
import numpy as np

V3_OUT = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

trial_path = os.path.join(
    V3_OUT,
    "v3_enhancement_ablation_trial_results.csv"
)

trial_results = pd.read_csv(trial_path)

print("=" * 70)
print("V3 ENHANCEMENT — PER-TRIAL RESULTS")
print("=" * 70)

print(
    trial_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# CHECK WHETHER V3-RS WINS EACH METRIC ACROSS TRIALS
# ============================================================

metrics = [
    "Top-1",
    "Hit@3",
    "Hit@5",
    "NDCG",
    "Pairwise"
]

variants = [
    "V3-Baseline",
    "V3-R",
    "V3-S",
    "V3-RS"
]

print("\n" + "=" * 70)
print("V3-RS TRIAL WIN CHECK")
print("=" * 70)

for metric in metrics:

    pivot = trial_results.pivot(
        index="Trial",
        columns="Variant",
        values=metric
    )

    winners = pivot.idxmax(axis=1)

    v3rs_wins = int(
        (winners == "V3-RS").sum()
    )

    print(
        f"{metric:<10}: "
        f"V3-RS wins {v3rs_wins}/10 trials"
    )


# ============================================================
# RANK VARIANTS BY MAIN METRIC
# ============================================================

summary_path = os.path.join(
    V3_OUT,
    "v3_enhancement_ablation_summary.csv"
)

summary = pd.read_csv(summary_path)

print("\n" + "=" * 70)
print("FINAL ABLATION RANKING")
print("=" * 70)

print(
    summary[
        [
            "Variant",
            "Top-1",
            "Hit@3",
            "Hit@5",
            "NDCG",
            "Pairwise"
        ]
    ]
    .sort_values(
        "NDCG",
        ascending=False
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# WINNER
# ============================================================

winner = (
    summary
    .sort_values(
        ["NDCG", "Pairwise", "Top-1"],
        ascending=False
    )
    .iloc[0]
)

print("\n" + "=" * 70)
print("SELECTED WINNER")
print("=" * 70)

print("Variant :", winner["Variant"])
print("NDCG    :", f"{winner['NDCG']:.6f}")
print("Top-1   :", f"{winner['Top-1']:.6f}")
print("Hit@3   :", f"{winner['Hit@3']:.6f}")
print("Hit@5   :", f"{winner['Hit@5']:.6f}")
print("Pairwise:", f"{winner['Pairwise']:.6f}")

assert winner["Variant"] == "V3-RS"

print("\n✅ V3-RS confirmed as the ablation winner.")

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
import google.auth

auth.authenticate_user()

creds, _ = google.auth.default()

drive_service = build(
    "drive",
    "v3",
    credentials=creds
)

about = drive_service.about().get(
    fields="storageQuota"
).execute()

quota = about["storageQuota"]

limit = quota.get("limit")
usage = quota.get("usage", 0)

def gb(x):
    return int(x) / (1024**3)

print("=" * 70)
print("GOOGLE DRIVE STORAGE QUOTA — AFTER EMPTYING TRASH")
print("=" * 70)

print(f"Total quota     : {gb(limit):.2f} GB")
print(f"Total used      : {gb(usage):.2f} GB")

if limit:
    free = int(limit) - int(usage)
    print(f"Space remaining : {gb(free):.2f} GB")

    if free > 0:
        print("\n✅ Space available")
    else:
        print("\n❌ Still over quota")

print("=" * 70)

In [ ]:
# Check whether the trained model is still in RAM

print("final_model exists:", "final_model" in globals())

if "final_model" in globals():
    print("Model type:", type(final_model).__name__)
    print("Features:", final_model.n_features_in_)
    print("Trees:", final_model.n_estimators)
    print("✅ Trained model is still in memory.")
else:
    print("❌ final_model is not in memory.")

In [ ]:
print("=" * 70)
print("FINAL MODEL TRAINING VERIFICATION")
print("=" * 70)

print("Model type:", type(final_model).__name__)
print("Trees:", len(final_model.estimators_))
print("Features:", final_model.n_features_in_)

# Root node of each ExtraTrees tree contains the number
# of training samples used by that tree.
root_samples = [
    tree.tree_.n_node_samples[0]
    for tree in final_model.estimators_
]

print("\nRoot samples per tree:")
print("Min:", min(root_samples))
print("Max:", max(root_samples))
print("Unique:", sorted(set(root_samples)))

assert len(final_model.estimators_) == 200
assert final_model.n_features_in_ == 9
assert min(root_samples) == 568210
assert max(root_samples) == 568210

print("\n✅ All 200 trees used all 568,210 training samples.")
print("✅ Final model is the full-data 9-feature V3-RS model.")

In [ ]:
import os
import joblib

local_path = "/content/v3_final_extratrees_rs_568210.joblib"

print("Saving to local Colab storage...")
joblib.dump(final_model, local_path)

size_gb = os.path.getsize(local_path) / (1024**3)

print(f"✅ Saved locally: {local_path}")
print(f"✅ Size: {size_gb:.2f} GB")

In [ ]:
# ============================================================
# CELL 10 — FINAL V3-RS ERROR ANALYSIS
# CORRECTED: RESTORES ORIGINAL 7 FEATURES FROM BENCHMARK
# ============================================================

import os
import numpy as np
import pandas as pd

V3_OUT = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"


# ============================================================
# 1. LOAD SAVED V3 ENHANCEMENT PREDICTIONS
# ============================================================

pred_path = os.path.join(
    V3_OUT,
    "v3_enhancement_ablation_predictions.csv"
)

pred_df = pd.read_csv(pred_path)

print("=" * 70)
print("V3-RS ERROR ANALYSIS")
print("=" * 70)

print("Prediction file shape:", pred_df.shape)

assert len(pred_df) == 25400

print("\nPrediction columns:")
print(list(pred_df.columns))


# ============================================================
# 2. LOAD ORIGINAL BENCHMARK FEATURES
# ============================================================

benchmark_path = os.path.join(
    V3_OUT,
    "v3_10x_unseen_validation_sampled_pairs.csv"
)

benchmark_df = pd.read_csv(
    benchmark_path
)

print("\nBenchmark shape:", benchmark_df.shape)

assert len(benchmark_df) == 25400


# ============================================================
# 3. RESTORE ORIGINAL 7 FEATURES
# ============================================================

BASE_FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity"
]

KEYS = [
    "Trial",
    "Resume_Index",
    "Job_Index"
]

benchmark_features = benchmark_df[
    KEYS + BASE_FEATURES
].copy()


# Check uniqueness before merging

assert not benchmark_features.duplicated(
    KEYS
).any(), (
    "Benchmark contains duplicate Trial + Resume + Job keys."
)

assert not pred_df.duplicated(
    KEYS
).any(), (
    "Prediction file contains duplicate Trial + Resume + Job keys."
)


# Merge base features into prediction file

df = pred_df.merge(
    benchmark_features,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("\nMerged shape:", df.shape)


# ============================================================
# 4. VERIFY EVERYTHING IS PRESENT
# ============================================================

REQUIRED_COLUMNS = [
    "Pair_ID",
    "Job_Index",
    "Resume_Index",
    "Label",
    "Trial",
    "Seed",
    "V3RS_Score",

    # Original V3 features
    *BASE_FEATURES,

    # New V3-RS features
    "Role_Title_Semantic_Similarity",
    "Weighted_Skill_Score"
]

missing = [
    c for c in REQUIRED_COLUMNS
    if c not in df.columns
]

assert not missing, (
    f"Missing columns after merge: {missing}"
)

assert len(df) == 25400

print("✅ Original 7 features restored.")
print("✅ V3-RS prediction data complete.")


# ============================================================
# 5. VERIFY LABEL CONSISTENCY
# ============================================================

label_check = df.merge(
    benchmark_df[
        KEYS + ["Label"]
    ],
    on=KEYS,
    how="left",
    suffixes=("_pred", "_benchmark"),
    validate="one_to_one"
)

label_matches = (
    label_check["Label_pred"]
    ==
    label_check["Label_benchmark"]
)

print(
    "\nLabel mismatches:",
    int((~label_matches).sum())
)

assert label_matches.all()

print("✅ Labels verified.")


# ============================================================
# 6. RANK EACH RESUME WITHIN EACH TRIAL
# ============================================================

case_rows = []
wrong_top1_rows = []

for (
    trial,
    resume_idx
), group in df.groupby(
    ["Trial", "Resume_Index"],
    sort=False
):

    ranked = group.sort_values(
        "V3RS_Score",
        ascending=False,
        kind="mergesort"
    ).reset_index(drop=True)

    ranked["Rank"] = np.arange(
        1,
        len(ranked) + 1
    )

    best_label = int(
        ranked["Label"].max()
    )

    best_label_rows = ranked[
        ranked["Label"] == best_label
    ]

    best_label_row = best_label_rows.iloc[0]

    top1 = ranked.iloc[0]

    best_label_rank = int(
        best_label_rows["Rank"].min()
    )

    top1_correct = (
        int(top1["Label"]) == best_label
    )

    row = {
        "Trial": trial,
        "Resume_Index": resume_idx,

        "Top1_Job_Index": int(
            top1["Job_Index"]
        ),

        "Best_Label_Job_Index": int(
            best_label_row["Job_Index"]
        ),

        "Top1_Label": int(
            top1["Label"]
        ),

        "Best_Label": best_label,

        "Top1_Correct": int(
            top1_correct
        ),

        "Best_Label_Rank": best_label_rank,

        "Top1_Score": float(
            top1["V3RS_Score"]
        ),

        "Best_Label_Score": float(
            best_label_row["V3RS_Score"]
        ),

        "Score_Gap": float(
            top1["V3RS_Score"]
            - best_label_row["V3RS_Score"]
        ),

        # -------------------------
        # Original features
        # -------------------------

        "Top1_Cosine": float(
            top1["Cosine"]
        ),

        "Best_Cosine": float(
            best_label_row["Cosine"]
        ),

        "Top1_Skill_Score": float(
            top1["Skill_Score"]
        ),

        "Best_Skill_Score": float(
            best_label_row["Skill_Score"]
        ),

        "Top1_Normalized_Cosine": float(
            top1["Normalized_Cosine"]
        ),

        "Best_Normalized_Cosine": float(
            best_label_row["Normalized_Cosine"]
        ),

        "Top1_Required_Skill_Count": int(
            top1["Required_Skill_Count"]
        ),

        "Best_Required_Skill_Count": int(
            best_label_row[
                "Required_Skill_Count"
            ]
        ),

        "Top1_Matched_Skill_Count": int(
            top1["Matched_Skill_Count"]
        ),

        "Best_Matched_Skill_Count": int(
            best_label_row[
                "Matched_Skill_Count"
            ]
        ),

        "Top1_Role_Similarity": float(
            top1["Role_Similarity"]
        ),

        "Best_Role_Similarity": float(
            best_label_row[
                "Role_Similarity"
            ]
        ),

        "Top1_Semantic_Similarity": float(
            top1["Semantic_Similarity"]
        ),

        "Best_Semantic_Similarity": float(
            best_label_row[
                "Semantic_Similarity"
            ]
        ),

        # -------------------------
        # New V3-RS features
        # -------------------------

        "Top1_Role_Title_Semantic": float(
            top1[
                "Role_Title_Semantic_Similarity"
            ]
        ),

        "Best_Role_Title_Semantic": float(
            best_label_row[
                "Role_Title_Semantic_Similarity"
            ]
        ),

        "Top1_Weighted_Skill": float(
            top1[
                "Weighted_Skill_Score"
            ]
        ),

        "Best_Weighted_Skill": float(
            best_label_row[
                "Weighted_Skill_Score"
            ]
        )
    }

    case_rows.append(row)

    if not top1_correct:

        wrong = top1.copy()

        wrong["Best_Label"] = best_label

        wrong["Best_Label_Job_Index"] = int(
            best_label_row["Job_Index"]
        )

        wrong["Best_Label_Rank"] = (
            best_label_rank
        )

        wrong["Best_Label_Score"] = float(
            best_label_row["V3RS_Score"]
        )

        wrong["Score_Gap"] = float(
            top1["V3RS_Score"]
            - best_label_row["V3RS_Score"]
        )

        wrong_top1_rows.append(wrong)


case_df = pd.DataFrame(case_rows)

wrong_top1_df = pd.DataFrame(
    wrong_top1_rows
)


# ============================================================
# 7. BASIC RESULTS
# ============================================================

print("\n" + "=" * 70)
print("ERROR ANALYSIS SUMMARY")
print("=" * 70)

print(
    "Total ranking cases:",
    len(case_df)
)

print(
    "Correct Top-1:",
    int(case_df["Top1_Correct"].sum())
)

print(
    "Wrong Top-1:",
    int(
        (case_df["Top1_Correct"] == 0).sum()
    )
)

print(
    "Observed Top-1:",
    f"{case_df['Top1_Correct'].mean():.6f}"
)


# ============================================================
# 8. BEST-LABEL RANK DISTRIBUTION
# ============================================================

rank_distribution = (
    case_df[
        "Best_Label_Rank"
    ]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 70)
print("BEST-LABEL RANK DISTRIBUTION")
print("=" * 70)

print(rank_distribution)


# ============================================================
# 9. ERROR-ONLY SUMMARY
# ============================================================

errors = case_df[
    case_df["Top1_Correct"] == 0
].copy()

print("\n" + "=" * 70)
print("ERROR-ONLY SUMMARY")
print("=" * 70)

print(
    "Total errors:",
    len(errors)
)

if len(errors) > 0:

    print(
        "Average best-label rank:",
        f"{errors['Best_Label_Rank'].mean():.3f}"
    )

    print(
        "Median best-label rank:",
        f"{errors['Best_Label_Rank'].median():.3f}"
    )

    print(
        "Average score gap:",
        f"{errors['Score_Gap'].mean():.6f}"
    )


# ============================================================
# 10. FEATURE GAPS ON WRONG TOP-1 CASES
# ============================================================

if len(errors) > 0:

    gap_table = pd.DataFrame({

        "Feature": [
            "Cosine",
            "Skill_Score",
            "Normalized_Cosine",
            "Matched_Skill_Count",
            "Role_Similarity",
            "Semantic_Similarity",
            "Role_Title_Semantic_Similarity",
            "Weighted_Skill_Score"
        ],

        "Top1_minus_Best": [

            (
                errors["Top1_Cosine"]
                -
                errors["Best_Cosine"]
            ).mean(),

            (
                errors["Top1_Skill_Score"]
                -
                errors["Best_Skill_Score"]
            ).mean(),

            (
                errors["Top1_Normalized_Cosine"]
                -
                errors["Best_Normalized_Cosine"]
            ).mean(),

            (
                errors["Top1_Matched_Skill_Count"]
                -
                errors["Best_Matched_Skill_Count"]
            ).mean(),

            (
                errors["Top1_Role_Similarity"]
                -
                errors["Best_Role_Similarity"]
            ).mean(),

            (
                errors["Top1_Semantic_Similarity"]
                -
                errors["Best_Semantic_Similarity"]
            ).mean(),

            (
                errors["Top1_Role_Title_Semantic"]
                -
                errors["Best_Role_Title_Semantic"]
            ).mean(),

            (
                errors["Top1_Weighted_Skill"]
                -
                errors["Best_Weighted_Skill"]
            ).mean()
        ]
    })

    print("\n" + "=" * 70)
    print("ERROR FEATURE GAPS")
    print("=" * 70)

    print(
        gap_table.to_string(
            index=False,
            float_format=lambda x: f"{x:+.6f}"
        )
    )


# ============================================================
# 11. TOP 30 WRONG TOP-1 CASES
# ============================================================

if len(wrong_top1_df) > 0:

    worst_errors = (
        wrong_top1_df
        .sort_values(
            "Score_Gap",
            ascending=False
        )
        .head(30)
    )

    print("\n" + "=" * 70)
    print("TOP 30 WRONG TOP-1 CASES")
    print("=" * 70)

    display_columns = [
        "Trial",
        "Resume_Index",
        "Job_Index",
        "Label",
        "Best_Label",
        "Best_Label_Job_Index",
        "Best_Label_Rank",
        "V3RS_Score",
        "Best_Label_Score",
        "Score_Gap",
        "Skill_Score",
        "Matched_Skill_Count",
        "Semantic_Similarity",
        "Role_Title_Semantic_Similarity",
        "Weighted_Skill_Score"
    ]

    print(
        worst_errors[
            [
                c for c in display_columns
                if c in worst_errors.columns
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ============================================================
# 12. SAVE RESULTS
# ============================================================

case_path = os.path.join(
    V3_OUT,
    "v3_final_rs_error_analysis_case_level.csv"
)

wrong_path = os.path.join(
    V3_OUT,
    "v3_final_rs_error_analysis_wrong_top1.csv"
)

rank_path = os.path.join(
    V3_OUT,
    "v3_final_rs_best_label_rank_distribution.csv"
)

gap_path = os.path.join(
    V3_OUT,
    "v3_final_rs_error_feature_gaps.csv"
)

case_df.to_csv(
    case_path,
    index=False
)

wrong_top1_df.to_csv(
    wrong_path,
    index=False
)

rank_distribution.rename(
    "Count"
).to_csv(
    rank_path
)

if len(errors) > 0:
    gap_table.to_csv(
        gap_path,
        index=False
    )


print("\n" + "=" * 70)
print("ERROR ANALYSIS SAVED")
print("=" * 70)

print(case_path)
print(wrong_path)
print(rank_path)

if len(errors) > 0:
    print(gap_path)

print("\n✅ CELL 10 COMPLETE")

In [ ]:
# ============================================================
# CELL 11 — V3-RS FEATURE IMPORTANCE
# RECOVERY VERSION — NO LARGE DRIVE OUTPUTS
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

import os
import gc
import joblib
import numpy as np
import pandas as pd


V3_OUT = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

MODEL_PATH = os.path.join(
    V3_OUT,
    "v3_extratrees_role_skill9.joblib"
)

FEATURES = [
    "Cosine",
    "Skill_Score",
    "Normalized_Cosine",
    "Required_Skill_Count",
    "Matched_Skill_Count",
    "Role_Similarity",
    "Semantic_Similarity",
    "Role_Title_Semantic_Similarity",
    "Weighted_Skill_Score"
]


# ------------------------------------------------------------
# 1. VERIFY MODEL EXISTS
# ------------------------------------------------------------

print("=" * 70)
print("V3-RS FEATURE IMPORTANCE")
print("=" * 70)

print("Checking model...")
print(MODEL_PATH)

assert os.path.exists(MODEL_PATH), (
    "V3-RS holdout model not found!"
)

print("✅ Model exists.")
print(
    f"Model size: "
    f"{os.path.getsize(MODEL_PATH)/(1024**3):.2f} GB"
)


# ------------------------------------------------------------
# 2. LOAD MODEL
# ------------------------------------------------------------

print("\nLoading model...")

model = joblib.load(
    MODEL_PATH
)

print("✅ Model loaded.")
print("Type:", type(model).__name__)
print("Trees:", model.n_estimators)
print("Features:", model.n_features_in_)

assert model.n_features_in_ == 9
assert model.n_estimators == 200


# ------------------------------------------------------------
# 3. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

importance_df["Percentage"] = (
    importance_df["Importance"] * 100
)

print("\n" + "=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)

print(
    importance_df.to_string(
        index=False,
        formatters={
            "Importance": lambda x: f"{x:.6f}",
            "Percentage": lambda x: f"{x:.2f}%"
        }
    )
)


# ------------------------------------------------------------
# 4. NEW FEATURES VS ORIGINAL FEATURES
# ------------------------------------------------------------

new_features = {
    "Role_Title_Semantic_Similarity",
    "Weighted_Skill_Score"
}

new_importance = importance_df[
    importance_df["Feature"].isin(new_features)
]["Importance"].sum()

old_importance = importance_df[
    ~importance_df["Feature"].isin(new_features)
]["Importance"].sum()

print("\n" + "=" * 70)
print("ORIGINAL VS NEW FEATURES")
print("=" * 70)

print(
    f"Original 7 features : "
    f"{old_importance:.6f} "
    f"({old_importance*100:.2f}%)"
)

print(
    f"New 2 features      : "
    f"{new_importance:.6f} "
    f"({new_importance*100:.2f}%)"
)


# ------------------------------------------------------------
# 5. SAVE ONLY TINY CSV LOCALLY
# ------------------------------------------------------------

local_csv = (
    "/content/v3_final_rs_feature_importance.csv"
)

importance_df.to_csv(
    local_csv,
    index=False
)

print("\n✅ Tiny CSV saved locally:")
print(local_csv)

print(
    f"Size: "
    f"{os.path.getsize(local_csv)/1024:.2f} KB"
)


# ------------------------------------------------------------
# 6. RELEASE MODEL
# ------------------------------------------------------------

del model
gc.collect()

print("\n✅ Model released from RAM.")
print("=" * 70)
print("CELL 11 COMPLETE ✅")
print("=" * 70)

In [ ]:
import os
import joblib

MODEL_PATH = "/content/v3_final_extratrees_rs_568210.joblib"

print("=" * 70)
print("FINAL V3-RS MODEL CHECK")
print("=" * 70)

assert os.path.exists(MODEL_PATH), (
    "Final model was not found in /content."
)

size_gb = os.path.getsize(MODEL_PATH) / (1024**3)

print(f"File size: {size_gb:.2f} GB")
print("\nLoading model...")

final_model = joblib.load(MODEL_PATH)

print("✅ Model loaded.")
print("Type:", type(final_model).__name__)
print("Features:", final_model.n_features_in_)
print("Trees:", final_model.n_estimators)
print("Min samples leaf:", final_model.min_samples_leaf)

assert final_model.n_features_in_ == 9
assert final_model.n_estimators == 200
assert final_model.min_samples_leaf == 2

print("\n✅ Final full-data V3-RS model is ready.")

In [ ]:
import numpy as np
import pandas as pd
import os

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

print("=" * 80)
print("V3 FINAL INFERENCE — FEATURE RECONSTRUCTION CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load saved full-pool matrices
# ------------------------------------------------------------

cosine = np.load(
    f"{BASE}/../..//V2_100k/Outputs/v1_similarity_matrix.npy"
)

skill_score = np.load(
    f"{BASE}/../..//V2_100k/Outputs/v1_skill_score_matrix.npy"
)

role_similarity = np.load(
    f"{BASE}/../..//V2_100k/Outputs/role_similarity_matrix.npy"
)

semantic_similarity = np.load(
    f"{BASE}/v3_semantic_similarity_matrix.npy"
)

role_title_semantic = np.load(
    f"{BASE}/v3_role_title_semantic_similarity_matrix.npy"
)

weighted_skill_score = np.load(
    f"{BASE}/v3_weighted_skill_score_matrix.npy"
)

print("Matrix shapes:")
print("Cosine:", cosine.shape)
print("Skill:", skill_score.shape)
print("Role:", role_similarity.shape)
print("Semantic:", semantic_similarity.shape)
print("Role-title semantic:", role_title_semantic.shape)
print("Weighted skill:", weighted_skill_score.shape)

# ------------------------------------------------------------
# 2. Convert semantic matrix to job x resume orientation
# ------------------------------------------------------------

semantic_similarity = semantic_similarity.T

assert cosine.shape == (434, 1540)
assert skill_score.shape == (434, 1540)
assert role_similarity.shape == (434, 1540)
assert semantic_similarity.shape == (434, 1540)
assert role_title_semantic.shape == (434, 1540)
assert weighted_skill_score.shape == (434, 1540)

print("\n✅ All matrices aligned as (jobs, resumes).")

# ------------------------------------------------------------
# 3. Load the saved 9-feature supervised dataset
# ------------------------------------------------------------

dataset_path = (
    f"{BASE}/v3_supervised_dataset_568210_9features.pkl"
)

v3_9df = pd.read_pickle(dataset_path)

print("\nTraining dataset:")
print("Rows:", len(v3_9df))
print("Columns:", list(v3_9df.columns))

# ------------------------------------------------------------
# 4. Validate Normalized_Cosine
#
# Expected implementation:
# cosine / maximum cosine for that resume
# ------------------------------------------------------------

resume_max_cosine = cosine.max(axis=0)

normalized_cosine_candidate = (
    cosine /
    np.maximum(resume_max_cosine, 1e-12)
)

job_idx = v3_9df["Job_Index"].to_numpy()
resume_idx = v3_9df["Resume_Index"].to_numpy()

stored_norm = v3_9df["Normalized_Cosine"].to_numpy()

reconstructed_norm = normalized_cosine_candidate[
    job_idx,
    resume_idx
]

norm_mae = np.mean(
    np.abs(stored_norm - reconstructed_norm)
)

norm_max_error = np.max(
    np.abs(stored_norm - reconstructed_norm)
)

print("\nNormalized_Cosine validation:")
print("MAE:", norm_mae)
print("Max error:", norm_max_error)

# ------------------------------------------------------------
# 5. Validate Matched_Skill_Count
# ------------------------------------------------------------

stored_required = (
    v3_9df["Required_Skill_Count"].to_numpy()
)

stored_matched = (
    v3_9df["Matched_Skill_Count"].to_numpy()
)

reconstructed_matched = np.rint(
    v3_9df["Skill_Score"].to_numpy()
    * stored_required
).astype(np.int16)

matched_diff = (
    stored_matched.astype(np.int16)
    - reconstructed_matched
)

print("\nMatched_Skill_Count validation:")
print("Maximum absolute difference:",
      np.max(np.abs(matched_diff)))
print("Mismatches:",
      np.count_nonzero(matched_diff))

# ------------------------------------------------------------
# 6. Build Required_Skill_Count lookup for every job
# ------------------------------------------------------------

required_by_job = (
    v3_9df[
        ["Job_Index", "Required_Skill_Count"]
    ]
    .drop_duplicates("Job_Index")
    .sort_values("Job_Index")
)

assert len(required_by_job) == 434

required_skill_count = (
    required_by_job["Required_Skill_Count"]
    .to_numpy()
    .astype(np.int16)
)

print("\nRequired-skill lookup:")
print("Jobs covered:", len(required_skill_count))
print("Min:", required_skill_count.min())
print("Max:", required_skill_count.max())
print("Mean:", required_skill_count.mean())

# ------------------------------------------------------------
# 7. Final reconstruction test
# ------------------------------------------------------------

assert norm_mae < 1e-10
assert norm_max_error < 1e-10

assert np.max(np.abs(matched_diff)) == 0

print("\n" + "=" * 80)
print("✅ FEATURE RECONSTRUCTION PASSED")
print("=" * 80)

print("""
We can now safely construct the exact 9 features:

1. Cosine
2. Skill_Score
3. Normalized_Cosine
4. Required_Skill_Count
5. Matched_Skill_Count
6. Role_Similarity
7. Semantic_Similarity
8. Role_Title_Semantic_Similarity
9. Weighted_Skill_Score
""")

# Free the validation dataframe before inference
del v3_9df

In [ ]:
import numpy as np

stored_norm = v3_9df["Normalized_Cosine"].to_numpy()

job_idx = v3_9df["Job_Index"].to_numpy()
resume_idx = v3_9df["Resume_Index"].to_numpy()

# ------------------------------------------------------------
# Candidate 1:
# Cosine / max cosine PER RESUME
# ------------------------------------------------------------

resume_max = cosine.max(axis=0)

norm_resume_max = (
    cosine /
    np.maximum(resume_max, 1e-12)
)

pred_resume_max = norm_resume_max[
    job_idx,
    resume_idx
]


# ------------------------------------------------------------
# Candidate 2:
# Cosine / max cosine PER JOB
# ------------------------------------------------------------

job_max = cosine.max(axis=1)

norm_job_max = (
    cosine /
    np.maximum(job_max[:, None], 1e-12)
)

pred_job_max = norm_job_max[
    job_idx,
    resume_idx
]


# ------------------------------------------------------------
# Candidate 3:
# Min-Max PER RESUME
# ------------------------------------------------------------

resume_min = cosine.min(axis=0)
resume_max = cosine.max(axis=0)

norm_resume_minmax = (
    (cosine - resume_min[None, :]) /
    np.maximum(
        resume_max[None, :] - resume_min[None, :],
        1e-12
    )
)

pred_resume_minmax = norm_resume_minmax[
    job_idx,
    resume_idx
]


# ------------------------------------------------------------
# Candidate 4:
# Min-Max PER JOB
# ------------------------------------------------------------

job_min = cosine.min(axis=1)
job_max = cosine.max(axis=1)

norm_job_minmax = (
    (cosine - job_min[:, None]) /
    np.maximum(
        job_max[:, None] - job_min[:, None],
        1e-12
    )
)

pred_job_minmax = norm_job_minmax[
    job_idx,
    resume_idx
]


# ------------------------------------------------------------
# Candidate 5:
# Global max normalization
# ------------------------------------------------------------

norm_global_max = (
    cosine /
    max(cosine.max(), 1e-12)
)

pred_global_max = norm_global_max[
    job_idx,
    resume_idx
]


# ------------------------------------------------------------
# Candidate 6:
# Global min-max
# ------------------------------------------------------------

cmin = cosine.min()
cmax = cosine.max()

norm_global_minmax = (
    (cosine - cmin) /
    max(cmax - cmin, 1e-12)
)

pred_global_minmax = norm_global_minmax[
    job_idx,
    resume_idx
]


# ------------------------------------------------------------
# Compare everything
# ------------------------------------------------------------

candidates = {
    "Resume Max": pred_resume_max,
    "Job Max": pred_job_max,
    "Resume MinMax": pred_resume_minmax,
    "Job MinMax": pred_job_minmax,
    "Global Max": pred_global_max,
    "Global MinMax": pred_global_minmax,
}

print("=" * 80)
print("NORMALIZED COSINE FORMULA DIAGNOSTIC")
print("=" * 80)

for name, pred in candidates.items():

    mae = np.mean(np.abs(stored_norm - pred))
    max_error = np.max(np.abs(stored_norm - pred))
    corr = np.corrcoef(stored_norm, pred)[0, 1]

    print(f"\n{name}")
    print(f"  MAE       = {mae:.12f}")
    print(f"  Max Error = {max_error:.12f}")
    print(f"  Corr      = {corr:.12f}")

print("\nStored Normalized_Cosine:")
print("  Min :", stored_norm.min())
print("  Max :", stored_norm.max())
print("  Mean:", stored_norm.mean())

In [ ]:
import numpy as np
import pandas as pd
import os
import time

print("=" * 80)
print("V3 FINAL INFERENCE — 668,360 RESUME-JOB PAIRS")
print("=" * 80)

# ============================================================
# 1. Exact Normalized_Cosine reconstruction
# ============================================================

job_min = cosine.min(axis=1)
job_max = cosine.max(axis=1)

normalized_cosine = (
    (cosine - job_min[:, None]) /
    np.maximum(
        job_max[:, None] - job_min[:, None],
        1e-12
    )
).astype(np.float32)

print("✅ Normalized_Cosine reconstructed exactly.")


# ============================================================
# 2. Required skill count per job
# ============================================================

required_skill_count = (
    required_by_job
    .sort_values("Job_Index")
    ["Required_Skill_Count"]
    .to_numpy()
    .astype(np.float32)
)

assert len(required_skill_count) == 434


# ============================================================
# 3. Load job metadata
# ============================================================

INPUT_BASE = (
    "/content/drive/MyDrive/"
    "Resume_Model_Project/V2_100k/Inputs"
)

entry_job_df = pd.read_pickle(
    f"{INPUT_BASE}/entry_job_df.pkl"
).reset_index(drop=True)

assert len(entry_job_df) == 434

print(f"✅ Loaded {len(entry_job_df)} jobs.")


# ============================================================
# 4. Dimensions
# ============================================================

N_JOBS = 434
N_RESUMES = 1540
TOP_K = 20

assert cosine.shape == (N_JOBS, N_RESUMES)
assert skill_score.shape == (N_JOBS, N_RESUMES)
assert normalized_cosine.shape == (N_JOBS, N_RESUMES)
assert role_similarity.shape == (N_JOBS, N_RESUMES)
assert semantic_similarity.shape == (N_JOBS, N_RESUMES)
assert role_title_semantic.shape == (N_JOBS, N_RESUMES)
assert weighted_skill_score.shape == (N_JOBS, N_RESUMES)

print(f"✅ Total pairs to score: {N_JOBS * N_RESUMES:,}")


# ============================================================
# 5. Initialize Top-K storage
# ============================================================

top_scores = np.full(
    (TOP_K, N_RESUMES),
    -np.inf,
    dtype=np.float32
)

top_jobs = np.full(
    (TOP_K, N_RESUMES),
    -1,
    dtype=np.int16
)


# ============================================================
# 6. Batch inference
# ============================================================

BATCH_JOBS = 25

start_time = time.time()

for start in range(0, N_JOBS, BATCH_JOBS):

    end = min(start + BATCH_JOBS, N_JOBS)

    jobs = np.arange(start, end)
    n_batch_jobs = len(jobs)

    # --------------------------------------------------------
    # Build the exact 9 training features
    # --------------------------------------------------------

    batch_cosine = cosine[jobs]
    batch_skill = skill_score[jobs]

    batch_norm_cosine = normalized_cosine[jobs]

    batch_required = np.repeat(
        required_skill_count[jobs],
        N_RESUMES
    )

    # Exact reconstruction of matched skill count
    batch_matched = np.rint(
        batch_skill
        * required_skill_count[jobs, None]
    ).astype(np.float32)

    batch_role = role_similarity[jobs]

    batch_semantic = semantic_similarity[jobs]

    batch_role_title = role_title_semantic[jobs]

    batch_weighted_skill = weighted_skill_score[jobs]

    # --------------------------------------------------------
    # Stack in EXACT feature order used during training
    # --------------------------------------------------------

    X_batch = np.column_stack([
        batch_cosine.ravel(),
        batch_skill.ravel(),
        batch_norm_cosine.ravel(),
        batch_required,
        batch_matched.ravel(),
        batch_role.ravel(),
        batch_semantic.ravel(),
        batch_role_title.ravel(),
        batch_weighted_skill.ravel()
    ]).astype(np.float32)

    assert X_batch.shape == (
        n_batch_jobs * N_RESUMES,
        9
    )

    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    batch_predictions = final_model.predict(X_batch)

    batch_scores = batch_predictions.reshape(
        n_batch_jobs,
        N_RESUMES
    ).astype(np.float32)

    batch_job_ids = np.repeat(
        jobs[:, None],
        N_RESUMES,
        axis=1
    ).astype(np.int16)

    # --------------------------------------------------------
    # Merge with current Top-K
    # --------------------------------------------------------

    combined_scores = np.vstack([
        top_scores,
        batch_scores
    ])

    combined_jobs = np.vstack([
        top_jobs,
        batch_job_ids
    ])

    # Candidate selection
    selected = np.argpartition(
        combined_scores,
        -TOP_K,
        axis=0
    )[-TOP_K:, :]

    selected_scores = np.take_along_axis(
        combined_scores,
        selected,
        axis=0
    )

    selected_jobs = np.take_along_axis(
        combined_jobs,
        selected,
        axis=0
    )

    # Sort selected candidates descending
    order = np.argsort(
        selected_scores,
        axis=0
    )[::-1, :]

    top_scores = np.take_along_axis(
        selected_scores,
        order,
        axis=0
    )

    top_jobs = np.take_along_axis(
        selected_jobs,
        order,
        axis=0
    )

    elapsed = time.time() - start_time

    print(
        f"Jobs {start:3d}-{end-1:3d} | "
        f"{end/N_JOBS*100:6.1f}% | "
        f"Elapsed: {elapsed/60:.1f} min"
    )


print("\n" + "=" * 80)
print("✅ FULL INFERENCE COMPLETE")
print("=" * 80)

print(
    f"Scored {N_JOBS * N_RESUMES:,} resume-job pairs."
)
print(
    f"Stored Top-{TOP_K} recommendations for "
    f"{N_RESUMES:,} resumes."
)

In [ ]:
print("=" * 80)
print("BUILDING FINAL V3-RS RECOMMENDATIONS")
print("=" * 80)

rows = []

for resume_idx in range(N_RESUMES):

    for rank in range(TOP_K):

        job_idx = int(top_jobs[rank, resume_idx])
        score = float(top_scores[rank, resume_idx])

        job = entry_job_df.iloc[job_idx]

        rows.append({
            "Resume_Index": resume_idx,
            "Rank": rank + 1,
            "Job_Index": job_idx,
            "Final_V3_RS_Score": score,
            "JobID": job["JobID"],
            "Title": job["Title"],
            "ExperienceLevel": job["ExperienceLevel"],
            "YearsOfExperience": job["YearsOfExperience"]
        })


final_recommendations = pd.DataFrame(rows)

print("\nShape:", final_recommendations.shape)

print("\nFirst recommendations:")
display(
    final_recommendations.head(20)
)

print("\n✅ Final recommendation table created.")

In [ ]:
print("=" * 80)
print("V3 FINAL RECOMMENDATION SANITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Basic checks
# ------------------------------------------------------------

assert len(final_recommendations) == 1540 * 20
assert final_recommendations["Resume_Index"].nunique() == 1540

print("✅ Correct number of recommendation rows.")

# ------------------------------------------------------------
# 2. Every resume has exactly 20 recommendations
# ------------------------------------------------------------

counts = (
    final_recommendations
    .groupby("Resume_Index")
    .size()
)

print("\nRecommendations per resume:")
print("Min:", counts.min())
print("Max:", counts.max())

assert counts.min() == 20
assert counts.max() == 20

print("✅ Every resume has exactly 20 recommendations.")

# ------------------------------------------------------------
# 3. Ranks
# ------------------------------------------------------------

rank_counts = (
    final_recommendations
    .groupby("Resume_Index")["Rank"]
    .agg(["min", "max", "count"])
)

assert (rank_counts["min"] == 1).all()
assert (rank_counts["max"] == 20).all()
assert (rank_counts["count"] == 20).all()

print("✅ Ranks 1–20 are present for every resume.")

# ------------------------------------------------------------
# 4. No duplicate Job_Index within a resume
# ------------------------------------------------------------

duplicates = (
    final_recommendations
    .groupby("Resume_Index")["Job_Index"]
    .nunique()
)

print("\nUnique jobs per resume:")
print("Min:", duplicates.min())
print("Max:", duplicates.max())

assert (duplicates == 20).all()

print("✅ No duplicate job listings within a Top-20 list.")

# ------------------------------------------------------------
# 5. Scores must be descending
# ------------------------------------------------------------

descending_ok = True

for resume_idx, group in final_recommendations.groupby(
    "Resume_Index"
):
    scores = group.sort_values("Rank")[
        "Final_V3_RS_Score"
    ].to_numpy()

    if not np.all(scores[:-1] >= scores[1:]):
        descending_ok = False
        print(
            "Descending-order failure for resume:",
            resume_idx
        )
        break

assert descending_ok

print("✅ Scores are correctly sorted descending.")

# ------------------------------------------------------------
# 6. Check for invalid values
# ------------------------------------------------------------

numeric_cols = [
    "Final_V3_RS_Score",
    "Resume_Index",
    "Rank",
    "Job_Index"
]

print("\nMissing values:")
print(
    final_recommendations[
        numeric_cols
    ].isna().sum()
)

assert not final_recommendations[
    numeric_cols
].isna().any().any()

assert np.isfinite(
    final_recommendations["Final_V3_RS_Score"]
).all()

print("✅ No NaN or infinite scores.")

# ------------------------------------------------------------
# 7. Score statistics
# ------------------------------------------------------------

scores = final_recommendations[
    "Final_V3_RS_Score"
]

print("\nScore statistics:")
print(f"Min    : {scores.min():.6f}")
print(f"Max    : {scores.max():.6f}")
print(f"Mean   : {scores.mean():.6f}")
print(f"Median : {scores.median():.6f}")

print("\n" + "=" * 80)
print("✅ ALL FINAL RECOMMENDATION CHECKS PASSED")
print("=" * 80)

In [ ]:
print("=" * 80)
print("V3 FINAL RECOMMENDATION QUALITY INSPECTION")
print("=" * 80)

# ------------------------------------------------------------
# 1. How many recommendations have exactly score 2.0?
# ------------------------------------------------------------

score_2_count = (
    np.isclose(
        final_recommendations["Final_V3_RS_Score"],
        2.0,
        atol=1e-6
    ).sum()
)

total = len(final_recommendations)

print(
    f"\nRecommendations with score exactly 2.0: "
    f"{score_2_count:,} / {total:,} "
    f"({score_2_count/total*100:.2f}%)"
)


# ------------------------------------------------------------
# 2. Unique job titles in each Top-20
# ------------------------------------------------------------

title_diversity = (
    final_recommendations
    .groupby("Resume_Index")["Title"]
    .nunique()
)

print("\nUnique titles per Top-20:")
print(f"Minimum : {title_diversity.min()}")
print(f"Maximum : {title_diversity.max()}")
print(f"Mean    : {title_diversity.mean():.2f}")
print(f"Median  : {title_diversity.median():.0f}")


# ------------------------------------------------------------
# 3. Distribution of title diversity
# ------------------------------------------------------------

print("\nTitle diversity distribution:")

display(
    title_diversity
    .value_counts()
    .sort_index()
    .rename_axis("Unique_Titles")
    .reset_index(name="Resume_Count")
)


# ------------------------------------------------------------
# 4. Most common Top-1 titles
# ------------------------------------------------------------

top1 = (
    final_recommendations[
        final_recommendations["Rank"] == 1
    ]
)

top1_titles = (
    top1["Title"]
    .value_counts()
    .head(15)
    .rename_axis("Title")
    .reset_index(name="Top1_Count")
)

print("\nMost common Top-1 titles:")

display(top1_titles)


# ------------------------------------------------------------
# 5. Inspect several complete recommendation lists
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAMPLE RECOMMENDATION LISTS")
print("=" * 80)

sample_resumes = [0, 1, 10, 50, 100]

for resume_idx in sample_resumes:

    sample = (
        final_recommendations[
            final_recommendations["Resume_Index"] == resume_idx
        ][[
            "Rank",
            "JobID",
            "Title",
            "ExperienceLevel",
            "Final_V3_RS_Score"
        ]]
        .sort_values("Rank")
    )

    print(f"\nResume {resume_idx}")
    display(sample)


print("\n" + "=" * 80)
print("✅ QUALITY INSPECTION COMPLETE")
print("=" * 80)

In [ ]:
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/Resume_Model_Project/V3/Outputs"

COMMON_PATH = (
    f"{BASE}/v3_10x_unseen_validation_sampled_pairs.csv"
)

RS_PATH = (
    f"{BASE}/v3_enhancement_ablation_predictions.csv"
)

common = pd.read_csv(COMMON_PATH)
rs = pd.read_csv(RS_PATH)

print("=" * 80)
print("FINAL V1 vs V2 vs V3-RS COMPARISON")
print("=" * 80)

print("\nCommon benchmark:")
print("Rows:", len(common))
print("Trials:", common["Trial"].nunique())
print("Resumes:", common["Resume_Index"].nunique())

print("\nV3-RS predictions:")
print("Rows:", len(rs))
print("Columns:", list(rs.columns))


# ============================================================
# Merge using the actual unique pair identity
# ============================================================

merge_keys = [
    "Trial",
    "Resume_Index",
    "Job_Index"
]

# Make sure RS has only one row per trial/resume/job
duplicate_count = (
    rs.duplicated(merge_keys).sum()
)

print("\nDuplicate RS keys:", duplicate_count)

assert duplicate_count == 0

comparison = common.merge(
    rs,
    on=merge_keys,
    how="inner",
    suffixes=("_common", "_rs")
)

print("Merged rows:", len(comparison))

assert len(comparison) == len(common)


# ============================================================
# Identify prediction columns
# ============================================================

# Common benchmark already contains V1 and V2.
# RS file supplies the final V3-RS score.

comparison["V1_Final_Score"] = comparison["V1_Score"]
comparison["V2_Final_Score"] = comparison["V2_Score"]

if "Final_V3_RS_Score" in comparison.columns:
    comparison["V3_RS_Final_Score"] = (
        comparison["Final_V3_RS_Score"]
    )
elif "Score" in comparison.columns:
    comparison["V3_RS_Final_Score"] = comparison["Score"]
else:
    raise ValueError(
        "Could not identify the V3-RS prediction column."
    )

label_col = "Label_common"

# ============================================================
# Ranking metric functions
# ============================================================

def top1_accuracy(df, score_col):
    correct = 0
    total = 0

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        g = g.sort_values(
            score_col,
            ascending=False
        )

        top_label = g.iloc[0][label_col]
        best_label = g[label_col].max()

        if top_label == best_label:
            correct += 1

        total += 1

    return correct / total


def hit_at_k(df, score_col, k):
    hits = 0
    total = 0

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        g = g.sort_values(
            score_col,
            ascending=False
        )

        best_label = g[label_col].max()

        if (g.head(k)[label_col] == best_label).any():
            hits += 1

        total += 1

    return hits / total


def ndcg_at_5(df, score_col):
    values = []

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        g = g.sort_values(
            score_col,
            ascending=False
        )

        labels = g.head(5)[label_col].to_numpy()

        gains = (2 ** labels) - 1

        discounts = np.log2(
            np.arange(2, len(labels) + 2)
        )

        dcg = np.sum(
            gains / discounts
        )

        ideal_labels = np.sort(
            g[label_col].to_numpy()
        )[::-1][:5]

        ideal_gains = (
            (2 ** ideal_labels) - 1
        )

        ideal_dcg = np.sum(
            ideal_gains / discounts[:len(ideal_labels)]
        )

        if ideal_dcg == 0:
            values.append(0.0)
        else:
            values.append(dcg / ideal_dcg)

    return np.mean(values)


def pairwise_accuracy(df, score_col):
    values = []

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        scores = g[score_col].to_numpy()
        labels = g[label_col].to_numpy()

        correct = 0
        total = 0

        for i in range(len(g)):
            for j in range(i + 1, len(g)):

                if labels[i] == labels[j]:
                    continue

                total += 1

                predicted_order = (
                    scores[i] > scores[j]
                )

                actual_order = (
                    labels[i] > labels[j]
                )

                if predicted_order == actual_order:
                    correct += 1

        if total > 0:
            values.append(correct / total)

    return np.mean(values)


# ============================================================
# Calculate metrics
# ============================================================

models = {
    "V1": "V1_Final_Score",
    "V2": "V2_Final_Score",
    "V3-RS": "V3_RS_Final_Score"
}

results = []

for model_name, score_col in models.items():

    print(f"\nCalculating {model_name}...")

    results.append({
        "Model": model_name,
        "Top1": top1_accuracy(
            comparison,
            score_col
        ),
        "Hit@3": hit_at_k(
            comparison,
            score_col,
            3
        ),
        "Hit@5": hit_at_k(
            comparison,
            score_col,
            5
        ),
        "NDCG@5": ndcg_at_5(
            comparison,
            score_col
        ),
        "Pairwise": pairwise_accuracy(
            comparison,
            score_col
        )
    })

final_comparison = pd.DataFrame(results)

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

display(
    final_comparison.style.format({
        "Top1": "{:.4f}",
        "Hit@3": "{:.4f}",
        "Hit@5": "{:.4f}",
        "NDCG@5": "{:.4f}",
        "Pairwise": "{:.4f}"
    })
)

In [ ]:
import pandas as pd
import numpy as np

print("=" * 80)
print("FINAL V1 vs V2 vs V3-RS COMPARISON")
print("=" * 80)

# ============================================================
# Use the already-created merged dataframe
# ============================================================

print("Merged rows:", len(comparison))

assert len(comparison) == 25400

# ------------------------------------------------------------
# Prediction columns
# ------------------------------------------------------------

comparison["V1_Final_Score"] = comparison["V1_Score"]
comparison["V2_Final_Score"] = comparison["V2_Score"]
comparison["V3_RS_Final_Score"] = comparison["V3RS_Score"]

# Label after merge
label_col = "Label_common"

assert label_col in comparison.columns

print("\nPrediction columns:")
print("V1  :", "V1_Final_Score")
print("V2  :", "V2_Final_Score")
print("V3RS:", "V3_RS_Final_Score")

# ============================================================
# Metric functions
# ============================================================

def top1_accuracy(df, score_col):
    correct = 0
    total = 0

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        g = g.sort_values(
            score_col,
            ascending=False
        )

        if g.iloc[0][label_col] == g[label_col].max():
            correct += 1

        total += 1

    return correct / total


def hit_at_k(df, score_col, k):
    hits = 0
    total = 0

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        g = g.sort_values(
            score_col,
            ascending=False
        )

        best_label = g[label_col].max()

        if (g.head(k)[label_col] == best_label).any():
            hits += 1

        total += 1

    return hits / total


def ndcg_at_5(df, score_col):
    values = []

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        g = g.sort_values(
            score_col,
            ascending=False
        )

        labels = g.head(5)[label_col].to_numpy()

        gains = (2 ** labels) - 1

        discounts = np.log2(
            np.arange(2, len(labels) + 2)
        )

        dcg = np.sum(
            gains / discounts
        )

        ideal_labels = np.sort(
            g[label_col].to_numpy()
        )[::-1][:5]

        ideal_gains = (
            (2 ** ideal_labels) - 1
        )

        ideal_dcg = np.sum(
            ideal_gains /
            discounts[:len(ideal_labels)]
        )

        values.append(
            0.0 if ideal_dcg == 0
            else dcg / ideal_dcg
        )

    return np.mean(values)


def pairwise_accuracy(df, score_col):
    """
    Same conceptual pairwise metric:
    ignore equal-label pairs and give credit only when
    the predicted score ordering agrees with label ordering.

    Implemented efficiently without nested O(n^2) Python loops.
    """

    case_scores = []

    for (_, resume_idx), g in df.groupby(
        ["Trial", "Resume_Index"]
    ):
        labels = g[label_col].to_numpy()
        scores = g[score_col].to_numpy()

        correct = 0
        total = 0

        unique_labels = np.unique(labels)

        for high_label in unique_labels:
            for low_label in unique_labels:

                if high_label <= low_label:
                    continue

                high_scores = scores[
                    labels == high_label
                ]

                low_scores = np.sort(
                    scores[labels == low_label]
                )

                if len(high_scores) == 0 or len(low_scores) == 0:
                    continue

                # Number of low-label scores strictly below
                # each high-label score.
                correct += np.searchsorted(
                    low_scores,
                    high_scores,
                    side="left"
                ).sum()

                total += (
                    len(high_scores)
                    * len(low_scores)
                )

        if total > 0:
            case_scores.append(
                correct / total
            )

    return np.mean(case_scores)


# ============================================================
# Calculate final metrics
# ============================================================

models = {
    "V1": "V1_Final_Score",
    "V2": "V2_Final_Score",
    "V3-RS": "V3_RS_Final_Score"
}

results = []

for model_name, score_col in models.items():

    print(f"\nCalculating {model_name}...")

    results.append({
        "Model": model_name,

        "Top1": top1_accuracy(
            comparison,
            score_col
        ),

        "Hit@3": hit_at_k(
            comparison,
            score_col,
            3
        ),

        "Hit@5": hit_at_k(
            comparison,
            score_col,
            5
        ),

        "NDCG@5": ndcg_at_5(
            comparison,
            score_col
        ),

        "Pairwise": pairwise_accuracy(
            comparison,
            score_col
        )
    })


# ============================================================
# Final table
# ============================================================

final_comparison = pd.DataFrame(results)

print("\n" + "=" * 80)
print("FINAL V1 vs V2 vs V3-RS RESULTS")
print("=" * 80)

display(
    final_comparison.style.format({
        "Top1": "{:.6f}",
        "Hit@3": "{:.6f}",
        "Hit@5": "{:.6f}",
        "NDCG@5": "{:.6f}",
        "Pairwise": "{:.6f}"
    })
)

In [ ]:
import os
import json
import pandas as pd

OUT = "/content/V3_FINAL_HANDOFF"
os.makedirs(OUT, exist_ok=True)

print("=" * 80)
print("SAVING FINAL V3 HANDOFF ARTIFACTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Final Top-20 recommendations
# ------------------------------------------------------------

recommendations_path = (
    f"{OUT}/v3_final_top20_recommendations.csv"
)

final_recommendations.to_csv(
    recommendations_path,
    index=False
)

print("✅", recommendations_path)


# ------------------------------------------------------------
# 2. Final V1/V2/V3 comparison
# ------------------------------------------------------------

comparison_path = (
    f"{OUT}/v1_v2_v3rs_final_comparison.csv"
)

final_comparison.to_csv(
    comparison_path,
    index=False
)

print("✅", comparison_path)


# ------------------------------------------------------------
# 3. Final model metadata
# ------------------------------------------------------------

metadata = {
    "model_name": "V3-RS",
    "model_type": "ExtraTreesRegressor",

    "training_rows": 568210,

    "candidate_count": 1540,
    "job_count": 434,
    "full_pair_count": 668360,

    "recommendation_count_per_resume": 20,
    "recommendation_rows": 30800,

    "n_estimators": 200,
    "min_samples_leaf": 2,

    "feature_order": [
        "Cosine",
        "Skill_Score",
        "Normalized_Cosine",
        "Required_Skill_Count",
        "Matched_Skill_Count",
        "Role_Similarity",
        "Semantic_Similarity",
        "Role_Title_Semantic_Similarity",
        "Weighted_Skill_Score"
    ],

    "benchmark": {
        "rows": 25400,
        "trials": 10,
        "heldout_resumes": 308
    },

    "final_model_file": (
        "v3_final_extratrees_rs_568210.joblib"
    )
}

metadata_path = (
    f"{OUT}/v3_final_model_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("✅", metadata_path)


# ------------------------------------------------------------
# 4. Feature importance
# ------------------------------------------------------------

feature_importance_src = (
    "/content/v3_final_rs_feature_importance.csv"
)

feature_importance_dst = (
    f"{OUT}/v3_final_rs_feature_importance.csv"
)

if os.path.exists(feature_importance_src):
    import shutil

    shutil.copy2(
        feature_importance_src,
        feature_importance_dst
    )

    print("✅", feature_importance_dst)
else:
    print("⚠️ Feature importance file not found")


# ------------------------------------------------------------
# 5. Display sizes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("HANDOFF DIRECTORY")
print("=" * 80)

for name in sorted(os.listdir(OUT)):

    path = os.path.join(OUT, name)

    size_mb = os.path.getsize(path) / (1024**2)

    print(f"{name:50s} {size_mb:8.2f} MB")

print("\n✅ Final outputs saved locally.")

In [ ]:
import os
import shutil

OUT = "/content/V3_FINAL_HANDOFF"

DRIVE_BASE = (
    "/content/drive/MyDrive/"
    "Resume_Model_Project/V3/Outputs"
)

V2_INPUT = (
    "/content/drive/MyDrive/"
    "Resume_Model_Project/V2_100k/Inputs"
)

# source -> destination filename
artifacts = {

    # Core reference data
    f"{V2_INPUT}/tfidf_vectorizer.pkl":
        "tfidf_vectorizer.pkl",

    f"{V2_INPUT}/candidate_df.pkl":
        "candidate_df.pkl",

    f"{V2_INPUT}/entry_job_df.pkl":
        "entry_job_df.pkl",

    # V3 metadata / feature definitions
    f"{DRIVE_BASE}/v3_9feature_manifest.csv":
        "v3_9feature_manifest.csv",

    f"{DRIVE_BASE}/v3_embedding_metadata.json":
        "v3_embedding_metadata.json",

    f"{DRIVE_BASE}/v3_weighted_skill_weights.csv":
        "v3_weighted_skill_weights.csv",

    # Evaluation
    f"{DRIVE_BASE}/v3_enhancement_ablation_summary.csv":
        "v3_enhancement_ablation_summary.csv",

    f"{DRIVE_BASE}/v3_enhancement_ablation_case_metrics.csv":
        "v3_enhancement_ablation_case_metrics.csv",

    f"{DRIVE_BASE}/v3_enhancement_ablation_trial_results.csv":
        "v3_enhancement_ablation_trial_results.csv",

    f"{DRIVE_BASE}/v3_10x_unseen_validation_summary.csv":
        "v3_10x_unseen_validation_summary.csv",

    f"{DRIVE_BASE}/v3_10x_unseen_validation_v2_to_v3_improvement.csv":
        "v3_10x_unseen_validation_v2_to_v3_improvement.csv",

    # Error analysis
    f"{DRIVE_BASE}/v3_final_rs_error_analysis_case_level.csv":
        "v3_final_rs_error_analysis_case_level.csv",

    f"{DRIVE_BASE}/v3_final_rs_error_analysis_wrong_top1.csv":
        "v3_final_rs_error_analysis_wrong_top1.csv",

    f"{DRIVE_BASE}/v3_final_rs_best_label_rank_distribution.csv":
        "v3_final_rs_best_label_rank_distribution.csv",

    f"{DRIVE_BASE}/v3_final_rs_error_feature_gaps.csv":
        "v3_final_rs_error_feature_gaps.csv",
}


copied = 0
missing = []

for src, filename in artifacts.items():

    dst = os.path.join(
        OUT,
        filename
    )

    if os.path.exists(src):

        shutil.copy2(src, dst)
        copied += 1

    else:
        missing.append(src)


print("=" * 80)
print("SUPPORTING ARTIFACT COPY")
print("=" * 80)

print(f"Copied: {copied}")

if missing:

    print("\n⚠️ Files not found:")
    for x in missing:
        print(" ", x)

else:
    print("✅ All requested supporting artifacts copied.")

In [ ]:
import shutil
import os

OUT = "/content/V3_FINAL_HANDOFF"

ZIP_BASE = "/content/V3_FINAL_TEAM_HANDOFF"

zip_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUT
)

print("=" * 80)
print("SMALL V3 TEAM PACKAGE CREATED")
print("=" * 80)

print("ZIP:", zip_path)

size_mb = (
    os.path.getsize(zip_path)
    / (1024**2)
)

print(f"Size: {size_mb:.2f} MB")